# PMCI Full

In [ ]:
# ============================================================
# PMCI FULL QUIET COLAB PIPELINE

# ============================================================

# -----------------------------
# 0) QUIET INSTALLS
# -----------------------------
import sys, subprocess, os

def pip_install_quiet(packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

pip_install_quiet([
    "ultralytics>=8.0.0",
    "opencv-python-headless",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "tqdm"
])


In [ ]:
# -----------------------------
# 1) IMPORTS AND QUIET MODE
# -----------------------------
import os, sys, gc, re, json, math, zipfile, urllib.request, contextlib, warnings, logging, random, time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"] = "3"
os.environ["ULTRALYTICS_VERBOSE"] = "False"
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import cv2

from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE_TORCH = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_YOLO = 0 if torch.cuda.is_available() else "cpu"

ROOT = Path("/content/cias_q1_min")
RAW_DIR = ROOT / "raw"
RAVDESS_RAW = RAW_DIR / "RAVDESS"
SHARD_ROOT = ROOT / "shards"
RAVDESS_SHARDS = SHARD_ROOT / "RAVDESS"
RESULTS_DIR = ROOT / "results_pmci_reviewer_continuation_quiet"

for d in [RAW_DIR, RAVDESS_RAW, SHARD_ROOT, RAVDESS_SHARDS, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

@contextlib.contextmanager
def suppress_stdout_stderr():
    """Suppress noisy stdout/stderr from external libraries."""
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

def progress_line(stage: str, i: int, total: int, extra: str = ""):
    pct = 100.0 * i / max(total, 1)
    clear_output(wait=True)
    print(f"{stage}: {pct:6.2f}% | {i}/{total} {extra}")

print("Device:", DEVICE_TORCH)
print("ROOT:", ROOT)


Device: cuda
ROOT: /content/cias_q1_min


In [ ]:
# -----------------------------
# 2) DOWNLOAD AND EXTRACT RAVDESS
# -----------------------------

def download_url_quiet(url: str, out_path: Path, stage_name: str = "Downloading"):
    if out_path.exists() and out_path.stat().st_size > 1_000_000:
        return

    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")

    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            total = int(response.headers.get("Content-Length", 0))
            downloaded = 0
            chunk_size = 1024 * 1024
            with open(tmp_path, "wb") as f:
                last_print = 0
                while True:
                    chunk = response.read(chunk_size)
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total > 0:
                        pct = int(downloaded * 100 / total)
                        if pct >= last_print + 5 or pct == 100:
                            clear_output(wait=True)
                            print(f"{stage_name}: {out_path.name} | {pct}%")
                            last_print = pct
        tmp_path.rename(out_path)
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(f"Download failed: {url} -> {out_path} | {e}")


def download_ravdess_actors(actor_start=1, actor_end=12):
    """Download Video_Speech_Actor_01.zip ... Video_Speech_Actor_12.zip from Zenodo."""
    base = "https://zenodo.org/records/1188976/files/Video_Speech_Actor_{actor:02d}.zip?download=1"
    total = actor_end - actor_start + 1
    for idx, actor in enumerate(range(actor_start, actor_end + 1), start=1):
        zip_path = RAVDESS_RAW / f"Video_Speech_Actor_{actor:02d}.zip"
        url = base.format(actor=actor)
        download_url_quiet(url, zip_path, stage_name=f"RAVDESS actor {actor:02d}/{actor_end:02d}")
        progress_line("RAVDESS download", idx, total, extra=f"actor={actor:02d}")


def extract_ravdess_zips():
    zips = sorted(RAVDESS_RAW.glob("Video_Speech_Actor_*.zip"))
    total = len(zips)
    for i, zp in enumerate(zips, start=1):
        # Check if actor folder already has mp4 files
        actor_num = re.search(r"Actor_(\d+)", zp.name)
        expected_folder = RAVDESS_RAW / f"Actor_{actor_num.group(1)}" if actor_num else None
        if expected_folder and len(list(expected_folder.rglob("*.mp4"))) > 0:
            progress_line("RAVDESS extract", i, total, extra=f"already extracted {zp.name}")
            continue
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(RAVDESS_RAW)
        progress_line("RAVDESS extract", i, total, extra=zp.name)


def find_ravdess_videos() -> List[Path]:
    videos = sorted(RAVDESS_RAW.rglob("*.mp4"))
    # Keep only standard RAVDESS file names
    videos = [p for p in videos if re.match(r"\d{2}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}\.mp4", p.name)]
    return videos


In [ ]:
# -----------------------------
# 3) RAVDESS PARSING
# -----------------------------

def parse_ravdess_actor(path: Path) -> int:
    # filename: 03-01-05-01-02-01-12.mp4, last field = actor
    try:
        return int(path.stem.split("-")[-1])
    except Exception:
        return -1


def parse_ravdess_emotion(path: Path) -> int:
    # third field = emotion: 01 neutral, 02 calm, 03 happy, ...
    try:
        return int(path.stem.split("-")[2])
    except Exception:
        return -1

RAVDESS_EMOTION_MAP = {
    1: "neutral", 2: "calm", 3: "happy", 4: "sad",
    5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"
}


In [ ]:
# -----------------------------
# 4) YOLOV8 POSE FEATURE EXTRACTION
# -----------------------------
# YOLO COCO keypoints: 0 nose, 1 left eye, 2 right eye, 3 left ear, 4 right ear,
# 5 left shoulder, 6 right shoulder, 7 left elbow, 8 right elbow,
# 9 left wrist, 10 right wrist, 11 left hip, 12 right hip,
# 13 left knee, 14 right knee, 15 left ankle, 16 right ankle.
FACE_IDXS = [0, 1, 2, 3, 4]                              # 5 keypoints -> 10 dims
BODY_IDXS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]   # 12 keypoints -> 24 dims
CENTER_IDXS = [5, 6, 11, 12]

# Load YOLO quietly. First run may download weights.
clear_output(wait=True)
print("Loading YOLOv8 pose model...")
with suppress_stdout_stderr():
    YOLO_MODEL = YOLO("yolov8n-pose.pt")
print("YOLOv8 pose loaded.")


def read_uniform_frames(
    video_path: Path,
    n_frames: int = 12,
    start_frac: float = 0.0,
    end_frac: float = 1.0,
) -> Optional[List[np.ndarray]]:
    """Read n uniformly spaced frames from a temporal crop of a video.

    start_frac/end_frac define the crop as a fraction of the full video duration.
    This keeps the window length fixed at 12 frames, while allowing several
    temporal windows to be extracted from the same source video.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None

    start_frac = float(np.clip(start_frac, 0.0, 1.0))
    end_frac = float(np.clip(end_frac, 0.0, 1.0))

    start_idx = int(round(start_frac * max(total - 1, 0)))
    end_idx = int(round(end_frac * max(total - 1, 0)))
    if end_idx <= start_idx:
        end_idx = total - 1

    idxs = np.linspace(start_idx, end_idx, n_frames).astype(int)

    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            frames.append(None)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

    cap.release()

    if all(f is None for f in frames):
        return None

    return frames


def pick_best_person_from_result(result) -> Optional[np.ndarray]:
    if result.keypoints is None or result.keypoints.xyn is None:
        return None
    xy = result.keypoints.xyn
    if xy is None:
        return None
    xy = xy.detach().cpu().numpy()  # (num_people, 17, 2)
    if xy.ndim != 3 or xy.shape[0] == 0 or xy.shape[1] < 17:
        return None

    conf = None
    try:
        conf = result.keypoints.conf
        if conf is not None:
            conf = conf.detach().cpu().numpy()  # (num_people, 17)
    except Exception:
        conf = None

    if conf is not None and conf.ndim == 2 and conf.shape[0] == xy.shape[0]:
        scores = np.nanmean(conf, axis=1)
        best = int(np.nanargmax(scores))
    else:
        # fallback: person with largest bbox area from keypoints
        areas = []
        for person in xy:
            xmin, ymin = np.nanmin(person[:, 0]), np.nanmin(person[:, 1])
            xmax, ymax = np.nanmax(person[:, 0]), np.nanmax(person[:, 1])
            areas.append(max(0, xmax - xmin) * max(0, ymax - ymin))
        best = int(np.nanargmax(areas))
    return xy[best, :17, :].astype(np.float32)


def interpolate_missing_keypoints(arr: np.ndarray) -> Optional[np.ndarray]:
    """arr shape: (T, 17, 2) with NaNs possible."""
    T = arr.shape[0]
    flat = arr.reshape(T, -1)
    finite_ratio = np.isfinite(flat).mean()
    if finite_ratio < 0.35:
        return None
    df = pd.DataFrame(flat)
    flat2 = df.interpolate(axis=0, limit_direction="both").fillna(method="bfill").fillna(method="ffill").fillna(0.0).values
    return flat2.reshape(T, 17, 2).astype(np.float32)


def person_centered_normalize(xy_seq: np.ndarray) -> np.ndarray:
    """Normalize each frame around body center and scale by shoulder/hip spread."""
    out = xy_seq.copy().astype(np.float32)
    T = out.shape[0]
    for t in range(T):
        pts = out[t]
        center = np.nanmean(pts[CENTER_IDXS], axis=0)
        if not np.isfinite(center).all():
            center = np.nanmean(pts, axis=0)
        left_sh, right_sh = pts[5], pts[6]
        left_hip, right_hip = pts[11], pts[12]
        shoulder_w = float(np.linalg.norm(left_sh - right_sh))
        hip_w = float(np.linalg.norm(left_hip - right_hip))
        bbox_w = float(np.nanmax(pts[:, 0]) - np.nanmin(pts[:, 0]))
        bbox_h = float(np.nanmax(pts[:, 1]) - np.nanmin(pts[:, 1]))
        scale = max(shoulder_w, hip_w, bbox_w, bbox_h, 1e-3)
        out[t] = (pts - center) / scale
    return out.astype(np.float32)


def add_temporal_deltas(base: np.ndarray) -> np.ndarray:
    """base shape: (T, D). returns (T, 3D): current, delta-1, delta-2."""
    T, D = base.shape
    d1 = np.zeros_like(base)
    d2 = np.zeros_like(base)
    d1[1:] = base[1:] - base[:-1]
    d2[2:] = base[2:] - base[:-2]
    return np.concatenate([base, d1, d2], axis=1).astype(np.float32)


def extract_window_from_video(
    video_path: Path,
    n_frames: int = 12,
    start_frac: float = 0.0,
    end_frac: float = 1.0,
) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    frames = read_uniform_frames(
        video_path,
        n_frames=n_frames,
        start_frac=start_frac,
        end_frac=end_frac,
    )
    if frames is None:
        return None

    seq = []
    valid = 0
    for frame in frames:
        if frame is None:
            seq.append(np.full((17, 2), np.nan, dtype=np.float32))
            continue
        try:
            with suppress_stdout_stderr():
                results = YOLO_MODEL.predict(
                    frame,
                    imgsz=320,
                    device=DEVICE_YOLO,
                    verbose=False,
                    conf=0.25
                )
            person = pick_best_person_from_result(results[0])
        except Exception:
            person = None
        if person is None:
            seq.append(np.full((17, 2), np.nan, dtype=np.float32))
        else:
            seq.append(person)
            valid += 1

    if valid < max(4, n_frames // 3):
        return None

    xy = np.stack(seq, axis=0)  # (T,17,2)
    xy = interpolate_missing_keypoints(xy)
    if xy is None:
        return None
    xy = person_centered_normalize(xy)

    face_base = xy[:, FACE_IDXS, :].reshape(n_frames, -1)  # (T,10)
    body_base = xy[:, BODY_IDXS, :].reshape(n_frames, -1)  # (T,24)

    face_feat = add_temporal_deltas(face_base)  # (T,30)
    body_feat = add_temporal_deltas(body_base)  # (T,72)

    if face_feat.shape != (n_frames, 30) or body_feat.shape != (n_frames, 72):
        return None
    if not np.isfinite(face_feat).all() or not np.isfinite(body_feat).all():
        return None

    return face_feat.astype(np.float32), body_feat.astype(np.float32)


Loading YOLOv8 pose model...
YOLOv8 pose loaded.


In [ ]:
# -----------------------------
# 4.1) TEMPORAL CROP HELPER
# -----------------------------

def make_temporal_crops(n_crops: int = 4, crop_fraction: float = 0.60):
    """Return temporal crop ranges inside one source video.

    The sequence length stays fixed at WINDOW_FRAMES=12. Increasing n_crops
    increases the number of 12-frame windows sampled from each video.
    """
    n_crops = int(n_crops)

    if n_crops <= 1:
        return [(0.0, 1.0)]

    crop_fraction = float(np.clip(crop_fraction, 0.20, 1.0))
    max_start = max(0.0, 1.0 - crop_fraction)

    starts = np.linspace(0.0, max_start, n_crops)
    crops = []

    for s in starts:
        e = min(1.0, float(s) + crop_fraction)
        crops.append((float(s), float(e)))

    return crops


In [ ]:
# -----------------------------
# 5) BUILD SHARDS QUIETLY
# -----------------------------

def save_shards(dataset_name: str, records: List[Dict], out_dir: Path, shard_size: int = 256):
    out_dir.mkdir(parents=True, exist_ok=True)
    # Clear old shard files only for this dataset folder
    for old in out_dir.glob("*.npz"):
        old.unlink()

    n = len(records)
    for s, start in enumerate(range(0, n, shard_size)):
        chunk = records[start:start + shard_size]
        face = np.stack([r["face"] for r in chunk]).astype(np.float32)
        body = np.stack([r["body"] for r in chunk]).astype(np.float32)
        video_id = np.array([r["video_id"] for r in chunk]).astype(str)
        subject_id = np.array([r["subject_id"] for r in chunk]).astype(np.int32)
        emotion_id = np.array([r["emotion_id"] for r in chunk]).astype(np.int32)
        emotion_name = np.array([r["emotion_name"] for r in chunk]).astype(str)
        dataset = np.array([dataset_name] * len(chunk)).astype(str)
        out_path = out_dir / f"{dataset_name.lower()}_shard_{s:03d}.npz"
        np.savez_compressed(
            out_path,
            face=face,
            body=body,
            video_id=video_id,
            subject_id=subject_id,
            emotion_id=emotion_id,
            emotion_name=emotion_name,
            dataset=dataset,
        )
    print(f"Saved {len(list(out_dir.glob('*.npz')))} shard files to {out_dir}")


def build_ravdess_shards(
    max_videos: Optional[int] = None,
    force_rebuild: bool = False,
    n_frames: int = 12,
    windows_per_video: int = 4,
    crop_fraction: float = 0.60,
):
    existing = sorted(RAVDESS_SHARDS.glob("*.npz"))
    if existing and not force_rebuild:
        print(f"RAVDESS shards already exist: {len(existing)} files in {RAVDESS_SHARDS}")
        return

    videos = find_ravdess_videos()
    if max_videos is not None:
        videos = videos[:max_videos]
    if not videos:
        raise RuntimeError("No RAVDESS mp4 videos found. Run download/extract first.")

    records = []
    failed = []
    total = len(videos)
    progress_line("RAVDESS feature extraction", 0, total, extra="kept=0 failed=0")

    crops = make_temporal_crops(
        n_crops=windows_per_video,
        crop_fraction=crop_fraction,
    )

    for i, vp in enumerate(videos, start=1):
        emotion_id = parse_ravdess_emotion(vp)
        actor_id = parse_ravdess_actor(vp)
        kept_for_video = 0

        for w_id, (start_frac, end_frac) in enumerate(crops):
            item = None
            err_short = ""

            try:
                item = extract_window_from_video(
                    vp,
                    n_frames=n_frames,
                    start_frac=start_frac,
                    end_frac=end_frac,
                )
            except Exception as e:
                err_short = repr(e)[:300]
                item = None

            if item is None:
                failed.append({
                    "video": str(vp),
                    "window_id": w_id,
                    "start_frac": start_frac,
                    "end_frac": end_frac,
                    "error_short": err_short if err_short else "no_valid_pose_or_too_few_frames",
                })
            else:
                face, body = item
                records.append({
                    "face": face,
                    "body": body,
                    "video_id": f"{vp.name}__w{w_id:02d}",
                    "subject_id": actor_id,
                    "emotion_id": emotion_id,
                    "emotion_name": RAVDESS_EMOTION_MAP.get(emotion_id, "unknown"),
                })
                kept_for_video += 1

        if i == 1 or i % 10 == 0 or i == total:
            progress_line(
                "RAVDESS feature extraction",
                i,
                total,
                extra=f"kept={len(records)} failed={len(failed)} last_video_kept={kept_for_video}"
            )

    fail_path = RESULTS_DIR / "ravdess_failed_videos.csv"
    pd.DataFrame(failed).to_csv(fail_path, index=False)

    clear_output(wait=True)
    print(f"RAVDESS extraction done | videos={total} | windows_kept={len(records)} | windows_failed={len(failed)}")
    print(f"Window settings | n_frames={n_frames} | windows_per_video={windows_per_video} | crop_fraction={crop_fraction}")
    print("Failed windows log:", fail_path)

    if not records:
        raise RuntimeError(f"No valid RAVDESS records extracted. Check {fail_path}")

    save_shards("RAVDESS", records, RAVDESS_SHARDS)


In [ ]:
# -----------------------------
# 6) LOAD SHARDS AND STANDARDIZATION
# -----------------------------

def load_shards(shard_dir: Path) -> Dict[str, np.ndarray]:
    files = sorted(Path(shard_dir).glob("*.npz"))
    if not files:
        raise RuntimeError(f"No shards found in {shard_dir}")
    faces, bodies, vids, subs, emos, emonames = [], [], [], [], [], []
    for f in files:
        z = np.load(f, allow_pickle=True)
        faces.append(z["face"])
        bodies.append(z["body"])
        vids.append(z["video_id"])
        subs.append(z["subject_id"])
        emos.append(z["emotion_id"])
        emonames.append(z["emotion_name"])
    return {
        "face": np.concatenate(faces, axis=0).astype(np.float32),
        "body": np.concatenate(bodies, axis=0).astype(np.float32),
        "video_id": np.concatenate(vids, axis=0),
        "subject_id": np.concatenate(subs, axis=0).astype(np.int32),
        "emotion_id": np.concatenate(emos, axis=0).astype(np.int32),
        "emotion_name": np.concatenate(emonames, axis=0),
    }


def ravdess_actor_split(data: Dict[str, np.ndarray], train_frac=0.70, val_frac=0.15):
    """Actor-independent split.
    If actors 1..12 are downloaded, this reproduces approximately the old 8/2/2 split.
    If actors 1..24 are downloaded, it uses more independent actors and helps address
    reviewer concerns about small effective sample size.
    """
    actors = data["subject_id"]
    unique_actors = np.array(sorted([int(a) for a in np.unique(actors) if int(a) > 0]))

    if len(unique_actors) >= 6:
        n = len(unique_actors)
        n_train = max(1, int(round(train_frac * n)))
        n_val = max(1, int(round(val_frac * n)))
        if n_train + n_val >= n:
            n_train = max(1, n - 2)
            n_val = 1
        train_actors = set(unique_actors[:n_train].tolist())
        val_actors = set(unique_actors[n_train:n_train + n_val].tolist())
        test_actors = set(unique_actors[n_train + n_val:].tolist())

        train_idx = np.array([i for i, a in enumerate(actors) if int(a) in train_actors], dtype=int)
        val_idx = np.array([i for i, a in enumerate(actors) if int(a) in val_actors], dtype=int)
        test_idx = np.array([i for i, a in enumerate(actors) if int(a) in test_actors], dtype=int)
    else:
        train_idx = val_idx = test_idx = np.array([], dtype=int)

    if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
        # fallback split if actor parsing failed
        rng = np.random.default_rng(SEED)
        idx = rng.permutation(len(actors))
        n = len(idx)
        train_idx = idx[:int(0.7*n)]
        val_idx = idx[int(0.7*n):int(0.85*n)]
        test_idx = idx[int(0.85*n):]
    return train_idx, val_idx, test_idx


def standardize_train_only(face, body, train_idx, val_idx, test_idx):
    T, Df = face.shape[1], face.shape[2]
    Db = body.shape[2]

    face_scaler = StandardScaler()
    body_scaler = StandardScaler()
    face_scaler.fit(face[train_idx].reshape(-1, Df))
    body_scaler.fit(body[train_idx].reshape(-1, Db))

    def tr_face(x):
        return face_scaler.transform(x.reshape(-1, Df)).reshape(x.shape).astype(np.float32)
    def tr_body(x):
        return body_scaler.transform(x.reshape(-1, Db)).reshape(x.shape).astype(np.float32)

    out = {
        "train_face": tr_face(face[train_idx]),
        "train_body": tr_body(body[train_idx]),
        "val_face": tr_face(face[val_idx]),
        "val_body": tr_body(body[val_idx]),
        "test_face": tr_face(face[test_idx]),
        "test_body": tr_body(body[test_idx]),
    }
    return out


In [ ]:
# -----------------------------
# 7) PMCI MODEL
# -----------------------------
class PMCIModel(nn.Module):
    def __init__(self, face_dim=30, body_dim=72, hidden=96, embed=96, K=8, tau=0.25):
        super().__init__()
        self.K = int(K)
        self.tau = float(tau)
        self.face_gru = nn.GRU(face_dim, hidden, batch_first=True)
        self.body_gru = nn.GRU(body_dim, hidden, batch_first=True)
        self.face_proj = nn.Sequential(nn.Linear(hidden, embed), nn.ReLU(), nn.Linear(embed, embed))
        self.body_proj = nn.Sequential(nn.Linear(hidden, embed), nn.ReLU(), nn.Linear(embed, embed))
        if self.K > 0:
            self.prototypes = nn.Parameter(torch.randn(self.K, embed) * 0.05)
        else:
            self.prototypes = None

    def encode(self, face, body):
        _, hf = self.face_gru(face)
        _, hb = self.body_gru(body)
        zf = self.face_proj(hf[-1])
        zb = self.body_proj(hb[-1])
        zf = F.normalize(zf, dim=-1)
        zb = F.normalize(zb, dim=-1)
        return zf, zb

    def score(self, face, body):
        zf, zb = self.encode(face, body)
        if self.K == 0:
            cos = F.cosine_similarity(zf, zb, dim=-1)
            return ((cos + 1.0) / 2.0).clamp(0.0, 1.0)
        proto = F.normalize(self.prototypes, dim=-1)
        sf = torch.matmul(zf, proto.T) / self.tau
        sb = torch.matmul(zb, proto.T) / self.tau
        pf = F.softmax(sf, dim=-1)
        pb = F.softmax(sb, dim=-1)
        m = 0.5 * (pf + pb)
        eps = 1e-8
        kl_f = torch.sum(pf * (torch.log(pf + eps) - torch.log(m + eps)), dim=-1)
        kl_b = torch.sum(pb * (torch.log(pb + eps) - torch.log(m + eps)), dim=-1)
        jsd = 0.5 * (kl_f + kl_b)
        jsd_norm = jsd / math.log(2.0)  # bounded approx [0,1]
        pmci = 1.0 - jsd_norm
        return pmci.clamp(0.0, 1.0)


def make_random_pair_batch(face_np, body_np, batch_size=64, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(face_np)
    half = batch_size // 2

    pos_idx = rng.integers(0, n, size=half)
    neg_i = rng.integers(0, n, size=batch_size - half)
    neg_j = rng.integers(0, n, size=batch_size - half)
    same = neg_i == neg_j
    neg_j[same] = (neg_j[same] + 1) % n

    f = np.concatenate([face_np[pos_idx], face_np[neg_i]], axis=0)
    b = np.concatenate([body_np[pos_idx], body_np[neg_j]], axis=0)
    y = np.concatenate([np.ones(half), np.zeros(batch_size - half)], axis=0).astype(np.float32)

    perm = rng.permutation(batch_size)
    return (
        torch.tensor(f[perm], dtype=torch.float32, device=DEVICE_TORCH),
        torch.tensor(b[perm], dtype=torch.float32, device=DEVICE_TORCH),
        torch.tensor(y[perm], dtype=torch.float32, device=DEVICE_TORCH),
    )


def train_model(train_face, train_body, val_face, val_body, K=8, seed=42, epochs=12, steps_per_epoch=80, batch_size=64):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)

    model = PMCIModel(K=K).to(DEVICE_TORCH)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_state = None
    best_auc = -1

    total_steps = epochs * steps_per_epoch
    step_counter = 0
    for ep in range(1, epochs + 1):
        model.train()
        losses = []
        for _ in range(steps_per_epoch):
            f, b, y = make_random_pair_batch(train_face, train_body, batch_size=batch_size, rng=rng)
            s = model.score(f, b)
            loss = F.binary_cross_entropy(s.clamp(1e-5, 1 - 1e-5), y)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))
            step_counter += 1

        val_auc = evaluate_random_auc(model, val_face, val_body, seed=seed + ep, max_pairs=512)["roc_auc"]
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        progress_line("Training", ep, epochs, extra=f"K={K} loss={np.mean(losses):.4f} val_auc={val_auc:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def score_pairs(model, face_np, body_np, batch_size=256):
    model.eval()

    # Fix for reversed arrays like x[:, ::-1, :]
    face_np = np.ascontiguousarray(face_np, dtype=np.float32)
    body_np = np.ascontiguousarray(body_np, dtype=np.float32)

    scores = []
    for start in range(0, len(face_np), batch_size):
        f = torch.from_numpy(face_np[start:start+batch_size]).to(DEVICE_TORCH, non_blocking=True)
        b = torch.from_numpy(body_np[start:start+batch_size]).to(DEVICE_TORCH, non_blocking=True)
        s = model.score(f, b).detach().cpu().numpy()
        scores.append(s)

    return np.concatenate(scores, axis=0) if scores else np.array([], dtype=np.float32)


def evaluate_random_auc(model, face_np, body_np, seed=42, max_pairs=None):
    rng = np.random.default_rng(seed)
    n = len(face_np)
    idx = np.arange(n)
    if max_pairs is not None and n > max_pairs:
        idx = rng.choice(idx, size=max_pairs, replace=False)
    n2 = len(idx)
    donor = rng.permutation(idx)
    same = donor == idx
    donor[same] = np.roll(donor, 1)[same]

    pos_scores = score_pairs(model, face_np[idx], body_np[idx])
    neg_scores = score_pairs(model, face_np[idx], body_np[donor])
    y = np.concatenate([np.ones(n2), np.zeros(n2)])
    s = np.concatenate([pos_scores, neg_scores])
    out = {
        "roc_auc": float(roc_auc_score(y, s)),
        "pr_auc": float(average_precision_score(y, s)),
        "pos_mean": float(np.mean(pos_scores)),
        "neg_mean": float(np.mean(neg_scores)),
    }
    return out


In [ ]:
# -----------------------------
# 8) REVIEWER-FOCUSED EXPERIMENTS
# -----------------------------

def make_partial_mismatch_body(body_np, frac, seed=42):
    rng = np.random.default_rng(seed)
    b = body_np.copy()
    n, T, D = b.shape
    donor = rng.permutation(n)
    same = donor == np.arange(n)
    donor[same] = np.roll(donor, 1)[same]
    n_replace = int(round(frac * T))
    if n_replace <= 0:
        return b
    for i in range(n):
        positions = rng.choice(T, size=n_replace, replace=False)
        b[i, positions, :] = body_np[donor[i], positions, :]
    return b


def graded_partial_mismatch_experiment(model, face_np, body_np, model_name, seed=42):
    rows = []
    fractions = [0.0, 0.25, 0.50, 0.75, 1.0]
    all_frac = []
    all_scores = []
    for frac in fractions:
        corrupted_body = make_partial_mismatch_body(body_np, frac, seed=seed + int(frac * 1000))
        scores = score_pairs(model, face_np, corrupted_body)
        rows.append({
            "model": model_name,
            "mismatch_fraction": frac,
            "mean_score": float(np.mean(scores)),
            "std_score": float(np.std(scores)),
            "median_score": float(np.median(scores)),
            "n": int(len(scores)),
        })
        all_frac.extend([frac] * len(scores))
        all_scores.extend(scores.tolist())

    rho, p = spearmanr(all_frac, all_scores)
    trend = {
        "model": model_name,
        "spearman_rho_fraction_vs_score": float(rho),
        "spearman_p": float(p),
        "interpretation": "negative rho supports graded consistency degradation"
    }
    return pd.DataFrame(rows), trend


def time_reversal_controls(model, face_np, body_np, model_name, seed=42):
    rng = np.random.default_rng(seed)
    n = len(face_np)
    donor = rng.permutation(n)
    same = donor == np.arange(n)
    donor[same] = np.roll(donor, 1)[same]

    conditions = {
        "true_pair": (face_np, body_np, 1),
        "random_body": (face_np, body_np[donor], 0),
        "face_time_reverse": (face_np[:, ::-1, :], body_np, 0),
        "body_time_reverse": (face_np, body_np[:, ::-1, :], 0),
        "both_time_reverse": (face_np[:, ::-1, :], body_np[:, ::-1, :], 0),
        "face_zero": (np.zeros_like(face_np), body_np, 0),
        "body_zero": (face_np, np.zeros_like(body_np), 0),
    }

    rows = []
    true_scores = None
    for cond, (f, b, label) in conditions.items():
        scores = score_pairs(model, f, b)
        if cond == "true_pair":
            true_scores = scores
        rows.append({
            "model": model_name,
            "condition": cond,
            "mean_score": float(np.mean(scores)),
            "std_score": float(np.std(scores)),
            "median_score": float(np.median(scores)),
            "n": int(len(scores)),
        })

    auc_rows = []
    for cond, (f, b, label) in conditions.items():
        if cond == "true_pair":
            continue
        disrupted_scores = score_pairs(model, f, b)
        y = np.concatenate([np.ones_like(true_scores), np.zeros_like(disrupted_scores)])
        s = np.concatenate([true_scores, disrupted_scores])
        auc_rows.append({
            "model": model_name,
            "comparison": f"true_pair_vs_{cond}",
            "roc_auc": float(roc_auc_score(y, s)),
            "true_mean": float(np.mean(true_scores)),
            "disrupted_mean": float(np.mean(disrupted_scores)),
        })
    return pd.DataFrame(rows), pd.DataFrame(auc_rows)


In [ ]:
# -----------------------------
# 9) RUN EVERYTHING
# -----------------------------
RUN_DOWNLOAD = True
RUN_EXTRACT = True
FORCE_REBUILD_SHARDS = True     # set False if shards already created
MAX_VIDEOS = None               # for quick test use e.g. 80; for paper use None

# Multi-window revision settings:
# keep each sequence at 12 frames, but extract several temporal crops per source video.
WINDOW_FRAMES = 12
WINDOWS_PER_VIDEO = 4
CROP_FRACTION = 0.60

# For exact reproduction of the older 672-window setup use RAVDESS_ACTOR_END = 12.
# For the revision, 24 is recommended because it increases independent actors/windows.
RAVDESS_ACTOR_START = 1
RAVDESS_ACTOR_END = 24

K_LIST = [0, 8, 16, 32]         # K=0 direct cosine baseline, K>0 PMCI
EPOCHS = 12                     # for quick debug use 3; for paper use 12-20
STEPS_PER_EPOCH = 80            # for quick debug use 20; for paper use 80+

if RUN_DOWNLOAD:
    download_ravdess_actors(RAVDESS_ACTOR_START, RAVDESS_ACTOR_END)
if RUN_EXTRACT:
    extract_ravdess_zips()

videos = find_ravdess_videos()
clear_output(wait=True)
print("Found RAVDESS videos:", len(videos))

build_ravdess_shards(
    max_videos=MAX_VIDEOS,
    force_rebuild=FORCE_REBUILD_SHARDS,
    n_frames=WINDOW_FRAMES,
    windows_per_video=WINDOWS_PER_VIDEO,
    crop_fraction=CROP_FRACTION,
)

# Load shards
data = load_shards(RAVDESS_SHARDS)
face, body = data["face"], data["body"]
train_idx, val_idx, test_idx = ravdess_actor_split(data)
std = standardize_train_only(face, body, train_idx, val_idx, test_idx)

clear_output(wait=True)
print("Loaded standardized data")
print("face:", face.shape, "body:", body.shape)
print("window settings:", {"frames": WINDOW_FRAMES, "windows_per_video": WINDOWS_PER_VIDEO, "crop_fraction": CROP_FRACTION})
print("train/val/test:", len(train_idx), len(val_idx), len(test_idx))
print("actors train:", sorted(set(data["subject_id"][train_idx].tolist())))
print("actors val:", sorted(set(data["subject_id"][val_idx].tolist())))
print("actors test:", sorted(set(data["subject_id"][test_idx].tolist())))

all_standard_rows = []
all_graded_rows = []
all_graded_trends = []
all_control_rows = []
all_control_auc_rows = []

trained_models = {}

for K in K_LIST:
    model_name = "DirectCosine_K0" if K == 0 else f"PMCI_K{K}"
    clear_output(wait=True)
    print(f"Training {model_name}...")
    model = train_model(
        std["train_face"], std["train_body"],
        std["val_face"], std["val_body"],
        K=K,
        seed=SEED + K,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        batch_size=64,
    )
    trained_models[model_name] = model

    # Standard exact pair matching AUC
    standard = evaluate_random_auc(model, std["test_face"], std["test_body"], seed=1000 + K)
    standard["model"] = model_name
    all_standard_rows.append(standard)

    # Graded partial mismatch
    gdf, trend = graded_partial_mismatch_experiment(model, std["test_face"], std["test_body"], model_name, seed=2000 + K)
    all_graded_rows.append(gdf)
    all_graded_trends.append(trend)

    # Time reversal and modality controls
    cdf, aucdf = time_reversal_controls(model, std["test_face"], std["test_body"], model_name, seed=3000 + K)
    all_control_rows.append(cdf)
    all_control_auc_rows.append(aucdf)

    # Save model
    torch.save(model.state_dict(), RESULTS_DIR / f"{model_name}.pt")

    # Free memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save all results
standard_df = pd.DataFrame(all_standard_rows)
graded_df = pd.concat(all_graded_rows, ignore_index=True)
graded_trend_df = pd.DataFrame(all_graded_trends)
control_df = pd.concat(all_control_rows, ignore_index=True)
control_auc_df = pd.concat(all_control_auc_rows, ignore_index=True)

standard_df.to_csv(RESULTS_DIR / "standard_pair_matching_auc.csv", index=False)
graded_df.to_csv(RESULTS_DIR / "graded_partial_mismatch_scores.csv", index=False)
graded_trend_df.to_csv(RESULTS_DIR / "graded_partial_mismatch_trend.csv", index=False)
control_df.to_csv(RESULTS_DIR / "time_reversal_and_modality_control_scores.csv", index=False)
control_auc_df.to_csv(RESULTS_DIR / "time_reversal_and_modality_control_auc.csv", index=False)

# Optional simple plots without custom colors
import matplotlib.pyplot as plt

for model_name in graded_df["model"].unique():
    sub = graded_df[graded_df["model"] == model_name].sort_values("mismatch_fraction")
    plt.figure(figsize=(6, 4))
    plt.errorbar(sub["mismatch_fraction"], sub["mean_score"], yerr=sub["std_score"], marker="o", capsize=4)
    plt.xlabel("Body-stream replacement fraction")
    plt.ylabel("Mean consistency score")
    plt.title(f"Graded partial mismatch: {model_name}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"graded_partial_mismatch_{model_name}.png", dpi=200)
    plt.close()

clear_output(wait=True)
print("DONE ✅")
print("Results saved to:", RESULTS_DIR)
print("\nStandard pair matching:")
print(standard_df)
print("\nGraded trend: negative Spearman rho means score decreases as mismatch increases.")
print(graded_trend_df)
print("\nControl AUC:")
print(control_auc_df)


DONE ✅
Results saved to: /content/cias_q1_min/results_pmci_reviewer_continuation_quiet

Standard pair matching:
    roc_auc    pr_auc  pos_mean  neg_mean            model
0  0.942861  0.916673  0.822540  0.495836  DirectCosine_K0
1  0.851373  0.813749  0.715331  0.364821          PMCI_K8
2  0.805214  0.737366  0.743179  0.409422         PMCI_K16
3  0.886568  0.846973  0.823251  0.397192         PMCI_K32

Graded trend: negative Spearman rho means score decreases as mismatch increases.
             model  spearman_rho_fraction_vs_score     spearman_p  \
0  DirectCosine_K0                       -0.623103   0.000000e+00   
1          PMCI_K8                       -0.450129   0.000000e+00   
2         PMCI_K16                       -0.420835  4.767961e-307   
3         PMCI_K32                       -0.537171   0.000000e+00   

                                      interpretation  
0  negative rho supports graded consistency degra...  
1  negative rho supports graded consistency degra...  


# PMCI Revision

In [ ]:
# ============================================================
# CELL 1 — Imports + configuration
# ============================================================
import os, gc, time, math, json, warnings, copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display, clear_output

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

DEVICE_TORCH = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Output folder is created fresh; no previous files are required.
REV_DIR = Path("/content/pmci_revision_results_v6")
REV_DIR.mkdir(parents=True, exist_ok=True)

# Fast/Final switches
TRAIN_CORE_IF_MISSING = True          # train K=0/8/16/32 if trained_models is absent
TRAIN_MISSING_K = True                # train K=4/64 for K-ablation if absent
TRAIN_INIT_ROBUSTNESS = False         # quick=False; final=True
TRAIN_TAU_ROBUSTNESS = False          # quick=False; final=True
TRAIN_K_MULTI_SEED_FINAL = False      # quick=False; final=True

# Reviewer settings
SEEDS = [42, 43, 44]
CORE_K = [0, 8, 16, 32]
K_VALUES_EXT = [4, 8, 16, 32, 64]
TAU_VALUES = [0.10, 0.25, 0.50, 1.00, 2.00, 4.00]
TAU_TRAIN_VALUES = [0.25, 0.50, 1.00, 2.00]
INIT_MODES = ["normal", "uniform", "xavier", "orthogonal"]

# To make quick run not explode in Colab
EXTRA_EPOCHS = 6
EXTRA_STEPS_PER_EPOCH = 50
EXTRA_BATCH_SIZE = 64
BATCH_SIZE = 256

MAX_RANDOM_EVAL = None                # None = full test split
MAX_HARD_ANCHORS = 240
HARD_NEGATIVES_PER_ANCHOR = 20
N_SHUFFLE_CONTROLS = 20               # final: 100
N_GROUP_BOOT = 200                    # final: 1000

print("Output:", REV_DIR)
print("Device:", DEVICE_TORCH)


Output: /content/pmci_revision_results_v6
Device: cuda


In [ ]:
# ============================================================
# CELL 2 — Runtime resolver: use memory only, no file loading
# ============================================================
def _as_np_float32(x):
    return np.asarray(x, dtype=np.float32)


def standardize_train_only_v6(face, body, train_idx, val_idx, test_idx):
    """Fallback standardization if base notebook has raw face/body arrays but no std dict."""
    face = _as_np_float32(face)
    body = _as_np_float32(body)
    Df = face.shape[2]
    Db = body.shape[2]

    face_scaler = StandardScaler()
    body_scaler = StandardScaler()
    face_scaler.fit(face[train_idx].reshape(-1, Df))
    body_scaler.fit(body[train_idx].reshape(-1, Db))

    def tr_face(x):
        x = _as_np_float32(x)
        return face_scaler.transform(x.reshape(-1, Df)).reshape(x.shape).astype(np.float32)

    def tr_body(x):
        x = _as_np_float32(x)
        return body_scaler.transform(x.reshape(-1, Db)).reshape(x.shape).astype(np.float32)

    return {
        "train_face": tr_face(face[train_idx]),
        "train_body": tr_body(body[train_idx]),
        "val_face": tr_face(face[val_idx]),
        "val_body": tr_body(body[val_idx]),
        "test_face": tr_face(face[test_idx]),
        "test_body": tr_body(body[test_idx]),
    }


# If std is missing but raw arrays exist, reconstruct it from memory.
if "std" not in globals():
    raw_needed = ["face", "body", "train_idx", "val_idx", "test_idx"]
    if all(name in globals() for name in raw_needed):
        print("std not found; reconstructing std from in-memory face/body/splits...")
        std = standardize_train_only_v6(face, body, train_idx, val_idx, test_idx)
    else:
        std = None


# If PMCIModel is missing, define a compatible fallback.
if "PMCIModel" not in globals():
    print("PMCIModel not found; defining compatible fallback PMCIModel...")
    class PMCIModel(nn.Module):
        def __init__(self, face_dim=30, body_dim=72, hidden=96, embed=96, K=8, tau=0.25):
            super().__init__()
            self.K = int(K)
            self.tau = float(tau)
            self.face_gru = nn.GRU(face_dim, hidden, batch_first=True)
            self.body_gru = nn.GRU(body_dim, hidden, batch_first=True)
            self.face_proj = nn.Sequential(nn.Linear(hidden, embed), nn.ReLU(), nn.Linear(embed, embed))
            self.body_proj = nn.Sequential(nn.Linear(hidden, embed), nn.ReLU(), nn.Linear(embed, embed))
            self.prototypes = nn.Parameter(torch.randn(self.K, embed) * 0.05) if self.K > 0 else None

        def encode(self, face, body):
            _, hf = self.face_gru(face)
            _, hb = self.body_gru(body)
            zf = F.normalize(self.face_proj(hf[-1]), dim=-1)
            zb = F.normalize(self.body_proj(hb[-1]), dim=-1)
            return zf, zb

        def score(self, face, body):
            zf, zb = self.encode(face, body)
            if self.K == 0:
                cos = F.cosine_similarity(zf, zb, dim=-1)
                return ((cos + 1.0) / 2.0).clamp(0.0, 1.0)
            proto = F.normalize(self.prototypes, dim=-1)
            pf = F.softmax((zf @ proto.T) / self.tau, dim=-1)
            pb = F.softmax((zb @ proto.T) / self.tau, dim=-1)
            m = 0.5 * (pf + pb)
            eps = 1e-8
            kl_f = torch.sum(pf * (torch.log(pf + eps) - torch.log(m + eps)), dim=-1)
            kl_b = torch.sum(pb * (torch.log(pb + eps) - torch.log(m + eps)), dim=-1)
            jsd = 0.5 * (kl_f + kl_b)
            pmci = 1.0 - jsd / math.log(2.0)
            return pmci.clamp(0.0, 1.0)


# If trained_models is missing, create empty dict; core models can be trained below.
if "trained_models" not in globals() or trained_models is None:
    trained_models = {}


def prepare_runtime_or_raise():
    """Final validation before experiments. No file search/loading here."""
    global train_face, train_body, val_face, val_body, test_face, test_body
    if std is None:
        raise RuntimeError(
            "No in-memory data found. Run your base loading/preprocessing script first. "
            "This extension needs std, or raw face/body/train_idx/val_idx/test_idx. "
            "It does not load files from /content."
        )
    required_std = ["train_face", "train_body", "val_face", "val_body", "test_face", "test_body"]
    missing = [k for k in required_std if k not in std]
    if missing:
        raise RuntimeError(f"std exists but lacks keys: {missing}")

    train_face = np.ascontiguousarray(std["train_face"].astype(np.float32))
    train_body = np.ascontiguousarray(std["train_body"].astype(np.float32))
    val_face = np.ascontiguousarray(std["val_face"].astype(np.float32))
    val_body = np.ascontiguousarray(std["val_body"].astype(np.float32))
    test_face = np.ascontiguousarray(std["test_face"].astype(np.float32))
    test_body = np.ascontiguousarray(std["test_body"].astype(np.float32))

    print("Runtime OK")
    print("train:", train_face.shape, train_body.shape)
    print("val:  ", val_face.shape, val_body.shape)
    print("test: ", test_face.shape, test_body.shape)
    print("available models:", sorted(list(trained_models.keys())))

prepare_runtime_or_raise()


Runtime OK
train: (8160, 12, 30) (8160, 12, 72)
val:   (1920, 12, 30) (1920, 12, 72)
test:  (1440, 12, 30) (1440, 12, 72)
available models: ['DirectCosine_K0', 'PMCI_K16', 'PMCI_K32', 'PMCI_K8']


In [ ]:
# ============================================================
# CELL 3 — Scoring, training and metrics helpers
# ============================================================
def model_key_for_K(K):
    return "DirectCosine_K0" if int(K) == 0 else f"PMCI_K{int(K)}"


def get_model_by_K(K):
    key = model_key_for_K(K)
    m = trained_models.get(key)
    if m is None:
        return None
    return m.to(DEVICE_TORCH).eval()


def safe_metric(fn, y, s, default=np.nan):
    try:
        if len(np.unique(y)) < 2 and fn in [roc_auc_score, average_precision_score]:
            return float(default)
        return float(fn(y, s))
    except Exception:
        return float(default)


def threshold_metrics(y, s, threshold=0.5):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    yhat = (s >= threshold).astype(int)
    return {
        "accuracy": safe_metric(accuracy_score, y, yhat),
        "balanced_accuracy": safe_metric(balanced_accuracy_score, y, yhat),
        "f1": safe_metric(f1_score, y, yhat),
        "mcc": safe_metric(matthews_corrcoef, y, yhat),
    }


def save_csv(df, name):
    path = REV_DIR / name
    df.to_csv(path, index=False)
    print("saved:", path)
    return path


@torch.no_grad()
def score_pairs_safe(model, face_np, body_np, batch_size=BATCH_SIZE):
    model.eval()
    face_np = np.ascontiguousarray(face_np.astype(np.float32))
    body_np = np.ascontiguousarray(body_np.astype(np.float32))
    out = []
    for st in range(0, len(face_np), batch_size):
        f = torch.tensor(face_np[st:st+batch_size], dtype=torch.float32, device=DEVICE_TORCH)
        b = torch.tensor(body_np[st:st+batch_size], dtype=torch.float32, device=DEVICE_TORCH)
        s = model.score(f, b).detach().cpu().numpy()
        out.append(s)
    return np.concatenate(out, axis=0)


def make_random_pair_batch_v6(face_np, body_np, batch_size=64, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(face_np)
    half = batch_size // 2
    pos_idx = rng.integers(0, n, size=half)
    neg_i = rng.integers(0, n, size=batch_size - half)
    neg_j = rng.integers(0, n, size=batch_size - half)
    same = neg_i == neg_j
    neg_j[same] = (neg_j[same] + 1) % n
    f = np.concatenate([face_np[pos_idx], face_np[neg_i]], axis=0)
    b = np.concatenate([body_np[pos_idx], body_np[neg_j]], axis=0)
    y = np.concatenate([np.ones(half), np.zeros(batch_size - half)]).astype(np.float32)
    perm = rng.permutation(batch_size)
    return (
        torch.tensor(np.ascontiguousarray(f[perm]), dtype=torch.float32, device=DEVICE_TORCH),
        torch.tensor(np.ascontiguousarray(b[perm]), dtype=torch.float32, device=DEVICE_TORCH),
        torch.tensor(y[perm], dtype=torch.float32, device=DEVICE_TORCH),
    )


def initialize_prototypes(model, mode="normal", seed=42):
    if getattr(model, "K", 0) <= 0 or getattr(model, "prototypes", None) is None:
        return model
    torch.manual_seed(seed)
    with torch.no_grad():
        p = model.prototypes
        if mode == "normal":
            p.normal_(0.0, 0.05)
        elif mode == "uniform":
            p.uniform_(-0.05, 0.05)
        elif mode == "xavier":
            nn.init.xavier_uniform_(p)
        elif mode == "orthogonal":
            tmp = torch.empty_like(p)
            nn.init.orthogonal_(tmp)
            p.copy_(tmp * 0.05)
        else:
            raise ValueError(f"Unknown init mode: {mode}")
    return model


def train_model_custom(K=8, tau=0.25, seed=42, init_mode="normal", epochs=EXTRA_EPOCHS,
                       steps_per_epoch=EXTRA_STEPS_PER_EPOCH, batch_size=EXTRA_BATCH_SIZE,
                       return_history=False):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)

    face_dim = train_face.shape[-1]
    body_dim = train_body.shape[-1]
    model = PMCIModel(face_dim=face_dim, body_dim=body_dim, K=K, tau=tau).to(DEVICE_TORCH)
    model = initialize_prototypes(model, mode=init_mode, seed=seed)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_state, best_auc = None, -1
    hist = []
    for ep in range(1, epochs + 1):
        model.train()
        losses = []
        for _ in range(steps_per_epoch):
            f, b, y = make_random_pair_batch_v6(train_face, train_body, batch_size=batch_size, rng=rng)
            s = model.score(f, b)
            loss = F.binary_cross_entropy(s.clamp(1e-5, 1-1e-5), y)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(float(loss.detach().cpu()))

        # validation random AUC
        f_val, b_val, y_val, *_ = make_random_pair_arrays(val_face, val_body, seed=seed + ep, max_anchors=min(300, len(val_face)))
        val_s = score_pairs_safe(model, f_val, b_val)
        val_auc = safe_metric(roc_auc_score, y_val, val_s)
        hist.append({"epoch": ep, "K": K, "tau": tau, "init_mode": init_mode,
                     "loss": float(np.mean(losses)), "val_auc": val_auc})
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    if return_history:
        return model, pd.DataFrame(hist)
    return model


def ensure_model(K, tau=0.25, seed=42, force_retrain=False):
    key = model_key_for_K(K)
    if (not force_retrain) and key in trained_models:
        return trained_models[key].to(DEVICE_TORCH).eval()
    print(f"Training missing model {key} ...")
    model = train_model_custom(K=K, tau=tau, seed=seed)
    trained_models[key] = model
    return model


def ensure_core_models():
    if not TRAIN_CORE_IF_MISSING:
        return
    for K in CORE_K:
        ensure_model(K, seed=1000 + K)

ensure_core_models()
print("models after ensure:", sorted(trained_models.keys()))


models after ensure: ['DirectCosine_K0', 'PMCI_K16', 'PMCI_K32', 'PMCI_K8']


In [ ]:
# ============================================================
# CELL 4 — Pair banks, hard negatives, grouped bootstrap
# ============================================================
def make_random_pair_arrays(face_np, body_np, seed=42, max_anchors=None):
    rng = np.random.default_rng(seed)
    n = len(face_np)
    idx = np.arange(n)
    if max_anchors is not None and n > max_anchors:
        idx = rng.choice(idx, size=max_anchors, replace=False)
    donor = rng.permutation(idx)
    same = donor == idx
    if np.any(same):
        donor[same] = np.roll(donor, 1)[same]
    f = np.concatenate([face_np[idx], face_np[idx]], axis=0)
    b = np.concatenate([body_np[idx], body_np[donor]], axis=0)
    y = np.concatenate([np.ones(len(idx)), np.zeros(len(idx))]).astype(int)
    anchor_id = np.concatenate([idx, idx])
    pair_type = np.array(["genuine"] * len(idx) + ["random_mismatch"] * len(idx))
    return f, b, y, anchor_id, pair_type


def evaluate_score_on_random_pairs(model, face_np, body_np, seed=42, max_anchors=None):
    f, b, y, anchor_id, pair_type = make_random_pair_arrays(face_np, body_np, seed=seed, max_anchors=max_anchors)
    s = score_pairs_safe(model, f, b)
    out = {
        "roc_auc": safe_metric(roc_auc_score, y, s),
        "pr_auc": safe_metric(average_precision_score, y, s),
        "score_mean": float(np.mean(s)),
        "score_std": float(np.std(s)),
        "pos_mean": float(np.mean(s[y == 1])),
        "neg_mean": float(np.mean(s[y == 0])),
    }
    out.update(threshold_metrics(y, s, threshold=0.5))
    return out


def make_direct_hard_negative_bank(direct_model, face_np, body_np,
                                   max_anchors=MAX_HARD_ANCHORS,
                                   hard_k=HARD_NEGATIVES_PER_ANCHOR,
                                   seed=42):
    """Clean version: 1 genuine + hard_k hard negatives per anchor; no repeated positive copies."""
    rng = np.random.default_rng(seed)
    n = len(face_np)
    anchors = np.arange(n)
    if max_anchors is not None and n > max_anchors:
        anchors = rng.choice(anchors, size=max_anchors, replace=False)

    f_list, b_list, y_list, anchor_list, pair_types = [], [], [], [], []
    body_all = np.ascontiguousarray(body_np)

    for a in anchors:
        # genuine
        f_list.append(face_np[a:a+1])
        b_list.append(body_np[a:a+1])
        y_list.append(np.array([1], dtype=int))
        anchor_list.append(np.array([a], dtype=int))
        pair_types.append(np.array(["genuine"]))

        # hard negatives among all non-self body streams
        face_rep = np.repeat(face_np[a:a+1], repeats=n, axis=0)
        scores = score_pairs_safe(direct_model, face_rep, body_all, batch_size=BATCH_SIZE)
        scores[a] = -np.inf
        hk = min(hard_k, n - 1)
        hard_idx = np.argpartition(scores, -hk)[-hk:]
        hard_idx = hard_idx[np.argsort(scores[hard_idx])[::-1]]

        f_list.append(np.repeat(face_np[a:a+1], repeats=len(hard_idx), axis=0))
        b_list.append(body_np[hard_idx])
        y_list.append(np.zeros(len(hard_idx), dtype=int))
        anchor_list.append(np.full(len(hard_idx), a, dtype=int))
        pair_types.append(np.array(["direct_hard_negative"] * len(hard_idx)))

    return (
        np.concatenate(f_list, axis=0),
        np.concatenate(b_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(anchor_list, axis=0),
        np.concatenate(pair_types, axis=0),
    )


def grouped_bootstrap_auc(y, s, groups, n_boot=N_GROUP_BOOT, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    s = np.asarray(s)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    vals = []
    for _ in range(n_boot):
        sampled = rng.choice(uniq, size=len(uniq), replace=True)
        mask = np.concatenate([np.where(groups == g)[0] for g in sampled])
        if len(np.unique(y[mask])) < 2:
            continue
        vals.append(roc_auc_score(y[mask], s[mask]))
    if len(vals) == 0:
        return {"auc_boot_mean": np.nan, "auc_boot_lo": np.nan, "auc_boot_hi": np.nan, "auc_boot_n": 0}
    vals = np.asarray(vals)
    return {
        "auc_boot_mean": float(np.mean(vals)),
        "auc_boot_lo": float(np.percentile(vals, 2.5)),
        "auc_boot_hi": float(np.percentile(vals, 97.5)),
        "auc_boot_n": int(len(vals)),
    }

print("Pair-bank helpers ready")


Pair-bank helpers ready


In [ ]:
# ============================================================
# CELL 5 — Prototype probabilities, tau, K, utilization
# ============================================================
@torch.no_grad()
def get_proto_probs(model, face_np, body_np, batch_size=BATCH_SIZE, tau_override=None):
    if getattr(model, "K", 0) <= 0 or getattr(model, "prototypes", None) is None:
        raise ValueError("Prototype probabilities exist only for K > 0 PMCI models.")
    model.eval()
    tau = float(tau_override if tau_override is not None else model.tau)
    all_pf, all_pb = [], []
    for st in range(0, len(face_np), batch_size):
        f = torch.tensor(np.ascontiguousarray(face_np[st:st+batch_size]), dtype=torch.float32, device=DEVICE_TORCH)
        b = torch.tensor(np.ascontiguousarray(body_np[st:st+batch_size]), dtype=torch.float32, device=DEVICE_TORCH)
        zf, zb = model.encode(f, b)
        proto = F.normalize(model.prototypes, dim=-1)
        pf = F.softmax((zf @ proto.T) / tau, dim=-1)
        pb = F.softmax((zb @ proto.T) / tau, dim=-1)
        all_pf.append(pf.detach().cpu().numpy())
        all_pb.append(pb.detach().cpu().numpy())
    return np.concatenate(all_pf, axis=0), np.concatenate(all_pb, axis=0)


def score_pmci_with_tau(model, face_np, body_np, tau_override, batch_size=BATCH_SIZE):
    pf, pb = get_proto_probs(model, face_np, body_np, batch_size=batch_size, tau_override=tau_override)
    m = 0.5 * (pf + pb)
    eps = 1e-12
    kl_f = np.sum(pf * (np.log(pf + eps) - np.log(m + eps)), axis=-1)
    kl_b = np.sum(pb * (np.log(pb + eps) - np.log(m + eps)), axis=-1)
    jsd = 0.5 * (kl_f + kl_b)
    return np.clip(1.0 - jsd / np.log(2.0), 0.0, 1.0)


def entropy_nats(p, eps=1e-12):
    return -np.sum(p * np.log(p + eps), axis=-1)


def prototype_utilization_summary(model, face_np, body_np, dataset="test", mode="genuine"):
    pf, pb = get_proto_probs(model, face_np, body_np)
    K = int(model.K)
    Hf, Hb = entropy_nats(pf), entropy_nats(pb)
    usage_f = np.bincount(np.argmax(pf, axis=1), minlength=K) / len(pf)
    usage_b = np.bincount(np.argmax(pb, axis=1), minlength=K) / len(pb)
    return {
        "dataset": dataset,
        "mode": mode,
        "model": model_key_for_K(K),
        "K": K,
        "H_face_mean": float(np.mean(Hf)),
        "H_body_mean": float(np.mean(Hb)),
        "H_face_norm": float(np.mean(Hf) / math.log(K)),
        "H_body_norm": float(np.mean(Hb) / math.log(K)),
        "effective_proto_face": float(np.exp(np.mean(Hf))),
        "effective_proto_body": float(np.exp(np.mean(Hb))),
        "max_proto_share_face": float(np.max(usage_f)),
        "max_proto_share_body": float(np.max(usage_b)),
        "argmax_proto_agreement": float(np.mean(np.argmax(pf, axis=1) == np.argmax(pb, axis=1))),
        "prob_dot_agreement_mean": float(np.mean(np.sum(pf * pb, axis=1))),
    }


def run_tau_sensitivity(K_list=(8,16,32), tau_values=TAU_VALUES):
    rows = []
    f, b, y, anchor_id, pair_type = make_random_pair_arrays(test_face, test_body, seed=1101, max_anchors=MAX_RANDOM_EVAL)
    for K in K_list:
        model = get_model_by_K(K)
        if model is None:
            print("skip tau, missing", model_key_for_K(K))
            continue
        for tau in tau_values:
            s = score_pmci_with_tau(model, f, b, tau_override=tau)
            row = {"experiment": "tau_sensitivity_eval", "K": K, "tau": tau,
                   "roc_auc": safe_metric(roc_auc_score, y, s),
                   "pr_auc": safe_metric(average_precision_score, y, s),
                   "score_mean": float(np.mean(s)), "score_std": float(np.std(s)),
                   "pos_mean": float(np.mean(s[y==1])), "neg_mean": float(np.mean(s[y==0]))}
            row.update(threshold_metrics(y, s))
            rows.append(row)
    df = pd.DataFrame(rows)
    save_csv(df, "01_tau_sensitivity_eval.csv")
    display(df)
    return df


def run_tau_training_sensitivity(K_values=(8,16), tau_values=TAU_TRAIN_VALUES):
    if not TRAIN_TAU_ROBUSTNESS:
        print("TRAIN_TAU_ROBUSTNESS=False, skipping tau retraining")
        return pd.DataFrame(), pd.DataFrame()
    rows, hists = [], []
    for K in K_values:
        for tau in tau_values:
            for seed in SEEDS:
                model, hist = train_model_custom(K=K, tau=tau, seed=seed, return_history=True)
                res = evaluate_score_on_random_pairs(model, test_face, test_body, seed=3000+seed+K)
                rows.append({"experiment":"tau_training_sensitivity", "K":K, "tau":tau, "seed":seed, **res})
                hist["seed"] = seed
                hists.append(hist)
    df = pd.DataFrame(rows)
    hdf = pd.concat(hists, ignore_index=True) if hists else pd.DataFrame()
    save_csv(df, "01b_tau_training_sensitivity.csv")
    save_csv(hdf, "01c_tau_training_history.csv")
    display(df)
    return df, hdf


def run_extended_K_ablation(K_values=K_VALUES_EXT):
    rows, util_rows = [], []
    for K in K_values:
        if get_model_by_K(K) is None and TRAIN_MISSING_K:
            ensure_model(K, seed=2000+K)
        model = get_model_by_K(K)
        if model is None:
            print("skip K missing", K)
            continue
        res = evaluate_score_on_random_pairs(model, test_face, test_body, seed=1200+K, max_anchors=MAX_RANDOM_EVAL)
        rows.append({"experiment":"extended_K_ablation", "K":K, "model":model_key_for_K(K), **res})
        if K > 0:
            util_rows.append(prototype_utilization_summary(model, test_face, test_body))
    df = pd.DataFrame(rows)
    udf = pd.DataFrame(util_rows)
    save_csv(df, "02_extended_K_ablation.csv")
    save_csv(udf, "03_prototype_utilization.csv")
    display(df)
    display(udf)
    return df, udf


def run_K_multiseed_final(K_values=K_VALUES_EXT):
    if not TRAIN_K_MULTI_SEED_FINAL:
        print("TRAIN_K_MULTI_SEED_FINAL=False, skipping multi-seed K final")
        return pd.DataFrame(), pd.DataFrame()
    rows, util_rows = [], []
    for K in K_values:
        for seed in SEEDS:
            model = train_model_custom(K=K, tau=0.25, seed=seed)
            res = evaluate_score_on_random_pairs(model, test_face, test_body, seed=4000+seed+K)
            rows.append({"experiment":"K_multiseed_final", "K":K, "seed":seed, **res})
            if K > 0:
                u = prototype_utilization_summary(model, test_face, test_body)
                u["seed"] = seed
                util_rows.append(u)
    df = pd.DataFrame(rows); udf = pd.DataFrame(util_rows)
    save_csv(df, "02b_K_multiseed_final.csv")
    save_csv(udf, "03b_K_multiseed_utilization.csv")
    display(df)
    return df, udf

print("Tau/K/prototype functions ready")


Tau/K/prototype functions ready


In [ ]:
# ============================================================
# CELL 6 — Initialization robustness, jitter/dropout, latency
# ============================================================
def run_initialization_robustness(K_values=(8,16), init_modes=INIT_MODES):
    if not TRAIN_INIT_ROBUSTNESS:
        print("TRAIN_INIT_ROBUSTNESS=False, skipping initialization robustness")
        return pd.DataFrame(), pd.DataFrame()
    rows, hists = [], []
    for K in K_values:
        for init_mode in init_modes:
            for seed in SEEDS:
                model, hist = train_model_custom(K=K, tau=0.25, seed=seed, init_mode=init_mode, return_history=True)
                res = evaluate_score_on_random_pairs(model, test_face, test_body, seed=5000+K+seed)
                rows.append({"experiment":"initialization_robustness", "K":K, "init_mode":init_mode, "seed":seed, **res})
                hist["init_mode"] = init_mode; hist["seed"] = seed
                hists.append(hist)
    df = pd.DataFrame(rows)
    hdf = pd.concat(hists, ignore_index=True) if hists else pd.DataFrame()
    save_csv(df, "04_initialization_robustness.csv")
    save_csv(hdf, "04b_initialization_training_history.csv")
    display(df)
    return df, hdf


def perturb_features(face_np, body_np, mode="noise_1pct", seed=42):
    rng = np.random.default_rng(seed)
    f = np.array(face_np, copy=True)
    b = np.array(body_np, copy=True)
    if mode.startswith("noise_"):
        pct = float(mode.split("_")[1].replace("pct", "")) / 100.0
        f += rng.normal(0, pct, size=f.shape).astype(np.float32)
        b += rng.normal(0, pct, size=b.shape).astype(np.float32)
    elif mode.startswith("dropout_"):
        pct = float(mode.split("_")[1].replace("pct", "")) / 100.0
        mf = rng.random(f.shape) < pct
        mb = rng.random(b.shape) < pct
        f[mf] = 0.0; b[mb] = 0.0
    elif mode.startswith("keypoint_dropout_"):
        pct = float(mode.split("_")[-1].replace("pct", "")) / 100.0
        # structured feature dropout: zero whole coordinate channels across all frames
        nf = max(1, int(round(f.shape[-1] * pct)))
        nb = max(1, int(round(b.shape[-1] * pct)))
        fi = rng.choice(np.arange(f.shape[-1]), size=nf, replace=False)
        bi = rng.choice(np.arange(b.shape[-1]), size=nb, replace=False)
        f[:, :, fi] = 0.0; b[:, :, bi] = 0.0
    elif mode == "body_shift_1":
        b = np.roll(b, shift=1, axis=1)
    elif mode == "body_shift_2":
        b = np.roll(b, shift=2, axis=1)
    elif mode == "time_reverse":
        f = f[:, ::-1, :].copy(); b = b[:, ::-1, :].copy()
    elif mode == "clean":
        pass
    else:
        raise ValueError(mode)
    return np.ascontiguousarray(f), np.ascontiguousarray(b)


def run_perturbation_robustness(K_list=(8,16)):
    modes = ["clean", "noise_1pct", "noise_3pct", "noise_5pct",
             "dropout_10pct", "dropout_20pct", "keypoint_dropout_10pct", "keypoint_dropout_20pct",
             "body_shift_1", "body_shift_2", "time_reverse"]
    base_f, base_b, y, anchor_id, pair_type = make_random_pair_arrays(test_face, test_body, seed=1301, max_anchors=MAX_RANDOM_EVAL)
    rows = []
    for K in K_list:
        model = get_model_by_K(K)
        if model is None:
            continue
        clean_s = score_pairs_safe(model, base_f, base_b)
        for mode in modes:
            pf, pb = perturb_features(base_f, base_b, mode=mode, seed=7000+K)
            s = score_pairs_safe(model, pf, pb)
            row = {"experiment":"jitter_dropout_robustness", "K":K, "mode":mode,
                   "roc_auc": safe_metric(roc_auc_score, y, s),
                   "pr_auc": safe_metric(average_precision_score, y, s),
                   "score_mean": float(np.mean(s)), "score_std": float(np.std(s)),
                   "delta_score_mean_from_clean": float(np.mean(s - clean_s)),
                   "delta_score_abs_mean_from_clean": float(np.mean(np.abs(s - clean_s)))}
            row.update(threshold_metrics(y, s))
            rows.append(row)
    df = pd.DataFrame(rows)
    save_csv(df, "05_jitter_dropout_robustness.csv")
    display(df)
    return df


def run_latency_test(K_values=(0,8,16,32,64), repeats=30, batch_n=128):
    rows = []
    idx = np.arange(min(batch_n, len(test_face)))
    f_np = np.ascontiguousarray(test_face[idx])
    b_np = np.ascontiguousarray(test_body[idx])
    for K in K_values:
        model = get_model_by_K(K)
        if model is None:
            continue
        # warmup
        _ = score_pairs_safe(model, f_np, b_np, batch_size=len(f_np))
        if DEVICE_TORCH.type == "cuda": torch.cuda.synchronize()
        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = score_pairs_safe(model, f_np, b_np, batch_size=len(f_np))
            if DEVICE_TORCH.type == "cuda": torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)
        rows.append({"experiment":"latency_model_only", "K":K, "model":model_key_for_K(K),
                     "device":str(DEVICE_TORCH), "batch_n":len(f_np),
                     "ms_per_batch_mean":float(np.mean(times)), "ms_per_batch_std":float(np.std(times)),
                     "ms_per_window_mean":float(np.mean(times)/len(f_np))})
    df = pd.DataFrame(rows)
    save_csv(df, "06_latency_model_only.csv")
    display(df)
    return df

print("Robustness/latency functions ready")


Robustness/latency functions ready


In [ ]:
# ============================================================
# CELL 7 — Non-redundancy, shuffled/random controls, gating, diagnostics
# ============================================================
def fit_gate_features(X_train, y_train, X_test, y_test, class_weight="balanced"):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight=class_weight))
    clf.fit(X_train, y_train)
    p = clf.predict_proba(X_test)[:, 1]
    out = {"roc_auc": safe_metric(roc_auc_score, y_test, p),
           "pr_auc": safe_metric(average_precision_score, y_test, p)}
    out.update(threshold_metrics(y_test, p, threshold=0.5))
    return out, p


def run_nonredundancy_controls(pmci_K=8):
    direct = get_model_by_K(0)
    pmci = get_model_by_K(pmci_K)
    if direct is None or pmci is None:
        print("missing direct or pmci model")
        return pd.DataFrame(), pd.DataFrame()

    tr_f, tr_b, tr_y, tr_anchor, _ = make_direct_hard_negative_bank(direct, train_face, train_body, seed=8100+pmci_K)
    te_f, te_b, te_y, te_anchor, te_type = make_direct_hard_negative_bank(direct, test_face, test_body, seed=8200+pmci_K)

    d_tr = score_pairs_safe(direct, tr_f, tr_b); d_te = score_pairs_safe(direct, te_f, te_b)
    p_tr = score_pairs_safe(pmci, tr_f, tr_b); p_te = score_pairs_safe(pmci, te_f, te_b)

    rng = np.random.default_rng(9000 + pmci_K)
    rows, pred_rows = [], []

    feature_sets = {
        "Direct_only": (d_tr.reshape(-1,1), d_te.reshape(-1,1)),
        "PMCI_only": (p_tr.reshape(-1,1), p_te.reshape(-1,1)),
        "Direct_plus_real_PMCI": (np.column_stack([d_tr, p_tr]), np.column_stack([d_te, p_te])),
        "Direct_plus_random_scalar": (np.column_stack([d_tr, rng.normal(size=len(d_tr))]), np.column_stack([d_te, rng.normal(size=len(d_te))])),
    }

    # shuffled PMCI controls, repeated
    for i in range(N_SHUFFLE_CONTROLS):
        sh_tr = rng.permutation(p_tr)
        sh_te = rng.permutation(p_te)
        feature_sets[f"Direct_plus_shuffled_PMCI_{i:03d}"] = (np.column_stack([d_tr, sh_tr]), np.column_stack([d_te, sh_te]))

    # untrained PMCI control
    untrained = PMCIModel(face_dim=train_face.shape[-1], body_dim=train_body.shape[-1], K=pmci_K, tau=0.25).to(DEVICE_TORCH).eval()
    u_tr = score_pairs_safe(untrained, tr_f, tr_b); u_te = score_pairs_safe(untrained, te_f, te_b)
    feature_sets["Direct_plus_untrained_PMCI"] = (np.column_stack([d_tr, u_tr]), np.column_stack([d_te, u_te]))

    for name, (Xtr, Xte) in feature_sets.items():
        res, pred = fit_gate_features(Xtr, tr_y, Xte, te_y, class_weight="balanced")
        boot = grouped_bootstrap_auc(te_y, pred, te_anchor, n_boot=N_GROUP_BOOT, seed=9100+pmci_K)
        rows.append({"experiment":"hard_negative_nonredundancy", "pmci_K":pmci_K, "feature_set":name,
                     "n_train":len(tr_y), "n_test":len(te_y), **res, **boot})
        if name in ["Direct_only", "Direct_plus_real_PMCI", "PMCI_only", "Direct_plus_untrained_PMCI"]:
            pred_rows.extend([{"feature_set":name, "y":int(y), "pred":float(pr), "anchor_id":int(a), "pair_type":str(t)}
                              for y, pr, a, t in zip(te_y, pred, te_anchor, te_type)])

    df = pd.DataFrame(rows)
    pdf = pd.DataFrame(pred_rows)
    save_csv(df, f"07_nonredundancy_PMCI_K{pmci_K}.csv")
    save_csv(pdf, f"07b_nonredundancy_predictions_PMCI_K{pmci_K}.csv")
    display(df)
    return df, pdf


def run_uncertainty_gating(pmci_K=8, bank_source="hard"):
    direct = get_model_by_K(0)
    pmci = get_model_by_K(pmci_K)
    if direct is None or pmci is None:
        return pd.DataFrame()
    if bank_source == "hard":
        f, b, y, anchor_id, pair_type = make_direct_hard_negative_bank(direct, test_face, test_body, seed=9300+pmci_K)
    else:
        f, b, y, anchor_id, pair_type = make_random_pair_arrays(test_face, test_body, seed=9400+pmci_K, max_anchors=MAX_RANDOM_EVAL)
    direct_s = score_pairs_safe(direct, f, b)
    pmci_s = score_pairs_safe(pmci, f, b)
    rows = []
    for reject in [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
        if reject == 0:
            keep = np.ones(len(y), dtype=bool)
            thr = -np.inf
        else:
            thr = np.quantile(pmci_s, reject)
            keep = pmci_s >= thr
        if len(np.unique(y[keep])) < 2:
            continue
        row = {"experiment":"uncertainty_aware_gating", "pmci_K":pmci_K, "bank_source":bank_source,
               "reject_lowest_pmci_fraction":reject, "coverage":float(np.mean(keep)),
               "retained_pairs":int(np.sum(keep)), "rejected_pairs":int(np.sum(~keep)),
               "pmci_threshold":float(thr) if np.isfinite(thr) else np.nan,
               "roc_auc":safe_metric(roc_auc_score, y[keep], direct_s[keep]),
               "pr_auc":safe_metric(average_precision_score, y[keep], direct_s[keep]),
               "rejected_positive_rate":float(np.mean(y[~keep])) if np.any(~keep) else np.nan}
        row.update(threshold_metrics(y[keep], direct_s[keep]))
        rows.append(row)
    df = pd.DataFrame(rows)
    save_csv(df, f"08_uncertainty_gating_PMCI_K{pmci_K}_{bank_source}.csv")
    display(df)
    return df


def run_near_orthogonal_diagnostic(pmci_K=8, bank_source="random"):
    direct = get_model_by_K(0)
    pmci = get_model_by_K(pmci_K)
    if pmci is None:
        return pd.DataFrame()
    if bank_source == "hard" and direct is not None:
        f, b, y, anchor_id, pair_type = make_direct_hard_negative_bank(direct, test_face, test_body, seed=9500+pmci_K)
    else:
        f, b, y, anchor_id, pair_type = make_random_pair_arrays(test_face, test_body, seed=9600+pmci_K, max_anchors=MAX_RANDOM_EVAL)
    pf, pb = get_proto_probs(pmci, f, b)
    pmci_s = score_pairs_safe(pmci, f, b)
    dot = np.sum(pf * pb, axis=1)
    arg_same = np.argmax(pf, axis=1) == np.argmax(pb, axis=1)
    near = dot <= np.quantile(dot, 0.10)
    rows = []
    for name, mask in [("near_orthogonal_lowest_10pct_dot", near), ("other_90pct", ~near), ("argmax_same", arg_same), ("argmax_different", ~arg_same)]:
        rows.append({"experiment":"near_orthogonal_diagnostic", "pmci_K":pmci_K, "bank_source":bank_source,
                     "group":name, "n":int(np.sum(mask)), "positive_rate":float(np.mean(y[mask])),
                     "pmci_mean":float(np.mean(pmci_s[mask])), "pmci_std":float(np.std(pmci_s[mask])),
                     "prob_dot_mean":float(np.mean(dot[mask])),
                     "argmax_agreement_rate":float(np.mean(arg_same[mask]))})
    df = pd.DataFrame(rows)
    save_csv(df, f"09_near_orthogonal_PMCI_K{pmci_K}_{bank_source}.csv")
    display(df)
    return df

print("Nonredundancy/gating functions ready")


Nonredundancy/gating functions ready


In [ ]:
# ============================================================
# CELL 8 — Master runners
# ============================================================
def preflight_revision_check():
    rows = []
    rows.append({"item":"std_available", "value": std is not None})
    rows.append({"item":"n_train", "value": len(train_face)})
    rows.append({"item":"n_val", "value": len(val_face)})
    rows.append({"item":"n_test", "value": len(test_face)})
    rows.append({"item":"models", "value": ", ".join(sorted(trained_models.keys()))})
    df = pd.DataFrame(rows)
    save_csv(df, "00_preflight.csv")
    display(df)
    return df


def run_pmci_revision_quick_v6():
    """Quick sanity run. Use this first."""
    summary = {}
    summary["preflight"] = len(preflight_revision_check())
    summary["tau_eval"] = len(run_tau_sensitivity(K_list=[8,16]))
    k_df, u_df = run_extended_K_ablation(K_values=[4,8,16,32,64])
    summary["k_ablation"] = len(k_df)
    summary["utilization"] = len(u_df)
    summary["perturbation"] = len(run_perturbation_robustness(K_list=(8,16)))
    summary["latency"] = len(run_latency_test(K_values=(0,8,16,32,64), repeats=10))
    nr8, _ = run_nonredundancy_controls(pmci_K=8)
    summary["nonred8"] = len(nr8)
    summary["gating8_hard"] = len(run_uncertainty_gating(pmci_K=8, bank_source="hard"))
    summary["orth8"] = len(run_near_orthogonal_diagnostic(pmci_K=8))
    summary["output_dir"] = str(REV_DIR)
    with open(REV_DIR / "00_summary_quick_v6.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))
    return summary


def run_pmci_revision_final_v6():
    """Final mode. Before running, set TRAIN_INIT_ROBUSTNESS=True, TRAIN_TAU_ROBUSTNESS=True, N_SHUFFLE_CONTROLS=100, N_GROUP_BOOT=1000 if time allows."""
    summary = {}
    summary["preflight"] = len(preflight_revision_check())
    summary["tau_eval"] = len(run_tau_sensitivity(K_list=[8,16,32]))
    tau_train, tau_hist = run_tau_training_sensitivity(K_values=(8,16))
    summary["tau_train"] = len(tau_train)
    k_df, u_df = run_extended_K_ablation(K_values=K_VALUES_EXT)
    summary["k_ablation"] = len(k_df)
    summary["utilization"] = len(u_df)
    kf, uf = run_K_multiseed_final(K_values=K_VALUES_EXT)
    summary["k_multiseed"] = len(kf)
    init_df, init_hist = run_initialization_robustness(K_values=(8,16))
    summary["init"] = len(init_df)
    summary["perturbation"] = len(run_perturbation_robustness(K_list=(8,16)))
    summary["latency"] = len(run_latency_test(K_values=(0,8,16,32,64), repeats=30))
    nr8, _ = run_nonredundancy_controls(pmci_K=8)
    nr16, _ = run_nonredundancy_controls(pmci_K=16)
    summary["nonred8"] = len(nr8)
    summary["nonred16"] = len(nr16)
    for K in [8,16]:
        for src in ["hard", "random"]:
            summary[f"gating_K{K}_{src}"] = len(run_uncertainty_gating(pmci_K=K, bank_source=src))
        summary[f"orth_K{K}"] = len(run_near_orthogonal_diagnostic(pmci_K=K))
    summary["output_dir"] = str(REV_DIR)
    with open(REV_DIR / "00_summary_final_v6.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))
    return summary

print("V6 loaded.")
print("First run quick:")
print("summary_quick_v6 = run_pmci_revision_quick_v6()")
print("For final set flags, then run:")
print("summary_final_v6 = run_pmci_revision_final_v6()")


V6 loaded.
First run quick:
summary_quick_v6 = run_pmci_revision_quick_v6()
For final set flags, then run:
summary_final_v6 = run_pmci_revision_final_v6()


In [ ]:
summary_quick_v6 = run_pmci_revision_quick_v6()


saved: /content/pmci_revision_results_v6/00_preflight.csv


,item,value
0,std_available,True
1,n_train,8160
2,n_val,1920
3,n_test,1440
4,models,"DirectCosine_K0, PMCI_K16, PMCI_K32, PMCI_K8"


saved: /content/pmci_revision_results_v6/01_tau_sensitivity_eval.csv


,experiment,K,tau,roc_auc,pr_auc,score_mean,score_std,pos_mean,neg_mean,accuracy,balanced_accuracy,f1,mcc
0,tau_sensitivity_eval,8,0.10,0.838560,0.797557,0.290015,0.289431,0.440746,0.139283,0.640972,0.640972,0.493634,0.346698
1,tau_sensitivity_eval,8,0.25,0.850415,0.812221,0.539438,0.280268,0.715331,0.363545,0.781944,0.781944,0.805813,0.581741
2,tau_sensitivity_eval,8,0.50,0.846219,0.801765,0.780627,0.159250,0.879946,0.681309,0.581944,0.581944,0.705191,0.298762
3,tau_sensitivity_eval,8,1.00,0.846405,0.795610,0.932227,0.052436,0.964353,0.900101,0.500000,0.500000,0.666667,0.000000
4,tau_sensitivity_eval,8,2.00,0.846161,0.791111,0.982413,0.013879,0.990818,0.974008,0.500000,0.500000,0.666667,0.000000
5,tau_sensitivity_eval,8,4.00,0.845795,0.789885,0.995612,0.003483,0.997709,0.993515,0.500000,0.500000,0.666667,0.000000
6,tau_sensitivity_eval,16,0.10,0.803095,0.743498,0.312373,0.301109,0.454990,0.169755,0.615972,0.615972,0.501353,0.261180
7,tau_sensitivity_eval,16,0.25,0.804018,0.732247,0.575378,0.299015,0.743179,0.407576,0.753472,0.753472,0.777708,0.519444
8,tau_sensitivity_eval,16,0.50,0.806112,0.726083,0.820534,0.150942,0.907401,0.733668,0.500000,0.500000,0.666667,0.000000
9,tau_sensitivity_eval,16,1.00,0.807696,0.725119,0.949541,0.044973,0.975533,0.923550,0.500000,0.500000,0.666667,0.000000


Training missing model PMCI_K4 ...
Training missing model PMCI_K64 ...
saved: /content/pmci_revision_results_v6/02_extended_K_ablation.csv
saved: /content/pmci_revision_results_v6/03_prototype_utilization.csv


...

saved: /content/pmci_revision_results_v6/08_uncertainty_gating_PMCI_K8_hard.csv


,experiment,pmci_K,bank_source,reject_lowest_pmci_fraction,coverage,retained_pairs,rejected_pairs,pmci_threshold,roc_auc,pr_auc,rejected_positive_rate,accuracy,balanced_accuracy,f1,mcc
0,uncertainty_aware_gating,8,hard,0.00,1.00,5040,0,NaN,0.476507,0.046161,NaN,0.047619,0.5,0.090909,0.0
1,uncertainty_aware_gating,8,hard,0.05,0.95,4788,252,0.230355,0.487719,0.052390,0.003968,0.049916,0.5,0.095087,0.0
2,uncertainty_aware_gating,8,hard,0.10,0.90,4536,504,0.344812,0.473915,0.051326,0.019841,0.050705,0.5,0.096517,0.0
3,uncertainty_aware_gating,8,hard,0.20,0.80,4032,1008,0.527964,0.479871,0.057104,0.022817,0.053819,0.5,0.102142,0.0
4,uncertainty_aware_gating,8,hard,0.30,0.70,3528,1512,0.639374,0.483852,0.062587,0.027116,0.056406,0.5,0.106788,0.0
5,uncertainty_aware_gating,8,hard,0.40,0.60,3024,2016,0.696479,0.500272,0.070270,0.029762,0.059524,0.5,0.112360,0.0
6,uncertainty_aware_gating,8,hard,0.50,0.50,2520,2520,0.734014,0.475248,0.062431,0.036111,0.059127,0.5,0.111652,0.0


saved: /content/pmci_revision_results_v6/09_near_orthogonal_PMCI_K8_random.csv


,experiment,pmci_K,bank_source,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,argmax_agreement_rate
0,near_orthogonal_diagnostic,8,random,near_orthogonal_lowest_10pct_dot,288,0.000000,0.080386,0.014824,0.004519,0.000000
1,near_orthogonal_diagnostic,8,random,other_90pct,2592,0.555556,0.584912,0.251672,0.220247,0.155864
2,near_orthogonal_diagnostic,8,random,argmax_same,404,0.814356,0.791578,0.068106,0.433092,1.000000
3,near_orthogonal_diagnostic,8,random,argmax_different,2476,0.448708,0.492506,0.282268,0.160425,0.000000


{
  "preflight": 5,
  "tau_eval": 12,
  "k_ablation": 5,
  "utilization": 5,
  "perturbation": 22,
  "latency": 5,
  "nonred8": 25,
  "gating8_hard": 7,
  "orth8": 4,
  "output_dir": "/content/pmci_revision_results_v6"
}


In [ ]:
# ============================================================
# CELL 10 — FINAL PAPER RUN
# ============================================================


TRAIN_INIT_ROBUSTNESS = True
TRAIN_TAU_ROBUSTNESS = True
TRAIN_K_MULTI_SEED_FINAL = True
N_SHUFFLE_CONTROLS = 100
N_GROUP_BOOT = 1000
summary_final_v6 = run_pmci_revision_final_v6()


saved: /content/pmci_revision_results_v6/00_preflight.csv


,item,value
0,std_available,True
1,n_train,8160
2,n_val,1920
3,n_test,1440
4,models,"DirectCosine_K0, PMCI_K16, PMCI_K32, PMCI_K4, ..."


saved: /content/pmci_revision_results_v6/01_tau_sensitivity_eval.csv


,experiment,K,tau,roc_auc,pr_auc,score_mean,score_std,pos_mean,neg_mean,accuracy,balanced_accuracy,f1,mcc
0,tau_sensitivity_eval,8,0.10,0.838560,0.797557,0.290015,0.289431,0.440746,0.139283,0.640972,0.640972,0.493634,0.346698
1,tau_sensitivity_eval,8,0.25,0.850415,0.812221,0.539438,0.280268,0.715331,0.363545,0.781944,0.781944,0.805813,0.581741
2,tau_sensitivity_eval,8,0.50,0.846219,0.801765,0.780627,0.159250,0.879946,0.681309,0.581944,0.581944,0.705191,0.298762
3,tau_sensitivity_eval,8,1.00,0.846405,0.795610,0.932227,0.052436,0.964353,0.900101,0.500000,0.500000,0.666667,0.000000
4,tau_sensitivity_eval,8,2.00,0.846161,0.791111,0.982413,0.013879,0.990818,0.974008,0.500000,0.500000,0.666667,0.000000
5,tau_sensitivity_eval,8,4.00,0.845795,0.789885,0.995612,0.003483,0.997709,0.993515,0.500000,0.500000,0.666667,0.000000
6,tau_sensitivity_eval,16,0.10,0.803095,0.743498,0.312373,0.301109,0.454990,0.169755,0.615972,0.615972,0.501353,0.261180
7,tau_sensitivity_eval,16,0.25,0.804018,0.732247,0.575378,0.299015,0.743179,0.407576,0.753472,0.753472,0.777708,0.519444
8,tau_sensitivity_eval,16,0.50,0.806112,0.726083,0.820534,0.150942,0.907401,0.733668,0.500000,0.500000,0.666667,0.000000
9,tau_sensitivity_eval,16,1.00,0.807696,0.725119,0.949541,0.044973,0.975533,0.923550,0.500000,0.500000,0.666667,0.000000


saved: /content/pmci_revision_results_v6/01b_tau_training_sensitivity.csv
saved: /content/pmci_revision_results_v6/01c_tau_training_history.csv


...

saved: /content/pmci_revision_results_v6/08_uncertainty_gating_PMCI_K16_random.csv


,experiment,pmci_K,bank_source,reject_lowest_pmci_fraction,coverage,retained_pairs,rejected_pairs,pmci_threshold,roc_auc,pr_auc,rejected_positive_rate,accuracy,balanced_accuracy,f1,mcc
0,uncertainty_aware_gating,16,random,0.00,1.00,2880,0,NaN,0.944442,0.921959,NaN,0.740625,0.740625,0.794045,0.562914
1,uncertainty_aware_gating,16,random,0.05,0.95,2736,144,0.124954,0.938280,0.921963,0.000000,0.728436,0.713349,0.794921,0.530535
2,uncertainty_aware_gating,16,random,0.10,0.90,2592,288,0.138678,0.930618,0.921988,0.000000,0.715664,0.680122,0.796240,0.488146
3,uncertainty_aware_gating,16,random,0.20,0.80,2304,576,0.190804,0.912550,0.923971,0.019097,0.700955,0.606286,0.805751,0.378709
4,uncertainty_aware_gating,16,random,0.30,0.70,2016,864,0.299471,0.902591,0.930939,0.103009,0.716270,0.569925,0.825290,0.313450
5,uncertainty_aware_gating,16,random,0.40,0.60,1728,1152,0.497979,0.896715,0.939316,0.171007,0.747106,0.549485,0.850496,0.270602
6,uncertainty_aware_gating,16,random,0.50,0.50,1440,1440,0.679452,0.899852,0.956841,0.237500,0.781250,0.539474,0.874552,0.247685


saved: /content/pmci_revision_results_v6/09_near_orthogonal_PMCI_K16_random.csv


,experiment,pmci_K,bank_source,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,argmax_agreement_rate
0,near_orthogonal_diagnostic,16,random,near_orthogonal_lowest_10pct_dot,288,0.000000,0.126924,0.010019,0.004906,0.000000
1,near_orthogonal_diagnostic,16,random,other_90pct,2592,0.555556,0.623470,0.274260,0.139179,0.125386
2,near_orthogonal_diagnostic,16,random,argmax_same,325,0.812308,0.908872,0.061407,0.271563,1.000000
3,near_orthogonal_diagnostic,16,random,argmax_different,2555,0.460274,0.531195,0.291129,0.107204,0.000000


{
  "preflight": 5,
  "tau_eval": 18,
  "tau_train": 24,
  "k_ablation": 5,
  "utilization": 5,
  "k_multiseed": 15,
  "init": 24,
  "perturbation": 22,
  "latency": 5,
  "nonred8": 105,
  "nonred16": 105,
  "gating_K8_hard": 7,
  "gating_K8_random": 7,
  "orth_K8": 4,
  "gating_K16_hard": 7,
  "gating_K16_random": 7,
  "orth_K16": 4,
  "output_dir": "/content/pmci_revision_results_v6"
}


In [ ]:
# =========================
# FINAL
# K16 / K32 hard-negative + permutation + gating + near-orthogonal
# =========================

import os, math, time, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import mannwhitneyu, ks_2samp

warnings.filterwarnings("ignore")

OUT_FINAL = Path("/content/pmci_revision_results_final_extra")
OUT_FINAL.mkdir(parents=True, exist_ok=True)

DEVICE_FINAL = globals().get(
    "DEVICE_TORCH",
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

print("Output:", OUT_FINAL)
print("Device:", DEVICE_FINAL)

# -------------------------
# Basic checks
# -------------------------
required = ["std", "trained_models", "score_pairs"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Run the main V6/V7 notebook first.")

for key in ["val_face", "val_body", "test_face", "test_body"]:
    if key not in std:
        raise RuntimeError(f"std['{key}'] is missing.")

def unwrap_model(obj):
    if isinstance(obj, dict):
        if "model" in obj:
            return obj["model"]
        if "net" in obj:
            return obj["net"]
    return obj

def get_trained_model(name):
    if name not in trained_models:
        raise RuntimeError(f"{name} not found in trained_models. Available: {list(trained_models.keys())}")
    model = unwrap_model(trained_models[name])
    model.to(DEVICE_FINAL)
    model.eval()
    return model

def safe_auc(y, s):
    y = np.asarray(y)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def binary_metrics(y, prob, thr=0.5):
    y = np.asarray(y).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= thr).astype(int)
    return {
        "roc_auc": safe_auc(y, prob),
        "pr_auc": safe_ap(y, prob),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
    }

def cohen_d(pos, neg):
    pos = np.asarray(pos)
    neg = np.asarray(neg)
    n1, n0 = len(pos), len(neg)
    if n1 < 2 or n0 < 2:
        return np.nan
    sp = np.sqrt(((n1 - 1) * np.var(pos, ddof=1) + (n0 - 1) * np.var(neg, ddof=1)) / (n1 + n0 - 2))
    if sp == 0:
        return np.nan
    return (np.mean(pos) - np.mean(neg)) / sp

def cliffs_delta(pos, neg):
    pos = np.asarray(pos)
    neg = np.asarray(neg)
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    # Efficient enough for current bank sizes if sampled
    max_n = 5000
    rng = np.random.default_rng(123)
    if len(pos) > max_n:
        pos = rng.choice(pos, size=max_n, replace=False)
    if len(neg) > max_n:
        neg = rng.choice(neg, size=max_n, replace=False)
    gt = sum(np.sum(p > neg) for p in pos)
    lt = sum(np.sum(p < neg) for p in pos)
    return (gt - lt) / (len(pos) * len(neg))

def evaluate_raw_score(y, s, name, K=None, bank=""):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    pos = s[y == 1]
    neg = s[y == 0]

    try:
        mw = mannwhitneyu(pos, neg, alternative="two-sided")
        mw_p = float(mw.pvalue)
    except Exception:
        mw_p = np.nan

    try:
        ks = ks_2samp(pos, neg)
        ks_stat, ks_p = float(ks.statistic), float(ks.pvalue)
    except Exception:
        ks_stat, ks_p = np.nan, np.nan

    return {
        "experiment": "score_distribution_effect",
        "bank": bank,
        "feature": name,
        "K": K,
        "n": len(y),
        "positive_rate": float(np.mean(y)),
        "roc_auc": safe_auc(y, s),
        "pr_auc": safe_ap(y, s),
        "pos_mean": float(np.mean(pos)) if len(pos) else np.nan,
        "neg_mean": float(np.mean(neg)) if len(neg) else np.nan,
        "pos_std": float(np.std(pos)) if len(pos) else np.nan,
        "neg_std": float(np.std(neg)) if len(neg) else np.nan,
        "cohen_d": cohen_d(pos, neg),
        "cliffs_delta": cliffs_delta(pos, neg),
        "mannwhitney_p": mw_p,
        "ks_stat": ks_stat,
        "ks_p": ks_p,
    }

print("Helpers ready.")

Output: /content/pmci_revision_results_final_extra
Device: cuda
Helpers ready.


In [ ]:
# =========================
# Build hard-negative banks
# Val bank trains logistic model; Test bank evaluates it.
# No repeated positives. Class imbalance handled with class_weight='balanced'.
# =========================

DIRECT_MODEL = get_trained_model("DirectCosine_K0")

def build_direct_mined_hard_bank(face_np, body_np, hard_k=20, max_anchors=None, seed=123):
    """
    For each anchor i:
      positive: (face_i, body_i, y=1)
      hard negatives: top-k wrong body_j according to DirectCosine
    """
    rng = np.random.default_rng(seed)
    n = len(face_np)
    anchors = np.arange(n)
    if max_anchors is not None and max_anchors < n:
        anchors = rng.choice(anchors, size=max_anchors, replace=False)
        anchors = np.sort(anchors)

    rows = []
    for count, i in enumerate(anchors):
        f_rep = np.repeat(face_np[i:i+1], n, axis=0)
        direct_scores = score_pairs(DIRECT_MODEL, f_rep, body_np)
        direct_scores = np.asarray(direct_scores).reshape(-1)
        direct_scores[i] = -np.inf

        top_idx = np.argsort(direct_scores)[-hard_k:][::-1]

        rows.append({
            "anchor": int(i),
            "face_idx": int(i),
            "body_idx": int(i),
            "y": 1,
            "pair_type": "true_pair"
        })

        for j in top_idx:
            rows.append({
                "anchor": int(i),
                "face_idx": int(i),
                "body_idx": int(j),
                "y": 0,
                "pair_type": "direct_mined_hard_negative"
            })

    return pd.DataFrame(rows)

HARD_K = 20

hard_val = build_direct_mined_hard_bank(
    std["val_face"], std["val_body"],
    hard_k=HARD_K,
    max_anchors=None,
    seed=2027
)

hard_test = build_direct_mined_hard_bank(
    std["test_face"], std["test_body"],
    hard_k=HARD_K,
    max_anchors=None,
    seed=2028
)

hard_val.to_csv(OUT_FINAL / "hard_bank_val_direct_mined.csv", index=False)
hard_test.to_csv(OUT_FINAL / "hard_bank_test_direct_mined.csv", index=False)

print("Hard val:", hard_val.shape, "positive rate:", hard_val["y"].mean())
print("Hard test:", hard_test.shape, "positive rate:", hard_test["y"].mean())
display(hard_test.head())

def bank_scores(model, face_np, body_np, bank_df, batch_size=512):
    fi = bank_df["face_idx"].to_numpy()
    bi = bank_df["body_idx"].to_numpy()
    f = np.ascontiguousarray(face_np[fi], dtype=np.float32)
    b = np.ascontiguousarray(body_np[bi], dtype=np.float32)
    return np.asarray(score_pairs(model, f, b, batch_size=batch_size)).reshape(-1)

def make_bank_feature_table(bank_df, face_np, body_np, K_list=(16, 32), split_name="test"):
    out = bank_df.copy()
    out["direct"] = bank_scores(DIRECT_MODEL, face_np, body_np, out)

    for K in K_list:
        m = get_trained_model(f"PMCI_K{K}")
        out[f"pmci_K{K}"] = bank_scores(m, face_np, body_np, out)

    out["split"] = split_name
    return out

K_FINAL_LIST = [16, 32]

hard_val_scores = make_bank_feature_table(
    hard_val, std["val_face"], std["val_body"],
    K_list=K_FINAL_LIST,
    split_name="val"
)

hard_test_scores = make_bank_feature_table(
    hard_test, std["test_face"], std["test_body"],
    K_list=K_FINAL_LIST,
    split_name="test"
)

hard_val_scores.to_csv(OUT_FINAL / "hard_val_scores_K16_K32.csv", index=False)
hard_test_scores.to_csv(OUT_FINAL / "hard_test_scores_K16_K32.csv", index=False)

print("Saved scored hard banks.")
display(hard_test_scores.head())

Hard val: (40320, 5) positive rate: 0.047619047619047616
Hard test: (30240, 5) positive rate: 0.047619047619047616


,anchor,face_idx,body_idx,y,pair_type
0,0,0,0,1,true_pair
1,0,0,259,0,direct_mined_hard_negative
2,0,0,19,0,direct_mined_hard_negative
3,0,0,38,0,direct_mined_hard_negative
4,0,0,278,0,direct_mined_hard_negative


Saved scored hard banks.


,anchor,face_idx,body_idx,y,pair_type,direct,pmci_K16,pmci_K32,split
0,0,0,0,1,true_pair,0.892318,0.474894,0.973587,test
1,0,0,259,0,direct_mined_hard_negative,0.897622,0.542527,0.969469,test
2,0,0,19,0,direct_mined_hard_negative,0.897308,0.577862,0.971944,test
3,0,0,38,0,direct_mined_hard_negative,0.896033,0.670714,0.937469,test
4,0,0,278,0,direct_mined_hard_negative,0.895518,0.651139,0.935825,test


In [ ]:
# =========================
# Grouped bootstrap by anchor_id
# =========================

def grouped_bootstrap_auc(y, score, groups, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score = np.asarray(score)
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)

    vals = []
    for _ in range(B):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        mask_idx = []
        for g in sampled_groups:
            mask_idx.extend(np.where(groups == g)[0].tolist())
        mask_idx = np.asarray(mask_idx)
        yy = y[mask_idx]
        ss = score[mask_idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, ss))

    vals = np.asarray(vals)
    return {
        "boot_mean": float(np.mean(vals)) if len(vals) else np.nan,
        "boot_lo": float(np.quantile(vals, 0.025)) if len(vals) else np.nan,
        "boot_hi": float(np.quantile(vals, 0.975)) if len(vals) else np.nan,
        "boot_n": int(len(vals)),
    }

def grouped_bootstrap_delta_auc(y, score_a, score_b, groups, B=1000, seed=123):
    """
    delta = AUC(score_a) - AUC(score_b)
    """
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score_a = np.asarray(score_a)
    score_b = np.asarray(score_b)
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)

    vals = []
    for _ in range(B):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = []
        for g in sampled_groups:
            idx.extend(np.where(groups == g)[0].tolist())
        idx = np.asarray(idx)
        yy = y[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, score_a[idx]) - roc_auc_score(yy, score_b[idx]))

    vals = np.asarray(vals)
    return {
        "delta_auc_mean": float(np.mean(vals)) if len(vals) else np.nan,
        "delta_auc_lo": float(np.quantile(vals, 0.025)) if len(vals) else np.nan,
        "delta_auc_hi": float(np.quantile(vals, 0.975)) if len(vals) else np.nan,
        "delta_auc_boot_n": int(len(vals)),
    }

def fit_eval_logreg(X_train, y_train, X_test, y_test):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs"
        )
    )
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    return prob, clf

def eval_feature_set(name, Xtr, ytr, Xte, yte, groups_te, B=1000, seed=123):
    prob, clf = fit_eval_logreg(Xtr, ytr, Xte, yte)
    met = binary_metrics(yte, prob)
    boot = grouped_bootstrap_auc(yte, prob, groups_te, B=B, seed=seed)
    row = {
        "experiment": "hard_negative_nonredundancy_final",
        "feature_set": name,
        "n_train": len(ytr),
        "n_test": len(yte),
        **met,
        **boot
    }
    return row, prob, clf

print("Bootstrap helpers ready.")

Bootstrap helpers ready.


In [ ]:
# =========================
# FINAL hard-negative nonredundancy for K16 and K32
# =========================

N_SHUFFLES_FINAL = 100
N_BOOT_FINAL = 1000
RANDOM_VECTOR_DIM = 32

def try_make_untrained_pmci(K):
    if "PMCIModel" not in globals():
        return None

    face_dim = std["train_face"].shape[-1] if "train_face" in std else std["val_face"].shape[-1]
    body_dim = std["train_body"].shape[-1] if "train_body" in std else std["val_body"].shape[-1]

    # Try several constructor signatures
    constructors = [
        lambda: PMCIModel(face_dim=face_dim, body_dim=body_dim, K=K, tau=0.25),
        lambda: PMCIModel(face_dim, body_dim, K=K, tau=0.25),
        lambda: PMCIModel(face_dim, body_dim, K),
    ]
    for c in constructors:
        try:
            m = c()
            m.to(DEVICE_FINAL)
            m.eval()
            return m
        except Exception:
            pass
    return None

def run_nonredundancy_for_K(K):
    rng = np.random.default_rng(8000 + K)

    ytr = hard_val_scores["y"].to_numpy().astype(int)
    yte = hard_test_scores["y"].to_numpy().astype(int)
    gte = hard_test_scores["anchor"].to_numpy().astype(int)

    dtr = hard_val_scores["direct"].to_numpy()
    dte = hard_test_scores["direct"].to_numpy()
    ptr = hard_val_scores[f"pmci_K{K}"].to_numpy()
    pte = hard_test_scores[f"pmci_K{K}"].to_numpy()

    rows = []
    pred_cols = pd.DataFrame({
        "y": yte,
        "anchor": gte,
        "direct": dte,
        f"pmci_K{K}": pte
    })

    # Raw score distributions
    dist_rows = [
        evaluate_raw_score(yte, dte, "Direct_raw", K=0, bank="hard_test"),
        evaluate_raw_score(yte, pte, f"PMCI_K{K}_raw", K=K, bank="hard_test")
    ]

    # Direct only
    row, prob_direct, _ = eval_feature_set(
        "Direct_only",
        dtr.reshape(-1, 1), ytr,
        dte.reshape(-1, 1), yte,
        gte,
        B=N_BOOT_FINAL,
        seed=100 + K
    )
    row["K"] = K
    rows.append(row)
    pred_cols["prob_Direct_only"] = prob_direct

    # PMCI only
    row, prob_pmci, _ = eval_feature_set(
        f"PMCI_K{K}_only",
        ptr.reshape(-1, 1), ytr,
        pte.reshape(-1, 1), yte,
        gte,
        B=N_BOOT_FINAL,
        seed=200 + K
    )
    row["K"] = K
    rows.append(row)
    pred_cols[f"prob_PMCI_K{K}_only"] = prob_pmci

    # Direct + real PMCI
    Xtr_real = np.column_stack([dtr, ptr])
    Xte_real = np.column_stack([dte, pte])
    row, prob_real, _ = eval_feature_set(
        f"Direct_plus_real_PMCI_K{K}",
        Xtr_real, ytr,
        Xte_real, yte,
        gte,
        B=N_BOOT_FINAL,
        seed=300 + K
    )
    row["K"] = K
    rows.append(row)
    pred_cols[f"prob_Direct_plus_real_PMCI_K{K}"] = prob_real

    # Direct + random scalar
    rtr = rng.normal(size=len(ytr))
    rte = rng.normal(size=len(yte))
    row, prob_rand_scalar, _ = eval_feature_set(
        "Direct_plus_random_scalar",
        np.column_stack([dtr, rtr]), ytr,
        np.column_stack([dte, rte]), yte,
        gte,
        B=N_BOOT_FINAL,
        seed=400 + K
    )
    row["K"] = K
    rows.append(row)
    pred_cols["prob_Direct_plus_random_scalar"] = prob_rand_scalar

    # Direct + random vector same / larger dimension
    RV_DIM = max(K, RANDOM_VECTOR_DIM)
    rvtr = rng.normal(size=(len(ytr), RV_DIM))
    rvte = rng.normal(size=(len(yte), RV_DIM))
    row, prob_rand_vec, _ = eval_feature_set(
        f"Direct_plus_random_vector_dim{RV_DIM}",
        np.column_stack([dtr.reshape(-1, 1), rvtr]), ytr,
        np.column_stack([dte.reshape(-1, 1), rvte]), yte,
        gte,
        B=N_BOOT_FINAL,
        seed=500 + K
    )
    row["K"] = K
    rows.append(row)
    pred_cols[f"prob_Direct_plus_random_vector_dim{RV_DIM}"] = prob_rand_vec

    # Direct + untrained PMCI
    untrained = try_make_untrained_pmci(K)
    if untrained is not None:
        utr = bank_scores(untrained, std["val_face"], std["val_body"], hard_val_scores)
        ute = bank_scores(untrained, std["test_face"], std["test_body"], hard_test_scores)
        row, prob_untrained, _ = eval_feature_set(
            f"Direct_plus_untrained_PMCI_K{K}",
            np.column_stack([dtr, utr]), ytr,
            np.column_stack([dte, ute]), yte,
            gte,
            B=N_BOOT_FINAL,
            seed=600 + K
        )
        row["K"] = K
        rows.append(row)
        pred_cols[f"untrained_pmci_K{K}"] = ute
        pred_cols[f"prob_Direct_plus_untrained_PMCI_K{K}"] = prob_untrained

    # 100 shuffled PMCI controls
    shuffle_aucs = []
    shuffle_rows = []
    for sidx in range(N_SHUFFLES_FINAL):
        shtr = rng.permutation(ptr)
        shte = rng.permutation(pte)
        row, prob_sh, _ = eval_feature_set(
            f"Direct_plus_shuffled_PMCI_K{K}_{sidx:03d}",
            np.column_stack([dtr, shtr]), ytr,
            np.column_stack([dte, shte]), yte,
            gte,
            B=200,  # lighter inside 100 controls
            seed=7000 + K * 1000 + sidx
        )
        row["K"] = K
        shuffle_rows.append(row)
        shuffle_aucs.append(row["roc_auc"])

    rows.extend(shuffle_rows)

    # Permutation p-value: how many shuffled controls >= real
    real_auc = [r for r in rows if r["feature_set"] == f"Direct_plus_real_PMCI_K{K}"][0]["roc_auc"]
    shuffle_aucs = np.asarray(shuffle_aucs)
    permutation_p = (1 + np.sum(shuffle_aucs >= real_auc)) / (len(shuffle_aucs) + 1)

    summary_row = {
        "experiment": "shuffled_control_summary",
        "K": K,
        "real_auc": real_auc,
        "shuffle_auc_mean": float(np.mean(shuffle_aucs)),
        "shuffle_auc_std": float(np.std(shuffle_aucs)),
        "shuffle_auc_max": float(np.max(shuffle_aucs)),
        "shuffle_auc_min": float(np.min(shuffle_aucs)),
        "permutation_p_real_greater_than_shuffle": float(permutation_p),
        "n_shuffles": int(len(shuffle_aucs))
    }

    # Delta AUC real vs direct
    delta_real_direct = grouped_bootstrap_delta_auc(
        yte,
        prob_real,
        prob_direct,
        gte,
        B=N_BOOT_FINAL,
        seed=900 + K
    )
    summary_row.update({
        "delta_real_vs_direct_mean": delta_real_direct["delta_auc_mean"],
        "delta_real_vs_direct_lo": delta_real_direct["delta_auc_lo"],
        "delta_real_vs_direct_hi": delta_real_direct["delta_auc_hi"],
    })

    # Save
    rows_df = pd.DataFrame(rows)
    dist_df = pd.DataFrame(dist_rows)
    summary_df = pd.DataFrame([summary_row])

    rows_df.to_csv(OUT_FINAL / f"nonredundancy_full_K{K}.csv", index=False)
    dist_df.to_csv(OUT_FINAL / f"score_distribution_effect_K{K}.csv", index=False)
    summary_df.to_csv(OUT_FINAL / f"nonredundancy_summary_K{K}.csv", index=False)
    pred_cols.to_csv(OUT_FINAL / f"nonredundancy_predictions_K{K}.csv", index=False)

    print(f"\n===== K={K} nonredundancy summary =====")
    display(summary_df)
    display(rows_df[rows_df["feature_set"].isin([
        "Direct_only",
        f"PMCI_K{K}_only",
        f"Direct_plus_real_PMCI_K{K}",
        "Direct_plus_random_scalar",
        f"Direct_plus_random_vector_dim{max(K, RANDOM_VECTOR_DIM)}",
        f"Direct_plus_untrained_PMCI_K{K}"
    ])])

    return rows_df, summary_df, dist_df, pred_cols

all_nonred = []
all_nonred_summary = []
all_dist = []

for K in [16, 32]:
    rows_df, summary_df, dist_df, pred_df = run_nonredundancy_for_K(K)
    all_nonred.append(rows_df)
    all_nonred_summary.append(summary_df)
    all_dist.append(dist_df)

pd.concat(all_nonred, ignore_index=True).to_csv(OUT_FINAL / "ALL_nonredundancy_full_K16_K32.csv", index=False)
pd.concat(all_nonred_summary, ignore_index=True).to_csv(OUT_FINAL / "ALL_nonredundancy_summary_K16_K32.csv", index=False)
pd.concat(all_dist, ignore_index=True).to_csv(OUT_FINAL / "ALL_score_distribution_effect_K16_K32.csv", index=False)

print("\nDONE nonredundancy K16/K32.")


===== K=16 nonredundancy summary =====


,experiment,K,real_auc,shuffle_auc_mean,shuffle_auc_std,shuffle_auc_max,shuffle_auc_min,permutation_p_real_greater_than_shuffle,n_shuffles,delta_real_vs_direct_mean,delta_real_vs_direct_lo,delta_real_vs_direct_hi
0,shuffled_control_summary,16,0.572802,0.51554,0.000518,0.517291,0.513477,0.009901,100,0.057234,0.047447,0.067076


,experiment,feature_set,n_train,n_test,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,mcc,boot_mean,boot_lo,boot_hi,boot_n,K
0,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515656,0.508670,0.522780,1000,16
1,hard_negative_nonredundancy_final,PMCI_K16_only,40320,30240,0.572225,0.064133,0.600397,0.551719,0.106081,0.044976,0.572369,0.560980,0.583605,1000,16
2,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K16,40320,30240,0.572802,0.083430,0.737864,0.542743,0.106213,0.042290,0.572665,0.562683,0.582702,1000,16
3,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515429,0.070568,0.756581,0.508038,0.083655,0.008292,0.515385,0.508452,0.522430,1000,16
4,hard_negative_nonredundancy_final,Direct_plus_random_vector_dim32,40320,30240,0.517101,0.067703,0.746892,0.511198,0.086199,0.011344,0.517198,0.508115,0.526333,1000,16
5,hard_negative_nonredundancy_final,Direct_plus_untrained_PMCI_K16,40320,30240,0.515463,0.070680,0.758168,0.508212,0.083699,0.008495,0.515498,0.508382,0.522980,1000,16



===== K=32 nonredundancy summary =====


,experiment,K,real_auc,shuffle_auc_mean,shuffle_auc_std,shuffle_auc_max,shuffle_auc_min,permutation_p_real_greater_than_shuffle,n_shuffles,delta_real_vs_direct_mean,delta_real_vs_direct_lo,delta_real_vs_direct_hi
0,shuffled_control_summary,32,0.582499,0.515503,0.000418,0.516645,0.513901,0.009901,100,0.067048,0.05856,0.075338


,experiment,feature_set,n_train,n_test,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,mcc,boot_mean,boot_lo,boot_hi,boot_n,K
0,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515386,0.508375,0.522480,1000,32
1,hard_negative_nonredundancy_final,PMCI_K32_only,40320,30240,0.569069,0.055293,0.395899,0.552552,0.102662,0.046249,0.569131,0.558205,0.579976,1000,32
2,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K32,40320,30240,0.582499,0.082229,0.747586,0.554774,0.114193,0.054835,0.582381,0.573435,0.592514,1000,32
3,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515441,0.070549,0.756878,0.507865,0.083520,0.008117,0.515562,0.508424,0.522841,1000,32
4,hard_negative_nonredundancy_final,Direct_plus_random_vector_dim32,40320,30240,0.512371,0.069832,0.751984,0.508264,0.084025,0.008454,0.512476,0.503846,0.520369,1000,32
5,hard_negative_nonredundancy_final,Direct_plus_untrained_PMCI_K32,40320,30240,0.514846,0.071003,0.746759,0.509479,0.085066,0.009606,0.514897,0.507967,0.522598,1000,32



DONE nonredundancy K16/K32.


In [ ]:
# =========================
# Normal/random bank for uncertainty-aware routing
# Show whether low PMCI enriches low-agreement / mismatched cases.
# =========================

def build_random_bank(face_np, body_np, n_random_per_anchor=1, seed=123):
    rng = np.random.default_rng(seed)
    n = len(face_np)
    rows = []
    for i in range(n):
        rows.append({
            "anchor": int(i),
            "face_idx": int(i),
            "body_idx": int(i),
            "y": 1,
            "pair_type": "true_pair"
        })
        for _ in range(n_random_per_anchor):
            j = int(rng.integers(0, n))
            while j == i:
                j = int(rng.integers(0, n))
            rows.append({
                "anchor": int(i),
                "face_idx": int(i),
                "body_idx": int(j),
                "y": 0,
                "pair_type": "random_mismatch"
            })
    return pd.DataFrame(rows)

random_test = build_random_bank(std["test_face"], std["test_body"], n_random_per_anchor=1, seed=9090)

random_test_scores = random_test.copy()
random_test_scores["direct"] = bank_scores(DIRECT_MODEL, std["test_face"], std["test_body"], random_test_scores)

for K in [16, 32]:
    m = get_trained_model(f"PMCI_K{K}")
    random_test_scores[f"pmci_K{K}"] = bank_scores(m, std["test_face"], std["test_body"], random_test_scores)

random_test_scores.to_csv(OUT_FINAL / "random_test_scores_K16_K32.csv", index=False)

print("Random bank:", random_test_scores.shape, "positive rate:", random_test_scores["y"].mean())
display(random_test_scores.head())

Random bank: (2880, 8) positive rate: 0.5


,anchor,face_idx,body_idx,y,pair_type,direct,pmci_K16,pmci_K32
0,0,0,0,1,true_pair,0.892318,0.474894,0.973587
1,0,0,711,0,random_mismatch,0.598700,0.269291,0.284930
2,1,1,1,1,true_pair,0.895245,0.477251,0.970687
3,1,1,765,0,random_mismatch,0.712458,0.772036,0.429303
4,2,2,2,1,true_pair,0.891614,0.581305,0.970380


In [ ]:
# =========================
# Gating / routing analysis
# Reject lowest PMCI fractions and inspect:
# - coverage
# - rejected mismatch rate
# - retained mismatch rate
# - Direct/PMCI AUC on retained subset
# =========================

def gating_routing_analysis(df, pmci_col, base_score_col="direct", fractions=(0, .05, .10, .20, .30, .40, .50), bank_name="random"):
    y = df["y"].to_numpy().astype(int)
    pmci = df[pmci_col].to_numpy()
    base = df[base_score_col].to_numpy()

    rows = []
    for frac in fractions:
        if frac == 0:
            keep = np.ones(len(df), dtype=bool)
            threshold = np.nan
        else:
            threshold = np.quantile(pmci, frac)
            keep = pmci > threshold

        reject = ~keep

        yy = y[keep]
        bb = base[keep]
        pp = pmci[keep]

        row = {
            "experiment": "uncertainty_routing_enrichment",
            "bank": bank_name,
            "pmci_col": pmci_col,
            "reject_lowest_pmci_fraction": frac,
            "coverage": float(np.mean(keep)),
            "retained_pairs": int(np.sum(keep)),
            "rejected_pairs": int(np.sum(reject)),
            "pmci_threshold": float(threshold) if not np.isnan(threshold) else np.nan,
            "rejected_mismatch_rate": float(np.mean(y[reject] == 0)) if np.sum(reject) else np.nan,
            "retained_mismatch_rate": float(np.mean(yy == 0)) if len(yy) else np.nan,
            "rejected_positive_rate": float(np.mean(y[reject] == 1)) if np.sum(reject) else np.nan,
            "retained_positive_rate": float(np.mean(yy == 1)) if len(yy) else np.nan,
            "direct_auc_retained": safe_auc(yy, bb),
            "direct_pr_auc_retained": safe_ap(yy, bb),
            "pmci_auc_retained": safe_auc(yy, pp),
            "pmci_pr_auc_retained": safe_ap(yy, pp),
            "pmci_mean_rejected": float(np.mean(pmci[reject])) if np.sum(reject) else np.nan,
            "pmci_mean_retained": float(np.mean(pmci[keep])) if np.sum(keep) else np.nan,
        }
        rows.append(row)

    return pd.DataFrame(rows)

gating_tables = []
for K in [16, 32]:
    gdf_random = gating_routing_analysis(
        random_test_scores,
        pmci_col=f"pmci_K{K}",
        base_score_col="direct",
        bank_name="random_test"
    )
    gdf_hard = gating_routing_analysis(
        hard_test_scores,
        pmci_col=f"pmci_K{K}",
        base_score_col="direct",
        bank_name="hard_test"
    )
    gating_tables.extend([gdf_random, gdf_hard])

    gdf_random.to_csv(OUT_FINAL / f"gating_routing_random_K{K}.csv", index=False)
    gdf_hard.to_csv(OUT_FINAL / f"gating_routing_hard_K{K}.csv", index=False)

gating_all = pd.concat(gating_tables, ignore_index=True)
gating_all.to_csv(OUT_FINAL / "ALL_gating_routing_K16_K32.csv", index=False)

display(gating_all)

,experiment,bank,pmci_col,reject_lowest_pmci_fraction,coverage,retained_pairs,rejected_pairs,pmci_threshold,rejected_mismatch_rate,retained_mismatch_rate,rejected_positive_rate,retained_positive_rate,direct_auc_retained,direct_pr_auc_retained,pmci_auc_retained,pmci_pr_auc_retained,pmci_mean_rejected,pmci_mean_retained
0,uncertainty_routing_enrichment,random_test,pmci_K16,0.00,1.00,2880,0,NaN,NaN,0.500000,NaN,0.500000,0.937080,0.913006,0.801111,0.730966,NaN,0.577779
1,uncertainty_routing_enrichment,random_test,pmci_K16,0.05,0.95,2736,144,0.124540,1.000000,0.473684,0.000000,0.526316,0.930125,0.913022,0.779012,0.730966,0.117539,0.602002
2,uncertainty_routing_enrichment,random_test,pmci_K16,0.10,0.90,2592,288,0.137350,1.000000,0.444444,0.000000,0.555556,0.921442,0.913045,0.751388,0.730966,0.124151,0.628182
3,uncertainty_routing_enrichment,random_test,pmci_K16,0.20,0.80,2304,576,0.197830,0.975694,0.381076,0.024306,0.618924,0.905417,0.917934,0.681359,0.732264,0.143397,0.686374
4,uncertainty_routing_enrichment,random_test,pmci_K16,0.30,0.70,2016,864,0.321718,0.878472,0.337798,0.121528,0.662202,0.898154,0.925002,0.637690,0.738265,0.180093,0.748216
5,uncertainty_routing_enrichment,random_test,pmci_K16,0.40,0.60,1728,1152,0.525182,0.816840,0.288773,0.183160,0.711227,0.898415,0.941889,0.564799,0.742791,0.240655,0.802528
6,uncertainty_routing_enrichment,random_test,pmci_K16,0.50,0.50,1440,1440,0.688789,0.752778,0.247222,0.247222,0.752778,0.906995,0.961624,0.461610,0.743159,0.316208,0.839350
7,uncertainty_routing_enrichment,hard_test,pmci_K16,0.00,1.00,30240,0,NaN,NaN,0.952381,NaN,0.047619,0.484431,0.045209,0.572225,0.064133,NaN,0.697470
8,uncertainty_routing_enrichment,hard_test,pmci_K16,0.05,0.95,28728,1512,0.264276,0.960979,0.951928,0.039021,0.048072,0.484586,0.045455,0.574217,0.064833,0.218375,0.722685
9,uncertainty_routing_enrichment,hard_test,pmci_K16,0.10,0.90,27216,3024,0.338667,0.960979,0.951426,0.039021,0.048574,0.482475,0.045749,0.576207,0.065573,0.260266,0.746048


In [ ]:
# =========================
# Near-orthogonal probability diagnostic for K16/K32
# Tries to infer pF and pB from PMCI model.
# =========================

@torch.no_grad()
def try_extract_probs(model, face_np, body_np, batch_size=256):
    model.eval()

    pF_all, pB_all = [], []

    # Common attribute guesses
    proto_attr_candidates = ["prototypes", "proto", "C", "centroids"]
    face_gru_candidates = ["face_gru", "gru_face", "facial_gru", "face_encoder"]
    body_gru_candidates = ["body_gru", "gru_body", "gestural_gru", "body_encoder"]
    face_proj_candidates = ["face_proj", "proj_face", "facial_proj"]
    body_proj_candidates = ["body_proj", "proj_body", "gestural_proj"]

    # Try method-based first
    method_candidates = ["get_probs", "probabilities", "prototype_probs", "compute_probs"]
    for meth in method_candidates:
        if hasattr(model, meth):
            fn = getattr(model, meth)
            try:
                for start in range(0, len(face_np), batch_size):
                    f = torch.tensor(np.ascontiguousarray(face_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_FINAL)
                    b = torch.tensor(np.ascontiguousarray(body_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_FINAL)
                    out = fn(f, b)
                    if isinstance(out, (tuple, list)) and len(out) >= 2:
                        pF, pB = out[0], out[1]
                        pF_all.append(pF.detach().cpu().numpy())
                        pB_all.append(pB.detach().cpu().numpy())
                if pF_all:
                    return np.vstack(pF_all), np.vstack(pB_all)
            except Exception:
                pF_all, pB_all = [], []

    # Manual extraction fallback
    proto = None
    for a in proto_attr_candidates:
        if hasattr(model, a):
            proto = getattr(model, a)
            break
    if proto is None:
        raise RuntimeError("Could not find prototype parameter in model.")

    face_gru = None
    body_gru = None
    face_proj = None
    body_proj = None

    for a in face_gru_candidates:
        if hasattr(model, a):
            face_gru = getattr(model, a)
            break
    for a in body_gru_candidates:
        if hasattr(model, a):
            body_gru = getattr(model, a)
            break
    for a in face_proj_candidates:
        if hasattr(model, a):
            face_proj = getattr(model, a)
            break
    for a in body_proj_candidates:
        if hasattr(model, a):
            body_proj = getattr(model, a)
            break

    if face_gru is None or body_gru is None:
        raise RuntimeError("Could not find GRU encoders for probability extraction.")

    tau = float(getattr(model, "tau", 0.25))

    for start in range(0, len(face_np), batch_size):
        f = torch.tensor(np.ascontiguousarray(face_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_FINAL)
        b = torch.tensor(np.ascontiguousarray(body_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_FINAL)

        outF, hF = face_gru(f)
        outB, hB = body_gru(b)

        zF = hF[-1]
        zB = hB[-1]

        if face_proj is not None:
            zF = face_proj(zF)
        if body_proj is not None:
            zB = body_proj(zB)

        zF = torch.nn.functional.normalize(zF, dim=1)
        zB = torch.nn.functional.normalize(zB, dim=1)
        C = torch.nn.functional.normalize(proto, dim=1)

        simF = zF @ C.T
        simB = zB @ C.T

        pF = torch.softmax(simF / tau, dim=1)
        pB = torch.softmax(simB / tau, dim=1)

        pF_all.append(pF.detach().cpu().numpy())
        pB_all.append(pB.detach().cpu().numpy())

    return np.vstack(pF_all), np.vstack(pB_all)

def near_orthogonal_table(df, face_np, body_np, K, bank_name):
    model = get_trained_model(f"PMCI_K{K}")
    fi = df["face_idx"].to_numpy()
    bi = df["body_idx"].to_numpy()
    f = np.ascontiguousarray(face_np[fi], dtype=np.float32)
    b = np.ascontiguousarray(body_np[bi], dtype=np.float32)

    pF, pB = try_extract_probs(model, f, b)
    pmci = df[f"pmci_K{K}"].to_numpy()
    y = df["y"].to_numpy().astype(int)

    prob_dot = np.sum(pF * pB, axis=1)
    argmax_same = np.argmax(pF, axis=1) == np.argmax(pB, axis=1)

    q10 = np.quantile(prob_dot, 0.10)
    low = prob_dot <= q10
    other = ~low

    rows = []
    groups = {
        "near_orthogonal_lowest_10pct_dot": low,
        "other_90pct": other,
        "argmax_same": argmax_same,
        "argmax_different": ~argmax_same,
    }

    for name, mask in groups.items():
        if np.sum(mask) == 0:
            continue
        rows.append({
            "experiment": "near_orthogonal_probability_diagnostic",
            "bank": bank_name,
            "K": K,
            "group": name,
            "n": int(np.sum(mask)),
            "positive_rate": float(np.mean(y[mask])),
            "pmci_mean": float(np.mean(pmci[mask])),
            "pmci_std": float(np.std(pmci[mask])),
            "prob_dot_mean": float(np.mean(prob_dot[mask])),
            "prob_dot_std": float(np.std(prob_dot[mask])),
            "argmax_agreement_rate": float(np.mean(argmax_same[mask])),
        })

    out = pd.DataFrame(rows)
    return out

orth_tables = []
for K in [16, 32]:
    for bank_name, df, face_np, body_np in [
        ("random_test", random_test_scores, std["test_face"], std["test_body"]),
        ("hard_test", hard_test_scores, std["test_face"], std["test_body"]),
    ]:
        try:
            odf = near_orthogonal_table(df, face_np, body_np, K, bank_name)
            odf.to_csv(OUT_FINAL / f"near_orthogonal_{bank_name}_K{K}.csv", index=False)
            orth_tables.append(odf)
            print(f"Near-orthogonal done: K={K}, bank={bank_name}")
            display(odf)
        except Exception as e:
            print(f"Near-orthogonal failed for K={K}, bank={bank_name}: {e}")

if orth_tables:
    pd.concat(orth_tables, ignore_index=True).to_csv(OUT_FINAL / "ALL_near_orthogonal_K16_K32.csv", index=False)

Near-orthogonal done: K=16, bank=random_test


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,random_test,16,near_orthogonal_lowest_10pct_dot,288,0.000000,0.126153,0.010941,0.004837,0.000681,0.000000
1,near_orthogonal_probability_diagnostic,random_test,16,other_90pct,2592,0.555556,0.627959,0.270906,0.140042,0.092894,0.124228
2,near_orthogonal_probability_diagnostic,random_test,16,argmax_same,322,0.819876,0.906969,0.060804,0.270204,0.054568,1.000000
3,near_orthogonal_probability_diagnostic,random_test,16,argmax_different,2558,0.459734,0.536340,0.289950,0.108435,0.085415,0.000000


Near-orthogonal done: K=16, bank=hard_test


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,hard_test,16,near_orthogonal_lowest_10pct_dot,3024,0.038029,0.262070,0.052810,0.018992,0.007101,0.000000
1,near_orthogonal_probability_diagnostic,hard_test,16,other_90pct,27216,0.048685,0.745848,0.155536,0.172951,0.064420,0.132018
2,near_orthogonal_probability_diagnostic,hard_test,16,argmax_same,3593,0.073476,0.892663,0.067115,0.256298,0.047695,1.000000
3,near_orthogonal_probability_diagnostic,hard_test,16,argmax_different,26647,0.044133,0.671151,0.206134,0.144241,0.069761,0.000000


Near-orthogonal done: K=32, bank=random_test


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,random_test,32,near_orthogonal_lowest_10pct_dot,288,0.000000,0.115301,0.009710,0.001986,0.000257,0.000000
1,near_orthogonal_probability_diagnostic,random_test,32,other_90pct,2592,0.555556,0.660276,0.271382,0.071312,0.044558,0.071759
2,near_orthogonal_probability_diagnostic,random_test,32,argmax_same,186,0.865591,0.932678,0.031303,0.126312,0.015019,1.000000
3,near_orthogonal_probability_diagnostic,random_test,32,argmax_different,2694,0.474759,0.583209,0.302474,0.060103,0.045541,0.000000


Near-orthogonal done: K=32, bank=hard_test


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,hard_test,32,near_orthogonal_lowest_10pct_dot,3024,0.022487,0.371856,0.121511,0.019732,0.010410,0.000000
1,near_orthogonal_probability_diagnostic,hard_test,32,other_90pct,27216,0.050412,0.823938,0.106812,0.098270,0.026583,0.104314
2,near_orthogonal_probability_diagnostic,hard_test,32,argmax_same,2839,0.056710,0.939849,0.033883,0.130081,0.014131,1.000000
3,near_orthogonal_probability_diagnostic,hard_test,32,argmax_different,27401,0.046677,0.762036,0.173706,0.086307,0.033554,0.000000


In [ ]:


summary_files = []

# Nonredundancy summaries
for K in [16, 32]:
    p = OUT_FINAL / f"nonredundancy_summary_K{K}.csv"
    if p.exists():
        summary_files.append(pd.read_csv(p))

summary_nonred = pd.concat(summary_files, ignore_index=True) if summary_files else pd.DataFrame()

# Key feature rows
feature_rows = []
for K in [16, 32]:
    p = OUT_FINAL / f"nonredundancy_full_K{K}.csv"
    if p.exists():
        df = pd.read_csv(p)
        keep_names = [
            "Direct_only",
            f"PMCI_K{K}_only",
            f"Direct_plus_real_PMCI_K{K}",
            "Direct_plus_random_scalar",
            f"Direct_plus_random_vector_dim{max(K, RANDOM_VECTOR_DIM)}",
            f"Direct_plus_untrained_PMCI_K{K}"
        ]
        feature_rows.append(df[df["feature_set"].isin(keep_names)])

summary_features = pd.concat(feature_rows, ignore_index=True) if feature_rows else pd.DataFrame()

# Gating summary: focus random bank
gating_summary = gating_all[
    (gating_all["bank"] == "random_test") &
    (gating_all["reject_lowest_pmci_fraction"].isin([0, 0.1, 0.2, 0.3]))
].copy()

# Orthogonal summary
orth_path = OUT_FINAL / "ALL_near_orthogonal_K16_K32.csv"
orth_summary = pd.read_csv(orth_path) if orth_path.exists() else pd.DataFrame()

summary_nonred.to_csv(OUT_FINAL / "PAPER_TABLE_nonredundancy_summary.csv", index=False)
summary_features.to_csv(OUT_FINAL / "PAPER_TABLE_nonredundancy_key_rows.csv", index=False)
gating_summary.to_csv(OUT_FINAL / "PAPER_TABLE_gating_random_summary.csv", index=False)
orth_summary.to_csv(OUT_FINAL / "PAPER_TABLE_near_orthogonal_summary.csv", index=False)

print("Saved paper-ready tables:")
print(OUT_FINAL / "PAPER_TABLE_nonredundancy_summary.csv")
print(OUT_FINAL / "PAPER_TABLE_nonredundancy_key_rows.csv")
print(OUT_FINAL / "PAPER_TABLE_gating_random_summary.csv")
print(OUT_FINAL / "PAPER_TABLE_near_orthogonal_summary.csv")

print("\nNonredundancy summary:")
display(summary_nonred)

print("\nKey feature rows:")
display(summary_features)

print("\nGating summary:")
display(gating_summary)

print("\nNear-orthogonal summary:")
display(orth_summary)

Saved paper-ready tables:
/content/pmci_revision_results_final_extra/PAPER_TABLE_nonredundancy_summary.csv
/content/pmci_revision_results_final_extra/PAPER_TABLE_nonredundancy_key_rows.csv
/content/pmci_revision_results_final_extra/PAPER_TABLE_gating_random_summary.csv
/content/pmci_revision_results_final_extra/PAPER_TABLE_near_orthogonal_summary.csv

Nonredundancy summary:


,experiment,K,real_auc,shuffle_auc_mean,shuffle_auc_std,shuffle_auc_max,shuffle_auc_min,permutation_p_real_greater_than_shuffle,n_shuffles,delta_real_vs_direct_mean,delta_real_vs_direct_lo,delta_real_vs_direct_hi
0,shuffled_control_summary,16,0.572802,0.515540,0.000518,0.517291,0.513477,0.009901,100,0.057234,0.047447,0.067076
1,shuffled_control_summary,32,0.582499,0.515503,0.000418,0.516645,0.513901,0.009901,100,0.067048,0.058560,0.075338



Key feature rows:


,experiment,feature_set,n_train,n_test,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,mcc,boot_mean,boot_lo,boot_hi,boot_n,K
0,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515656,0.508670,0.522780,1000,16
1,hard_negative_nonredundancy_final,PMCI_K16_only,40320,30240,0.572225,0.064133,0.600397,0.551719,0.106081,0.044976,0.572369,0.560980,0.583605,1000,16
2,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K16,40320,30240,0.572802,0.083430,0.737864,0.542743,0.106213,0.042290,0.572665,0.562683,0.582702,1000,16
3,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515429,0.070568,0.756581,0.508038,0.083655,0.008292,0.515385,0.508452,0.522430,1000,16
4,hard_negative_nonredundancy_final,Direct_plus_random_vector_dim32,40320,30240,0.517101,0.067703,0.746892,0.511198,0.086199,0.011344,0.517198,0.508115,0.526333,1000,16
5,hard_negative_nonredundancy_final,Direct_plus_untrained_PMCI_K16,40320,30240,0.515463,0.070680,0.758168,0.508212,0.083699,0.008495,0.515498,0.508382,0.522980,1000,16
6,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515386,0.508375,0.522480,1000,32
7,hard_negative_nonredundancy_final,PMCI_K32_only,40320,30240,0.569069,0.055293,0.395899,0.552552,0.102662,0.046249,0.569131,0.558205,0.579976,1000,32
8,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K32,40320,30240,0.582499,0.082229,0.747586,0.554774,0.114193,0.054835,0.582381,0.573435,0.592514,1000,32
9,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515441,0.070549,0.756878,0.507865,0.083520,0.008117,0.515562,0.508424,0.522841,1000,32



Gating summary:


,experiment,bank,pmci_col,reject_lowest_pmci_fraction,coverage,retained_pairs,rejected_pairs,pmci_threshold,rejected_mismatch_rate,retained_mismatch_rate,rejected_positive_rate,retained_positive_rate,direct_auc_retained,direct_pr_auc_retained,pmci_auc_retained,pmci_pr_auc_retained,pmci_mean_rejected,pmci_mean_retained
0,uncertainty_routing_enrichment,random_test,pmci_K16,0.0,1.0,2880,0,NaN,NaN,0.500000,NaN,0.500000,0.937080,0.913006,0.801111,0.730966,NaN,0.577779
2,uncertainty_routing_enrichment,random_test,pmci_K16,0.1,0.9,2592,288,0.137350,1.000000,0.444444,0.000000,0.555556,0.921442,0.913045,0.751388,0.730966,0.124151,0.628182
3,uncertainty_routing_enrichment,random_test,pmci_K16,0.2,0.8,2304,576,0.197830,0.975694,0.381076,0.024306,0.618924,0.905417,0.917934,0.681359,0.732264,0.143397,0.686374
4,uncertainty_routing_enrichment,random_test,pmci_K16,0.3,0.7,2016,864,0.321718,0.878472,0.337798,0.121528,0.662202,0.898154,0.925002,0.637690,0.738265,0.180093,0.748216
14,uncertainty_routing_enrichment,random_test,pmci_K32,0.0,1.0,2880,0,NaN,NaN,0.500000,NaN,0.500000,0.937080,0.913006,0.895530,0.857839,NaN,0.605779
16,uncertainty_routing_enrichment,random_test,pmci_K32,0.1,0.9,2592,288,0.128520,1.000000,0.444444,0.000000,0.555556,0.921430,0.913040,0.869412,0.857839,0.113621,0.660463
17,uncertainty_routing_enrichment,random_test,pmci_K32,0.2,0.8,2304,576,0.206379,1.000000,0.375000,0.000000,0.625000,0.899816,0.915207,0.825883,0.857839,0.137806,0.722772
18,uncertainty_routing_enrichment,random_test,pmci_K32,0.3,0.7,2016,864,0.379488,0.981481,0.293651,0.018519,0.706349,0.871875,0.921392,0.756487,0.859952,0.186048,0.785663



Near-orthogonal summary:


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,random_test,16,near_orthogonal_lowest_10pct_dot,288,0.000000,0.126153,0.010941,0.004837,0.000681,0.000000
1,near_orthogonal_probability_diagnostic,random_test,16,other_90pct,2592,0.555556,0.627959,0.270906,0.140042,0.092894,0.124228
2,near_orthogonal_probability_diagnostic,random_test,16,argmax_same,322,0.819876,0.906969,0.060804,0.270204,0.054568,1.000000
3,near_orthogonal_probability_diagnostic,random_test,16,argmax_different,2558,0.459734,0.536340,0.289950,0.108435,0.085415,0.000000
4,near_orthogonal_probability_diagnostic,hard_test,16,near_orthogonal_lowest_10pct_dot,3024,0.038029,0.262070,0.052810,0.018992,0.007101,0.000000
5,near_orthogonal_probability_diagnostic,hard_test,16,other_90pct,27216,0.048685,0.745848,0.155536,0.172951,0.064420,0.132018
6,near_orthogonal_probability_diagnostic,hard_test,16,argmax_same,3593,0.073476,0.892663,0.067115,0.256298,0.047695,1.000000
7,near_orthogonal_probability_diagnostic,hard_test,16,argmax_different,26647,0.044133,0.671151,0.206134,0.144241,0.069761,0.000000
8,near_orthogonal_probability_diagnostic,random_test,32,near_orthogonal_lowest_10pct_dot,288,0.000000,0.115301,0.009710,0.001986,0.000257,0.000000
9,near_orthogonal_probability_diagnostic,random_test,32,other_90pct,2592,0.555556,0.660276,0.271382,0.071312,0.044558,0.071759


In [ ]:
# =========================
# Zip all final extra results
# =========================

import shutil

zip_path = shutil.make_archive(
    "/content/pmci_revision_results_final_extra",
    "zip",
    str(OUT_FINAL)
)

print("DONE ✅")
print("Final extra results:", OUT_FINAL)
print("ZIP:", zip_path)

DONE ✅
Final extra results: /content/pmci_revision_results_final_extra
ZIP: /content/pmci_revision_results_final_extra.zip


In [ ]:
import os, json, platform, subprocess, sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

OUT_AUDIT = Path("/content/pmci_final_audit_pack")
OUT_AUDIT.mkdir(parents=True, exist_ok=True)

def get_pkg_version(pkg):
    try:
        mod = __import__(pkg)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not_installed"

def shell(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as e:
        return str(e)

audit = {
    "created_at": datetime.now().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": get_pkg_version("sklearn"),
    "scipy": get_pkg_version("scipy"),
    "ultralytics": get_pkg_version("ultralytics"),
}

if "std" in globals():
    audit["dataset_shapes"] = {
        k: list(v.shape) for k, v in std.items()
        if hasattr(v, "shape")
    }
    for k in ["train_face", "val_face", "test_face"]:
        if k in std:
            audit[f"n_{k.replace('_face','')}"] = int(len(std[k]))

if "trained_models" in globals():
    audit["trained_models"] = list(trained_models.keys())

with open(OUT_AUDIT / "environment_and_data_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

pd.DataFrame([
    {"item": k, "value": str(v)} for k, v in audit.items()
]).to_csv(OUT_AUDIT / "environment_and_data_audit.csv", index=False)

print("Saved:", OUT_AUDIT)
display(pd.DataFrame([{"item": k, "value": str(v)} for k, v in audit.items()]))

Saved: /content/pmci_final_audit_pack


,item,value
0,created_at,2026-07-11T19:56:50.581127
1,python,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
2,platform,Linux-6.6.122+-x86_64-with-glibc2.35
3,torch,2.11.0+cu128
4,cuda_available,True
5,cuda_version,12.8
6,gpu_name,NVIDIA A100-SXM4-80GB
7,numpy,2.0.2
8,pandas,2.2.2
9,sklearn,1.6.1


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

candidate_dirs = [
    Path("/content/pmci_revision_results_final_extra"),
    Path("/content/pmci_revision_results_v6"),
    Path("/content/pmci_final_extra"),
]

RESULT_DIR = None
for d in candidate_dirs:
    if d.exists():
        RESULT_DIR = d
        break

if RESULT_DIR is None:
    raise RuntimeError("Не нашёл папку с результатами. Проверь путь.")

print("Using result dir:", RESULT_DIR)

def read_if_exists(name):
    p = RESULT_DIR / name
    if p.exists():
        return pd.read_csv(p)
    print("Missing:", p)
    return pd.DataFrame()

nonred_summary = read_if_exists("PAPER_TABLE_nonredundancy_summary.csv")
nonred_key = read_if_exists("PAPER_TABLE_nonredundancy_key_rows.csv")
gating = read_if_exists("PAPER_TABLE_gating_random_summary.csv")
orth = read_if_exists("PAPER_TABLE_near_orthogonal_summary.csv")

paper_numbers = {}

if not nonred_summary.empty:
    for _, r in nonred_summary.iterrows():
        K = int(r["K"])
        paper_numbers[f"K{K}_real_auc"] = float(r["real_auc"])
        paper_numbers[f"K{K}_shuffle_auc_mean"] = float(r["shuffle_auc_mean"])
        paper_numbers[f"K{K}_permutation_p"] = float(r["permutation_p_real_greater_than_shuffle"])
        paper_numbers[f"K{K}_delta_auc_mean"] = float(r["delta_real_vs_direct_mean"])
        paper_numbers[f"K{K}_delta_auc_ci"] = [
            float(r["delta_real_vs_direct_lo"]),
            float(r["delta_real_vs_direct_hi"])
        ]

if not nonred_key.empty:
    selected = nonred_key[
        nonred_key["feature_set"].isin([
            "Direct_only",
            "PMCI_K16_only",
            "PMCI_K32_only",
            "Direct_plus_real_PMCI_K16",
            "Direct_plus_real_PMCI_K32",
            "Direct_plus_random_scalar",
            "Direct_plus_random_vector_dim32",
            "Direct_plus_untrained_PMCI_K16",
            "Direct_plus_untrained_PMCI_K32",
        ])
    ].copy()
    selected.to_csv(OUT_AUDIT / "paper_key_nonredundancy_rows.csv", index=False)
    display(selected)

if not gating.empty:
    gating.to_csv(OUT_AUDIT / "paper_key_gating_rows.csv", index=False)
    display(gating)

if not orth.empty:
    orth.to_csv(OUT_AUDIT / "paper_key_near_orthogonal_rows.csv", index=False)
    display(orth)

with open(OUT_AUDIT / "paper_key_numbers.json", "w") as f:
    json.dump(paper_numbers, f, indent=2)

pd.DataFrame([
    {"key": k, "value": str(v)} for k, v in paper_numbers.items()
]).to_csv(OUT_AUDIT / "paper_key_numbers.csv", index=False)

print("Saved key numbers to:", OUT_AUDIT)

Using result dir: /content/pmci_revision_results_final_extra


,experiment,feature_set,n_train,n_test,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,mcc,boot_mean,boot_lo,boot_hi,boot_n,K
0,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515656,0.508670,0.522780,1000,16
1,hard_negative_nonredundancy_final,PMCI_K16_only,40320,30240,0.572225,0.064133,0.600397,0.551719,0.106081,0.044976,0.572369,0.560980,0.583605,1000,16
2,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K16,40320,30240,0.572802,0.083430,0.737864,0.542743,0.106213,0.042290,0.572665,0.562683,0.582702,1000,16
3,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515429,0.070568,0.756581,0.508038,0.083655,0.008292,0.515385,0.508452,0.522430,1000,16
4,hard_negative_nonredundancy_final,Direct_plus_random_vector_dim32,40320,30240,0.517101,0.067703,0.746892,0.511198,0.086199,0.011344,0.517198,0.508115,0.526333,1000,16
5,hard_negative_nonredundancy_final,Direct_plus_untrained_PMCI_K16,40320,30240,0.515463,0.070680,0.758168,0.508212,0.083699,0.008495,0.515498,0.508382,0.522980,1000,16
6,hard_negative_nonredundancy_final,Direct_only,40320,30240,0.515569,0.070577,0.756944,0.508559,0.083998,0.008833,0.515386,0.508375,0.522480,1000,32
7,hard_negative_nonredundancy_final,PMCI_K32_only,40320,30240,0.569069,0.055293,0.395899,0.552552,0.102662,0.046249,0.569131,0.558205,0.579976,1000,32
8,hard_negative_nonredundancy_final,Direct_plus_real_PMCI_K32,40320,30240,0.582499,0.082229,0.747586,0.554774,0.114193,0.054835,0.582381,0.573435,0.592514,1000,32
9,hard_negative_nonredundancy_final,Direct_plus_random_scalar,40320,30240,0.515441,0.070549,0.756878,0.507865,0.083520,0.008117,0.515562,0.508424,0.522841,1000,32


,experiment,bank,pmci_col,reject_lowest_pmci_fraction,coverage,retained_pairs,rejected_pairs,pmci_threshold,rejected_mismatch_rate,retained_mismatch_rate,rejected_positive_rate,retained_positive_rate,direct_auc_retained,direct_pr_auc_retained,pmci_auc_retained,pmci_pr_auc_retained,pmci_mean_rejected,pmci_mean_retained
0,uncertainty_routing_enrichment,random_test,pmci_K16,0.0,1.0,2880,0,NaN,NaN,0.500000,NaN,0.500000,0.937080,0.913006,0.801111,0.730966,NaN,0.577779
1,uncertainty_routing_enrichment,random_test,pmci_K16,0.1,0.9,2592,288,0.137350,1.000000,0.444444,0.000000,0.555556,0.921442,0.913045,0.751388,0.730966,0.124151,0.628182
2,uncertainty_routing_enrichment,random_test,pmci_K16,0.2,0.8,2304,576,0.197830,0.975694,0.381076,0.024306,0.618924,0.905417,0.917934,0.681359,0.732264,0.143397,0.686374
3,uncertainty_routing_enrichment,random_test,pmci_K16,0.3,0.7,2016,864,0.321718,0.878472,0.337798,0.121528,0.662202,0.898154,0.925002,0.637690,0.738265,0.180093,0.748216
4,uncertainty_routing_enrichment,random_test,pmci_K32,0.0,1.0,2880,0,NaN,NaN,0.500000,NaN,0.500000,0.937080,0.913006,0.895530,0.857839,NaN,0.605779
5,uncertainty_routing_enrichment,random_test,pmci_K32,0.1,0.9,2592,288,0.128520,1.000000,0.444444,0.000000,0.555556,0.921430,0.913040,0.869412,0.857839,0.113621,0.660463
6,uncertainty_routing_enrichment,random_test,pmci_K32,0.2,0.8,2304,576,0.206379,1.000000,0.375000,0.000000,0.625000,0.899816,0.915207,0.825883,0.857839,0.137806,0.722772
7,uncertainty_routing_enrichment,random_test,pmci_K32,0.3,0.7,2016,864,0.379488,0.981481,0.293651,0.018519,0.706349,0.871875,0.921392,0.756487,0.859952,0.186048,0.785663


,experiment,bank,K,group,n,positive_rate,pmci_mean,pmci_std,prob_dot_mean,prob_dot_std,argmax_agreement_rate
0,near_orthogonal_probability_diagnostic,random_test,16,near_orthogonal_lowest_10pct_dot,288,0.000000,0.126153,0.010941,0.004837,0.000681,0.000000
1,near_orthogonal_probability_diagnostic,random_test,16,other_90pct,2592,0.555556,0.627959,0.270906,0.140042,0.092894,0.124228
2,near_orthogonal_probability_diagnostic,random_test,16,argmax_same,322,0.819876,0.906969,0.060804,0.270204,0.054568,1.000000
3,near_orthogonal_probability_diagnostic,random_test,16,argmax_different,2558,0.459734,0.536340,0.289950,0.108435,0.085415,0.000000
4,near_orthogonal_probability_diagnostic,hard_test,16,near_orthogonal_lowest_10pct_dot,3024,0.038029,0.262070,0.052810,0.018992,0.007101,0.000000
5,near_orthogonal_probability_diagnostic,hard_test,16,other_90pct,27216,0.048685,0.745848,0.155536,0.172951,0.064420,0.132018
6,near_orthogonal_probability_diagnostic,hard_test,16,argmax_same,3593,0.073476,0.892663,0.067115,0.256298,0.047695,1.000000
7,near_orthogonal_probability_diagnostic,hard_test,16,argmax_different,26647,0.044133,0.671151,0.206134,0.144241,0.069761,0.000000
8,near_orthogonal_probability_diagnostic,random_test,32,near_orthogonal_lowest_10pct_dot,288,0.000000,0.115301,0.009710,0.001986,0.000257,0.000000
9,near_orthogonal_probability_diagnostic,random_test,32,other_90pct,2592,0.555556,0.660276,0.271382,0.071312,0.044558,0.071759


Saved key numbers to: /content/pmci_final_audit_pack


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def read_if_exists_dir(name):
    p = RESULT_DIR / name
    if p.exists():
        return pd.read_csv(p)
    return pd.DataFrame()

hard_scores = read_if_exists_dir("hard_test_scores_K16_K32.csv")
random_scores = read_if_exists_dir("random_test_scores_K16_K32.csv")

qual_rows = []

def collect_extremes(df, bank_name, K, n=20):
    if df.empty:
        return pd.DataFrame()
    col = f"pmci_K{K}"
    if col not in df.columns:
        return pd.DataFrame()

    low = df.sort_values(col, ascending=True).head(n).copy()
    low["extreme_type"] = f"lowest_PMCI_K{K}"
    low["bank"] = bank_name

    high = df.sort_values(col, ascending=False).head(n).copy()
    high["extreme_type"] = f"highest_PMCI_K{K}"
    high["bank"] = bank_name

    # hard false-like zones: positives with low PMCI, negatives with high PMCI
    pos_low = df[df["y"] == 1].sort_values(col, ascending=True).head(n).copy()
    pos_low["extreme_type"] = f"true_pairs_low_PMCI_K{K}"
    pos_low["bank"] = bank_name

    neg_high = df[df["y"] == 0].sort_values(col, ascending=False).head(n).copy()
    neg_high["extreme_type"] = f"mismatches_high_PMCI_K{K}"
    neg_high["bank"] = bank_name

    return pd.concat([low, high, pos_low, neg_high], ignore_index=True)

all_extreme = []
for bank_name, df in [("hard_test", hard_scores), ("random_test", random_scores)]:
    for K in [16, 32]:
        all_extreme.append(collect_extremes(df, bank_name, K, n=20))

qual_df = pd.concat(all_extreme, ignore_index=True)
qual_df.to_csv(OUT_AUDIT / "qualitative_extreme_cases_PMCI_K16_K32.csv", index=False)

print("Saved qualitative extremes:", OUT_AUDIT / "qualitative_extreme_cases_PMCI_K16_K32.csv")
display(qual_df.head(30))

Saved qualitative extremes: /content/pmci_final_audit_pack/qualitative_extreme_cases_PMCI_K16_K32.csv


,anchor,face_idx,body_idx,y,pair_type,direct,pmci_K16,pmci_K32,split,extreme_type,bank
0,1360,1360,1242,0,direct_mined_hard_negative,0.779033,0.127600,0.189860,test,lowest_PMCI_K16,hard_test
1,1120,1120,1242,0,direct_mined_hard_negative,0.784107,0.129124,0.188220,test,lowest_PMCI_K16,hard_test
2,682,682,1180,0,direct_mined_hard_negative,0.609425,0.130493,0.144853,test,lowest_PMCI_K16,hard_test
3,1347,1347,1252,0,direct_mined_hard_negative,0.779582,0.131112,0.147459,test,lowest_PMCI_K16,hard_test
4,1347,1347,1012,0,direct_mined_hard_negative,0.778790,0.131194,0.145899,test,lowest_PMCI_K16,hard_test
5,682,682,1420,0,direct_mined_hard_negative,0.611081,0.131361,0.144218,test,lowest_PMCI_K16,hard_test
6,1347,1347,988,0,direct_mined_hard_negative,0.786875,0.131914,0.202507,test,lowest_PMCI_K16,hard_test
7,1347,1347,1228,0,direct_mined_hard_negative,0.783408,0.132323,0.220696,test,lowest_PMCI_K16,hard_test
8,1353,1353,1055,0,direct_mined_hard_negative,0.847553,0.132516,0.134467,test,lowest_PMCI_K16,hard_test
9,1107,1107,1252,0,direct_mined_hard_negative,0.761624,0.132613,0.144821,test,lowest_PMCI_K16,hard_test


In [ ]:
from pathlib import Path
import json
import pandas as pd

with open(OUT_AUDIT / "paper_key_numbers.json") as f:
    nums = json.load(f)

report = []

report.append("# PMCI Final Evidence Pack\n")
report.append("## Core interpretation\n")
report.append(
    "PMCI is evaluated as an auxiliary diagnostic index of cross-modal affective agreement, "
    "not as an emotion classifier and not as a replacement for DirectCosine or LateFusionMLP.\n"
)

report.append("## Hard-negative non-redundancy\n")

for K in [16, 32]:
    if f"K{K}_real_auc" in nums:
        ci = nums[f"K{K}_delta_auc_ci"]
        report.append(
            f"- PMCI_K{K}: real AUC = {nums[f'K{K}_real_auc']:.4f}; "
            f"shuffled mean AUC = {nums[f'K{K}_shuffle_auc_mean']:.4f}; "
            f"permutation p = {nums[f'K{K}_permutation_p']:.4f}; "
            f"ΔAUC over Direct-only = {nums[f'K{K}_delta_auc_mean']:.4f}, "
            f"95% CI [{ci[0]:.4f}, {ci[1]:.4f}]."
        )

report.append("\n## Reviewer-oriented conclusion\n")
report.append(
    "DirectCosine remains stronger for ordinary exact pair matching. "
    "However, under DirectCosine-mined hard negatives, DirectCosine becomes close to chance, "
    "whereas PMCI retains diagnostic separation. Random scalar, random vector, shuffled-PMCI, "
    "and untrained-PMCI controls remain near chance, supporting that PMCI captures learned "
    "cross-modal probabilistic structure rather than merely adding feature dimensionality."
)

report.append("\n## Uncertainty-aware routing\n")
report.append(
    "Low PMCI should be interpreted as a low-agreement signal suitable for cautious response, "
    "delayed interpretation, or human review. It should not be framed as a universal "
    "AUC-improving gating mechanism."
)

report.append("\n## Near-orthogonal diagnostic\n")
report.append(
    "Near-orthogonal modality-specific prototype distributions produce low PMCI values, "
    "while same-argmax prototype assignments produce high PMCI values. This confirms that PMCI "
    "does not force artificial agreement when pF and pB are highly divergent."
)

md = "\n".join(report)

with open(OUT_AUDIT / "PMCI_FINAL_EVIDENCE_REPORT.md", "w") as f:
    f.write(md)

print(md)
print("\nSaved:", OUT_AUDIT / "PMCI_FINAL_EVIDENCE_REPORT.md")

# PMCI Final Evidence Pack

## Core interpretation

PMCI is evaluated as an auxiliary diagnostic index of cross-modal affective agreement, not as an emotion classifier and not as a replacement for DirectCosine or LateFusionMLP.

## Hard-negative non-redundancy

- PMCI_K16: real AUC = 0.5728; shuffled mean AUC = 0.5155; permutation p = 0.0099; ΔAUC over Direct-only = 0.0572, 95% CI [0.0474, 0.0671].
- PMCI_K32: real AUC = 0.5825; shuffled mean AUC = 0.5155; permutation p = 0.0099; ΔAUC over Direct-only = 0.0670, 95% CI [0.0586, 0.0753].

## Reviewer-oriented conclusion

DirectCosine remains stronger for ordinary exact pair matching. However, under DirectCosine-mined hard negatives, DirectCosine becomes close to chance, whereas PMCI retains diagnostic separation. Random scalar, random vector, shuffled-PMCI, and untrained-PMCI controls remain near chance, supporting that PMCI captures learned cross-modal probabilistic structure rather than merely adding feature dimensionality.

## Uncerta

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/pmci_final_audit_pack",
    "zip",
    str(OUT_AUDIT)
)

print("DONE ✅")
print("Audit pack:", OUT_AUDIT)
print("ZIP:", zip_path)

DONE ✅
Audit pack: /content/pmci_final_audit_pack
ZIP: /content/pmci_final_audit_pack.zip


In [ ]:
import os, json, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

OUT_LAST = Path("/content/pmci_last_extra_checks")
OUT_LAST.mkdir(parents=True, exist_ok=True)

DEVICE_LAST = globals().get(
    "DEVICE_TORCH",
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

print("Output:", OUT_LAST)
print("Device:", DEVICE_LAST)

required = ["std", "trained_models", "score_pairs"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Run previous PMCI notebook first.")

def unwrap_model(obj):
    if isinstance(obj, dict):
        return obj.get("model", obj.get("net", obj))
    return obj

def get_model(name):
    m = unwrap_model(trained_models[name])
    m.to(DEVICE_LAST)
    m.eval()
    return m

def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def bank_scores(model, face_np, body_np, bank_df, batch_size=512):
    fi = bank_df["face_idx"].to_numpy()
    bi = bank_df["body_idx"].to_numpy()
    f = np.ascontiguousarray(face_np[fi], dtype=np.float32)
    b = np.ascontiguousarray(body_np[bi], dtype=np.float32)
    return np.asarray(score_pairs(model, f, b, batch_size=batch_size)).reshape(-1)

print("Setup ready.")

Output: /content/pmci_last_extra_checks
Device: cuda
Setup ready.


In [ ]:
@torch.no_grad()
def extract_probs_manual(model, face_np, body_np, batch_size=256):
    model.eval()
    pF_all, pB_all = [], []

    # Try model method first
    for meth in ["get_probs", "probabilities", "prototype_probs", "compute_probs"]:
        if hasattr(model, meth):
            try:
                fn = getattr(model, meth)
                for start in range(0, len(face_np), batch_size):
                    f = torch.tensor(np.ascontiguousarray(face_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_LAST)
                    b = torch.tensor(np.ascontiguousarray(body_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_LAST)
                    out = fn(f, b)
                    if isinstance(out, (tuple, list)) and len(out) >= 2:
                        pF_all.append(out[0].detach().cpu().numpy())
                        pB_all.append(out[1].detach().cpu().numpy())
                if pF_all:
                    return np.vstack(pF_all), np.vstack(pB_all)
            except Exception:
                pF_all, pB_all = [], []

    proto = None
    for a in ["prototypes", "proto", "C", "centroids"]:
        if hasattr(model, a):
            proto = getattr(model, a)
            break
    if proto is None:
        raise RuntimeError("Could not find prototypes in model.")

    face_gru = None
    body_gru = None
    for a in ["face_gru", "gru_face", "facial_gru", "face_encoder"]:
        if hasattr(model, a):
            face_gru = getattr(model, a)
            break
    for a in ["body_gru", "gru_body", "gestural_gru", "body_encoder"]:
        if hasattr(model, a):
            body_gru = getattr(model, a)
            break

    face_proj = None
    body_proj = None
    for a in ["face_proj", "proj_face", "facial_proj"]:
        if hasattr(model, a):
            face_proj = getattr(model, a)
            break
    for a in ["body_proj", "proj_body", "gestural_proj"]:
        if hasattr(model, a):
            body_proj = getattr(model, a)
            break

    if face_gru is None or body_gru is None:
        raise RuntimeError("Could not find GRU encoders.")

    tau = float(getattr(model, "tau", 0.25))

    for start in range(0, len(face_np), batch_size):
        f = torch.tensor(np.ascontiguousarray(face_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_LAST)
        b = torch.tensor(np.ascontiguousarray(body_np[start:start+batch_size]), dtype=torch.float32, device=DEVICE_LAST)

        _, hF = face_gru(f)
        _, hB = body_gru(b)

        zF = hF[-1]
        zB = hB[-1]

        if face_proj is not None:
            zF = face_proj(zF)
        if body_proj is not None:
            zB = body_proj(zB)

        zF = torch.nn.functional.normalize(zF, dim=1)
        zB = torch.nn.functional.normalize(zB, dim=1)
        C = torch.nn.functional.normalize(proto, dim=1)

        simF = zF @ C.T
        simB = zB @ C.T

        pF = torch.softmax(simF / tau, dim=1)
        pB = torch.softmax(simB / tau, dim=1)

        pF_all.append(pF.detach().cpu().numpy())
        pB_all.append(pB.detach().cpu().numpy())

    return np.vstack(pF_all), np.vstack(pB_all)


def divergence_scores(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    q = q / q.sum(axis=1, keepdims=True)

    m = 0.5 * (p + q)

    kl_pm = np.sum(p * np.log(p / m), axis=1)
    kl_qm = np.sum(q * np.log(q / m), axis=1)
    jsd = 0.5 * (kl_pm + kl_qm)

    hell = np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2, axis=1))
    tv = 0.5 * np.sum(np.abs(p - q), axis=1)
    dot = np.sum(p * q, axis=1)

    cosine_prob = dot / (np.linalg.norm(p, axis=1) * np.linalg.norm(q, axis=1) + eps)
    overlap = np.sum(np.minimum(p, q), axis=1)

    return {
        "PMCI_JSD": 1.0 - jsd,
        "Agreement_Hellinger": 1.0 - hell,
        "Agreement_TotalVariation": 1.0 - tv,
        "Prob_Dot": dot,
        "Prob_Cosine": cosine_prob,
        "Prob_Overlap": overlap,
    }


def make_random_bank(n, seed=123):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        rows.append({"anchor": i, "face_idx": i, "body_idx": i, "y": 1, "pair_type": "true_pair"})
        j = int(rng.integers(0, n))
        while j == i:
            j = int(rng.integers(0, n))
        rows.append({"anchor": i, "face_idx": i, "body_idx": j, "y": 0, "pair_type": "random_mismatch"})
    return pd.DataFrame(rows)


random_bank = make_random_bank(len(std["test_face"]), seed=12345)

div_rows = []

for K in [16, 32]:
    model = get_model(f"PMCI_K{K}")

    fi = random_bank["face_idx"].to_numpy()
    bi = random_bank["body_idx"].to_numpy()

    f = np.ascontiguousarray(std["test_face"][fi], dtype=np.float32)
    b = np.ascontiguousarray(std["test_body"][bi], dtype=np.float32)

    pF, pB = extract_probs_manual(model, f, b)
    y = random_bank["y"].to_numpy().astype(int)

    scores = divergence_scores(pF, pB)

    for name, s in scores.items():
        div_rows.append({
            "experiment": "divergence_ablation",
            "K": K,
            "agreement_score": name,
            "roc_auc": safe_auc(y, s),
            "pr_auc": safe_ap(y, s),
            "pos_mean": float(np.mean(s[y == 1])),
            "neg_mean": float(np.mean(s[y == 0])),
            "score_std": float(np.std(s)),
        })

div_df = pd.DataFrame(div_rows)
div_df.to_csv(OUT_LAST / "divergence_ablation_K16_K32.csv", index=False)

display(div_df)
print("Saved:", OUT_LAST / "divergence_ablation_K16_K32.csv")

,experiment,K,agreement_score,roc_auc,pr_auc,pos_mean,neg_mean,score_std
0,divergence_ablation,16,PMCI_JSD,0.809169,0.743828,0.821985,0.586901,0.206763
1,divergence_ablation,16,Agreement_Hellinger,0.809259,0.743123,0.587943,0.326711,0.235446
2,divergence_ablation,16,Agreement_TotalVariation,0.810898,0.758701,0.529309,0.250749,0.254323
3,divergence_ablation,16,Prob_Dot,0.806286,0.748161,0.177084,0.073789,0.097284
4,divergence_ablation,16,Prob_Cosine,0.808361,0.757861,0.602630,0.253015,0.322076
5,divergence_ablation,16,Prob_Overlap,0.810898,0.758701,0.529309,0.250749,0.254323
6,divergence_ablation,32,PMCI_JSD,0.896736,0.862032,0.877487,0.576317,0.211577
7,divergence_ablation,32,Agreement_Hellinger,0.897330,0.862756,0.659694,0.311190,0.244289
8,divergence_ablation,32,Agreement_TotalVariation,0.890831,0.851589,0.601332,0.238231,0.259111
9,divergence_ablation,32,Prob_Dot,0.883519,0.845978,0.096529,0.032421,0.047039


Saved: /content/pmci_last_extra_checks/divergence_ablation_K16_K32.csv


In [ ]:
def build_mined_bank(face_np, body_np, miner_model, hard_k=20, seed=42):
    n = len(face_np)
    rows = []

    for i in range(n):
        f_rep = np.repeat(face_np[i:i+1], n, axis=0)
        s = score_pairs(miner_model, f_rep, body_np)
        s = np.asarray(s).reshape(-1)
        s[i] = -np.inf

        top = np.argsort(s)[-hard_k:][::-1]

        rows.append({
            "anchor": i,
            "face_idx": i,
            "body_idx": i,
            "y": 1,
            "pair_type": "true_pair"
        })

        for j in top:
            rows.append({
                "anchor": i,
                "face_idx": i,
                "body_idx": int(j),
                "y": 0,
                "pair_type": "mined_hard_negative"
            })

    return pd.DataFrame(rows)


miners = {
    "DirectCosine_mined": get_model("DirectCosine_K0"),
    "PMCI_K16_mined": get_model("PMCI_K16"),
    "PMCI_K32_mined": get_model("PMCI_K32"),
}

eval_models = {
    "DirectCosine_K0": get_model("DirectCosine_K0"),
    "PMCI_K16": get_model("PMCI_K16"),
    "PMCI_K32": get_model("PMCI_K32"),
}

fair_rows = []

for miner_name, miner_model in miners.items():
    bank = build_mined_bank(
        std["test_face"],
        std["test_body"],
        miner_model,
        hard_k=20,
        seed=700
    )

    y = bank["y"].to_numpy().astype(int)

    for model_name, model in eval_models.items():
        s = bank_scores(model, std["test_face"], std["test_body"], bank)
        fair_rows.append({
            "experiment": "fair_hard_negative_mining",
            "miner": miner_name,
            "evaluated_model": model_name,
            "n_pairs": len(bank),
            "positive_rate": float(np.mean(y)),
            "roc_auc": safe_auc(y, s),
            "pr_auc": safe_ap(y, s),
            "pos_mean": float(np.mean(s[y == 1])),
            "neg_mean": float(np.mean(s[y == 0])),
        })

fair_df = pd.DataFrame(fair_rows)
fair_df.to_csv(OUT_LAST / "fair_hard_negative_mining_matrix.csv", index=False)

display(fair_df)
print("Saved:", OUT_LAST / "fair_hard_negative_mining_matrix.csv")

,experiment,miner,evaluated_model,n_pairs,positive_rate,roc_auc,pr_auc,pos_mean,neg_mean
0,fair_hard_negative_mining,DirectCosine_mined,DirectCosine_K0,30240,0.047619,0.484431,0.045209,0.822540,0.829706
1,fair_hard_negative_mining,DirectCosine_mined,PMCI_K16,30240,0.047619,0.572225,0.064133,0.743179,0.695184
2,fair_hard_negative_mining,DirectCosine_mined,PMCI_K32,30240,0.047619,0.569069,0.055293,0.823251,0.776504
3,fair_hard_negative_mining,PMCI_K16_mined,DirectCosine_K0,30240,0.047619,0.878927,0.347147,0.822540,0.686461
4,fair_hard_negative_mining,PMCI_K16_mined,PMCI_K16,30240,0.047619,0.114952,0.025584,0.743179,0.928232
5,fair_hard_negative_mining,PMCI_K16_mined,PMCI_K32,30240,0.047619,0.639634,0.088993,0.823251,0.753973
6,fair_hard_negative_mining,PMCI_K32_mined,DirectCosine_K0,30240,0.047619,0.786682,0.110956,0.822540,0.709857
7,fair_hard_negative_mining,PMCI_K32_mined,PMCI_K16,30240,0.047619,0.441750,0.040925,0.743179,0.763882
8,fair_hard_negative_mining,PMCI_K32_mined,PMCI_K32,30240,0.047619,0.287263,0.030744,0.823251,0.902596


Saved: /content/pmci_last_extra_checks/fair_hard_negative_mining_matrix.csv


In [ ]:
decile_rows = []

for K in [16, 32]:
    model = get_model(f"PMCI_K{K}")
    bank = random_bank.copy()

    bank[f"pmci_K{K}"] = bank_scores(model, std["test_face"], std["test_body"], bank)
    bank["direct"] = bank_scores(get_model("DirectCosine_K0"), std["test_face"], std["test_body"], bank)

    # qcut can fail if duplicated edges; rank fixes it
    rank_score = pd.Series(bank[f"pmci_K{K}"]).rank(method="first")
    bank["pmci_decile"] = pd.qcut(rank_score, 10, labels=False) + 1

    for decile, g in bank.groupby("pmci_decile"):
        y = g["y"].to_numpy().astype(int)
        decile_rows.append({
            "experiment": "pmci_decile_diagnostic",
            "K": K,
            "pmci_decile": int(decile),
            "n": len(g),
            "pmci_min": float(g[f"pmci_K{K}"].min()),
            "pmci_max": float(g[f"pmci_K{K}"].max()),
            "pmci_mean": float(g[f"pmci_K{K}"].mean()),
            "positive_rate": float(np.mean(y)),
            "mismatch_rate": float(np.mean(y == 0)),
            "direct_mean": float(g["direct"].mean()),
        })

decile_df = pd.DataFrame(decile_rows)
decile_df.to_csv(OUT_LAST / "pmci_decile_diagnostic_K16_K32.csv", index=False)

display(decile_df)
print("Saved:", OUT_LAST / "pmci_decile_diagnostic_K16_K32.csv")

,experiment,K,pmci_decile,n,pmci_min,pmci_max,pmci_mean,positive_rate,mismatch_rate,direct_mean
0,pmci_decile_diagnostic,16,1,288,0.101085,0.137323,0.123823,0.000000,1.000000,0.253131
1,pmci_decile_diagnostic,16,2,288,0.137535,0.194713,0.163220,0.045139,0.954861,0.365531
2,pmci_decile_diagnostic,16,3,288,0.194813,0.313329,0.250012,0.298611,0.701389,0.639206
3,pmci_decile_diagnostic,16,4,288,0.315085,0.501306,0.403409,0.347222,0.652778,0.698662
4,pmci_decile_diagnostic,16,5,288,0.501553,0.683561,0.608064,0.513889,0.486111,0.758867
5,pmci_decile_diagnostic,16,6,288,0.683637,0.767136,0.726694,0.739583,0.260417,0.777028
6,pmci_decile_diagnostic,16,7,288,0.767241,0.820100,0.794824,0.822917,0.177083,0.799260
7,pmci_decile_diagnostic,16,8,288,0.820228,0.861200,0.842125,0.805556,0.194444,0.777316
8,pmci_decile_diagnostic,16,9,288,0.861251,0.909901,0.883021,0.708333,0.291667,0.770883
9,pmci_decile_diagnostic,16,10,288,0.910490,0.982796,0.940829,0.718750,0.281250,0.771799


Saved: /content/pmci_last_extra_checks/pmci_decile_diagnostic_K16_K32.csv


In [ ]:
import shutil

summary = {
    "divergence_ablation_rows": len(div_df),
    "fair_mining_rows": len(fair_df),
    "decile_rows": len(decile_df),
    "output_dir": str(OUT_LAST),
}

with open(OUT_LAST / "last_extra_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive(
    "/content/pmci_last_extra_checks",
    "zip",
    str(OUT_LAST)
)

print("DONE ✅")
print(json.dumps(summary, indent=2))
print("ZIP:", zip_path)

DONE ✅
{
  "divergence_ablation_rows": 12,
  "fair_mining_rows": 9,
  "decile_rows": 20,
  "output_dir": "/content/pmci_last_extra_checks"
}
ZIP: /content/pmci_last_extra_checks.zip


In [ ]:
# ============================================================
# FINAL
# 1) Balanced hard-negative rank test
# 2) Full pipeline YOLO latency
# 3) Optional MELD K32 diagnostic
# ============================================================

import os, time, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

OUT_OPT = Path("/content/pmci_optional_final_experiments")
OUT_OPT.mkdir(parents=True, exist_ok=True)

DEVICE_OPT = globals().get(
    "DEVICE_TORCH",
    torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

print("Output:", OUT_OPT)
print("Device:", DEVICE_OPT)

required = ["std", "trained_models", "score_pairs"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}. Run the main PMCI notebook first.")

def unwrap_model(obj):
    if isinstance(obj, dict):
        return obj.get("model", obj.get("net", obj))
    return obj

def get_model(name):
    if name not in trained_models:
        raise RuntimeError(f"{name} not found. Available: {list(trained_models.keys())}")
    m = unwrap_model(trained_models[name])
    m.to(DEVICE_OPT)
    m.eval()
    return m

def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def metrics_from_score(y, s, threshold=0.5):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    pred = (s >= threshold).astype(int)
    return {
        "roc_auc": safe_auc(y, s),
        "pr_auc": safe_ap(y, s),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
        "pos_mean": float(np.mean(s[y == 1])) if np.any(y == 1) else np.nan,
        "neg_mean": float(np.mean(s[y == 0])) if np.any(y == 0) else np.nan,
    }

def bank_scores(model, face_np, body_np, bank_df, batch_size=512):
    fi = bank_df["face_idx"].to_numpy()
    bi = bank_df["body_idx"].to_numpy()
    f = np.ascontiguousarray(face_np[fi], dtype=np.float32)
    b = np.ascontiguousarray(body_np[bi], dtype=np.float32)
    return np.asarray(score_pairs(model, f, b, batch_size=batch_size)).reshape(-1)

DIRECT = get_model("DirectCosine_K0")
PMCI16 = get_model("PMCI_K16")
PMCI32 = get_model("PMCI_K32")

print("Setup ready.")

Output: /content/pmci_optional_final_experiments
Device: cuda
Setup ready.


In [ ]:
# ============================================================
# Balanced hard-negative rank test
# ============================================================

def build_rank_balanced_bank(face_np, body_np, miner_model, ranks=(1, 5, 10, 20)):
    """
    For each rank r:
      for each anchor i:
        positive: (face_i, body_i)
        negative: body at rank r among wrong bodies according to miner_model
    Result is balanced for every rank: 50% positives, 50% negatives.
    """
    n = len(face_np)
    rows = []

    for i in range(n):
        f_rep = np.repeat(face_np[i:i+1], n, axis=0)
        s = np.asarray(score_pairs(miner_model, f_rep, body_np)).reshape(-1)
        s[i] = -np.inf
        order = np.argsort(s)[::-1]  # descending hardest first

        for r in ranks:
            idx = min(r - 1, len(order) - 1)
            j = int(order[idx])

            rows.append({
                "rank": int(r),
                "anchor": int(i),
                "face_idx": int(i),
                "body_idx": int(i),
                "y": 1,
                "pair_type": "true_pair"
            })
            rows.append({
                "rank": int(r),
                "anchor": int(i),
                "face_idx": int(i),
                "body_idx": int(j),
                "y": 0,
                "pair_type": f"rank_{r}_hard_negative"
            })

    return pd.DataFrame(rows)

RANKS = [1, 5, 10, 20]

rank_val = build_rank_balanced_bank(
    std["val_face"], std["val_body"], DIRECT, ranks=RANKS
)

rank_test = build_rank_balanced_bank(
    std["test_face"], std["test_body"], DIRECT, ranks=RANKS
)

rank_val.to_csv(OUT_OPT / "balanced_rank_hard_bank_val.csv", index=False)
rank_test.to_csv(OUT_OPT / "balanced_rank_hard_bank_test.csv", index=False)

print("Val bank:", rank_val.shape)
print("Test bank:", rank_test.shape)
display(rank_test.head())

Val bank: (15360, 6)
Test bank: (11520, 6)


,rank,anchor,face_idx,body_idx,y,pair_type
0,1,0,0,0,1,true_pair
1,1,0,0,259,0,rank_1_hard_negative
2,5,0,0,0,1,true_pair
3,5,0,0,43,0,rank_5_hard_negative
4,10,0,0,0,1,true_pair


In [ ]:
# ============================================================
# Evaluate Direct / PMCI / Direct+PMCI on balanced rank banks
# ============================================================

def add_scores_to_rank_bank(df, face_np, body_np):
    out = df.copy()
    out["direct"] = bank_scores(DIRECT, face_np, body_np, out)
    out["pmci_K16"] = bank_scores(PMCI16, face_np, body_np, out)
    out["pmci_K32"] = bank_scores(PMCI32, face_np, body_np, out)
    return out

rank_val_s = add_scores_to_rank_bank(rank_val, std["val_face"], std["val_body"])
rank_test_s = add_scores_to_rank_bank(rank_test, std["test_face"], std["test_body"])

rank_val_s.to_csv(OUT_OPT / "balanced_rank_hard_val_scores.csv", index=False)
rank_test_s.to_csv(OUT_OPT / "balanced_rank_hard_test_scores.csv", index=False)

def fit_logreg_predict(Xtr, ytr, Xte):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced")
    )
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]

rank_rows = []

for r in RANKS:
    tr = rank_val_s[rank_val_s["rank"] == r].copy()
    te = rank_test_s[rank_test_s["rank"] == r].copy()

    ytr = tr["y"].to_numpy().astype(int)
    yte = te["y"].to_numpy().astype(int)

    feature_sets = {
        "Direct_raw": te["direct"].to_numpy(),
        "PMCI_K16_raw": te["pmci_K16"].to_numpy(),
        "PMCI_K32_raw": te["pmci_K32"].to_numpy(),
    }

    # Direct + PMCI logistic gates trained on val rank bank
    prob_d_p16 = fit_logreg_predict(
        tr[["direct", "pmci_K16"]].to_numpy(), ytr,
        te[["direct", "pmci_K16"]].to_numpy()
    )
    prob_d_p32 = fit_logreg_predict(
        tr[["direct", "pmci_K32"]].to_numpy(), ytr,
        te[["direct", "pmci_K32"]].to_numpy()
    )

    feature_sets["Direct_plus_PMCI_K16"] = prob_d_p16
    feature_sets["Direct_plus_PMCI_K32"] = prob_d_p32

    for name, score in feature_sets.items():
        met = metrics_from_score(yte, score, threshold=0.5)
        rank_rows.append({
            "experiment": "balanced_hard_negative_rank_test",
            "rank": r,
            "feature_set": name,
            "n_test": len(yte),
            "positive_rate": float(np.mean(yte)),
            **met
        })

rank_result = pd.DataFrame(rank_rows)
rank_result.to_csv(OUT_OPT / "balanced_hard_negative_rank_test.csv", index=False)

display(rank_result)
print("Saved:", OUT_OPT / "balanced_hard_negative_rank_test.csv")

,experiment,rank,feature_set,n_test,positive_rate,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,mcc,pos_mean,neg_mean
0,balanced_hard_negative_rank_test,1,Direct_raw,2880,0.5,0.385592,0.429142,0.500000,0.500000,0.666667,0.000000,0.822540,0.842791
1,balanced_hard_negative_rank_test,1,PMCI_K16_raw,2880,0.5,0.554577,0.553686,0.510417,0.510417,0.637904,0.029341,0.743179,0.714399
2,balanced_hard_negative_rank_test,1,PMCI_K32_raw,2880,0.5,0.524810,0.508221,0.510417,0.510417,0.663484,0.050175,0.823251,0.804541
3,balanced_hard_negative_rank_test,1,Direct_plus_PMCI_K16,2880,0.5,0.621203,0.666778,0.594097,0.594097,0.465966,0.214505,0.417144,0.309458
4,balanced_hard_negative_rank_test,1,Direct_plus_PMCI_K32,2880,0.5,0.624144,0.663149,0.593403,0.593403,0.473708,0.209761,0.443997,0.343834
5,balanced_hard_negative_rank_test,5,Direct_raw,2880,0.5,0.446412,0.461988,0.500000,0.500000,0.666667,0.000000,0.822540,0.835155
6,balanced_hard_negative_rank_test,5,PMCI_K16_raw,2880,0.5,0.560923,0.559709,0.526042,0.526042,0.645362,0.070410,0.743179,0.700887
7,balanced_hard_negative_rank_test,5,PMCI_K32_raw,2880,0.5,0.541671,0.516611,0.527778,0.527778,0.671498,0.114755,0.823251,0.785885
8,balanced_hard_negative_rank_test,5,Direct_plus_PMCI_K16,2880,0.5,0.585758,0.609575,0.556250,0.556250,0.416438,0.128172,0.406387,0.340572
9,balanced_hard_negative_rank_test,5,Direct_plus_PMCI_K32,2880,0.5,0.591919,0.612672,0.571875,0.571875,0.440816,0.162736,0.445459,0.387984


Saved: /content/pmci_optional_final_experiments/balanced_hard_negative_rank_test.csv


In [ ]:
# ============================================================
# Full pipeline latency: YOLOv8 pose extraction + PMCI model latency
# ============================================================

import glob
import cv2
import time
import numpy as np
import pandas as pd

def find_video_files():
    search_roots = [
        "/content/cias_q1_min",
        "/content",
        "/content/drive/MyDrive"
    ]
    exts = ["*.mp4", "*.avi", "*.mov", "*.mkv"]
    files = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for ext in exts:
            files.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    # remove duplicates
    files = sorted(list(dict.fromkeys(files)))
    return files

video_files = find_video_files()
print("Found videos:", len(video_files))
print(video_files[:5])

if len(video_files) == 0:
    print("No videos found. Skipping YOLO full latency.")
else:
    try:
        from ultralytics import YOLO
        yolo_model = YOLO("yolov8n-pose.pt")

        if torch.cuda.is_available():
            try:
                yolo_model.to("cuda")
            except Exception:
                pass

        SAMPLE_VIDEOS = video_files[:5]
        MAX_FRAMES_PER_VIDEO = 48

        yolo_times = []
        frame_counts = []

        # warm-up
        dummy = np.zeros((480, 640, 3), dtype=np.uint8)
        for _ in range(3):
            _ = yolo_model.predict(dummy, verbose=False, imgsz=320)

        for vf in SAMPLE_VIDEOS:
            cap = cv2.VideoCapture(vf)
            frames = []
            count = 0

            while count < MAX_FRAMES_PER_VIDEO:
                ok, frame = cap.read()
                if not ok:
                    break
                frames.append(frame)
                count += 1

            cap.release()

            if len(frames) == 0:
                continue

            start = time.perf_counter()
            for frame in frames:
                _ = yolo_model.predict(frame, verbose=False, imgsz=320)
            end = time.perf_counter()

            total_ms = (end - start) * 1000
            yolo_times.append(total_ms / len(frames))
            frame_counts.append(len(frames))

        # PMCI model-level latency for one 12-frame window
        face_batch = np.ascontiguousarray(std["test_face"][:128], dtype=np.float32)
        body_batch = np.ascontiguousarray(std["test_body"][:128], dtype=np.float32)

        # warm-up PMCI
        for _ in range(10):
            _ = score_pairs(PMCI32, face_batch, body_batch)

        start = time.perf_counter()
        N_REPEAT = 100
        for _ in range(N_REPEAT):
            _ = score_pairs(PMCI32, face_batch, body_batch)
        end = time.perf_counter()

        pmci_ms_per_batch = (end - start) * 1000 / N_REPEAT
        pmci_ms_per_window = pmci_ms_per_batch / len(face_batch)

        yolo_ms_frame_mean = float(np.mean(yolo_times))
        yolo_ms_frame_std = float(np.std(yolo_times))
        estimated_12frame_yolo_ms = yolo_ms_frame_mean * 12
        estimated_total_ms_window = estimated_12frame_yolo_ms + pmci_ms_per_window

        latency_df = pd.DataFrame([{
            "experiment": "full_pipeline_latency_estimate",
            "device": str(DEVICE_OPT),
            "n_sample_videos": len(SAMPLE_VIDEOS),
            "total_frames_measured": int(np.sum(frame_counts)),
            "yolo_ms_per_frame_mean": yolo_ms_frame_mean,
            "yolo_ms_per_frame_std": yolo_ms_frame_std,
            "pmci_K32_ms_per_window_model_only": float(pmci_ms_per_window),
            "estimated_yolo_ms_per_12frame_window": float(estimated_12frame_yolo_ms),
            "estimated_total_ms_per_12frame_window": float(estimated_total_ms_window),
            "note": "YOLO timing includes pose model inference only; feature formatting overhead is not separately optimized."
        }])

        latency_df.to_csv(OUT_OPT / "full_pipeline_latency_yolo_pmci.csv", index=False)
        display(latency_df)
        print("Saved:", OUT_OPT / "full_pipeline_latency_yolo_pmci.csv")

    except Exception as e:
        print("YOLO latency failed:", repr(e))

Found videos: 2880
['/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-01-01-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-01-02-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-02-01-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-02-02-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-02-01-01-01-01.mp4']


,experiment,device,n_sample_videos,total_frames_measured,yolo_ms_per_frame_mean,yolo_ms_per_frame_std,pmci_K32_ms_per_window_model_only,estimated_yolo_ms_per_12frame_window,estimated_total_ms_per_12frame_window,note
0,full_pipeline_latency_estimate,cuda,5,240,10.397314,0.200518,0.018298,124.767772,124.78607,YOLO timing includes pose model inference only...


Saved: /content/pmci_optional_final_experiments/full_pipeline_latency_yolo_pmci.csv


In [ ]:
# ============================================================
# Full pipeline latency: YOLOv8 pose extraction + PMCI model latency
# ============================================================

import glob
import cv2
import time
import numpy as np
import pandas as pd

def find_video_files():
    search_roots = [
        "/content/cias_q1_min",
        "/content",
        "/content/drive/MyDrive"
    ]
    exts = ["*.mp4", "*.avi", "*.mov", "*.mkv"]
    files = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for ext in exts:
            files.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    # remove duplicates
    files = sorted(list(dict.fromkeys(files)))
    return files

video_files = find_video_files()
print("Found videos:", len(video_files))
print(video_files[:5])

if len(video_files) == 0:
    print("No videos found. Skipping YOLO full latency.")
else:
    try:
        from ultralytics import YOLO
        yolo_model = YOLO("yolov8n-pose.pt")

        if torch.cuda.is_available():
            try:
                yolo_model.to("cuda")
            except Exception:
                pass

        SAMPLE_VIDEOS = video_files[:5]
        MAX_FRAMES_PER_VIDEO = 48

        yolo_times = []
        frame_counts = []

        # warm-up
        dummy = np.zeros((480, 640, 3), dtype=np.uint8)
        for _ in range(3):
            _ = yolo_model.predict(dummy, verbose=False, imgsz=320)

        for vf in SAMPLE_VIDEOS:
            cap = cv2.VideoCapture(vf)
            frames = []
            count = 0

            while count < MAX_FRAMES_PER_VIDEO:
                ok, frame = cap.read()
                if not ok:
                    break
                frames.append(frame)
                count += 1

            cap.release()

            if len(frames) == 0:
                continue

            start = time.perf_counter()
            for frame in frames:
                _ = yolo_model.predict(frame, verbose=False, imgsz=320)
            end = time.perf_counter()

            total_ms = (end - start) * 1000
            yolo_times.append(total_ms / len(frames))
            frame_counts.append(len(frames))

        # PMCI model-level latency for one 12-frame window
        face_batch = np.ascontiguousarray(std["test_face"][:128], dtype=np.float32)
        body_batch = np.ascontiguousarray(std["test_body"][:128], dtype=np.float32)

        # warm-up PMCI
        for _ in range(10):
            _ = score_pairs(PMCI32, face_batch, body_batch)

        start = time.perf_counter()
        N_REPEAT = 100
        for _ in range(N_REPEAT):
            _ = score_pairs(PMCI32, face_batch, body_batch)
        end = time.perf_counter()

        pmci_ms_per_batch = (end - start) * 1000 / N_REPEAT
        pmci_ms_per_window = pmci_ms_per_batch / len(face_batch)

        yolo_ms_frame_mean = float(np.mean(yolo_times))
        yolo_ms_frame_std = float(np.std(yolo_times))
        estimated_12frame_yolo_ms = yolo_ms_frame_mean * 12
        estimated_total_ms_window = estimated_12frame_yolo_ms + pmci_ms_per_window

        latency_df = pd.DataFrame([{
            "experiment": "full_pipeline_latency_estimate",
            "device": str(DEVICE_OPT),
            "n_sample_videos": len(SAMPLE_VIDEOS),
            "total_frames_measured": int(np.sum(frame_counts)),
            "yolo_ms_per_frame_mean": yolo_ms_frame_mean,
            "yolo_ms_per_frame_std": yolo_ms_frame_std,
            "pmci_K32_ms_per_window_model_only": float(pmci_ms_per_window),
            "estimated_yolo_ms_per_12frame_window": float(estimated_12frame_yolo_ms),
            "estimated_total_ms_per_12frame_window": float(estimated_total_ms_window),
            "note": "YOLO timing includes pose model inference only; feature formatting overhead is not separately optimized."
        }])

        latency_df.to_csv(OUT_OPT / "full_pipeline_latency_yolo_pmci.csv", index=False)
        display(latency_df)
        print("Saved:", OUT_OPT / "full_pipeline_latency_yolo_pmci.csv")

    except Exception as e:
        print("YOLO latency failed:", repr(e))

Found videos: 2880
['/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-01-01-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-01-02-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-02-01-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-01-01-02-02-01.mp4', '/content/cias_q1_min/raw/RAVDESS/Actor_01/01-01-02-01-01-01-01.mp4']


,experiment,device,n_sample_videos,total_frames_measured,yolo_ms_per_frame_mean,yolo_ms_per_frame_std,pmci_K32_ms_per_window_model_only,estimated_yolo_ms_per_12frame_window,estimated_total_ms_per_12frame_window,note
0,full_pipeline_latency_estimate,cuda,5,240,10.499858,0.238089,0.018746,125.998298,126.017045,YOLO timing includes pose model inference only...


Saved: /content/pmci_optional_final_experiments/full_pipeline_latency_yolo_pmci.csv


In [ ]:
# ============================================================
# Optional MELD K32 diagnostic if MELD standardized arrays exist
# ============================================================

def find_meld_like_dataset():
    """
    Search globals for dict-like object containing test_face/test_body and name with 'meld'.
    """
    candidates = []
    for name, obj in globals().items():
        lname = name.lower()
        if "meld" not in lname:
            continue
        if isinstance(obj, dict) and "test_face" in obj and "test_body" in obj:
            candidates.append((name, obj))
    return candidates

meld_candidates = find_meld_like_dataset()
print("MELD candidates:", [x[0] for x in meld_candidates])

def build_random_bank_for_n(n, seed=123):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        rows.append({
            "anchor": i,
            "face_idx": i,
            "body_idx": i,
            "y": 1,
            "pair_type": "true_pair"
        })
        j = int(rng.integers(0, n))
        while j == i:
            j = int(rng.integers(0, n))
        rows.append({
            "anchor": i,
            "face_idx": i,
            "body_idx": j,
            "y": 0,
            "pair_type": "random_mismatch"
        })
    return pd.DataFrame(rows)

meld_rows = []

if not meld_candidates:
    print("No MELD standardized dataset found in runtime. Skipping MELD optional diagnostic.")
else:
    for ds_name, ds in meld_candidates:
        face_np = ds["test_face"]
        body_np = ds["test_body"]

        if face_np.shape[-1] != std["test_face"].shape[-1] or body_np.shape[-1] != std["test_body"].shape[-1]:
            print(f"Skipping {ds_name}: feature dims do not match.")
            continue

        bank = build_random_bank_for_n(len(face_np), seed=321)
        bank["pmci_K32"] = bank_scores(PMCI32, face_np, body_np, bank)
        bank["direct"] = bank_scores(DIRECT, face_np, body_np, bank)

        y = bank["y"].to_numpy().astype(int)

        meld_rows.append({
            "experiment": "MELD_optional_random_bank_K32",
            "dataset_object": ds_name,
            "n_pairs": len(bank),
            "positive_rate": float(np.mean(y)),
            "direct_auc": safe_auc(y, bank["direct"]),
            "direct_pr_auc": safe_ap(y, bank["direct"]),
            "pmci_K32_auc": safe_auc(y, bank["pmci_K32"]),
            "pmci_K32_pr_auc": safe_ap(y, bank["pmci_K32"]),
            "pmci_pos_mean": float(bank.loc[bank["y"] == 1, "pmci_K32"].mean()),
            "pmci_neg_mean": float(bank.loc[bank["y"] == 0, "pmci_K32"].mean()),
        })

        # decile diagnostic
        rank_score = pd.Series(bank["pmci_K32"]).rank(method="first")
        bank["pmci_decile"] = pd.qcut(rank_score, 10, labels=False) + 1

        dec_rows = []
        for dec, g in bank.groupby("pmci_decile"):
            yy = g["y"].to_numpy().astype(int)
            dec_rows.append({
                "dataset_object": ds_name,
                "decile": int(dec),
                "n": len(g),
                "pmci_mean": float(g["pmci_K32"].mean()),
                "positive_rate": float(np.mean(yy)),
                "mismatch_rate": float(np.mean(yy == 0)),
            })

        bank.to_csv(OUT_OPT / f"MELD_optional_bank_{ds_name}.csv", index=False)
        pd.DataFrame(dec_rows).to_csv(OUT_OPT / f"MELD_optional_deciles_{ds_name}.csv", index=False)

meld_summary = pd.DataFrame(meld_rows)
meld_summary.to_csv(OUT_OPT / "MELD_optional_K32_summary.csv", index=False)

display(meld_summary)
print("Saved:", OUT_OPT / "MELD_optional_K32_summary.csv")

MELD candidates: []
No MELD standardized dataset found in runtime. Skipping MELD optional diagnostic.


""


Saved: /content/pmci_optional_final_experiments/MELD_optional_K32_summary.csv


In [ ]:
# ============================================================
# Zip optional final experiments
# ============================================================

import shutil, json

summary = {
    "output_dir": str(OUT_OPT),
    "files": sorted([p.name for p in OUT_OPT.glob("*")])
}

with open(OUT_OPT / "optional_final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive(
    "/content/pmci_optional_final_experiments",
    "zip",
    str(OUT_OPT)
)

print("DONE ✅")
print("Optional final experiments:", OUT_OPT)
print("ZIP:", zip_path)
print(json.dumps(summary, indent=2))

DONE ✅
Optional final experiments: /content/pmci_optional_final_experiments
ZIP: /content/pmci_optional_final_experiments.zip
{
  "output_dir": "/content/pmci_optional_final_experiments",
  "files": [
    "MELD_optional_K32_summary.csv",
    "balanced_hard_negative_rank_test.csv",
    "balanced_rank_hard_bank_test.csv",
    "balanced_rank_hard_bank_val.csv",
    "balanced_rank_hard_test_scores.csv",
    "balanced_rank_hard_val_scores.csv",
    "full_pipeline_latency_yolo_pmci.csv"
  ]
}


In [ ]:
import os, json, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

OUT_STAT = Path("/content/pmci_final_statistical_robustness")
OUT_STAT.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [
    Path("/content/pmci_revision_results_final_extra"),
    Path("/content/pmci_final_extra"),
    Path("/content/pmci_optional_final_experiments"),
    Path("/content/pmci_last_extra_checks"),
    Path("/content/pmci_revision_results_v6"),
    Path("/content"),
]

def find_csv(name):
    for d in SEARCH_DIRS:
        p = d / name
        if p.exists():
            return p
    matches = list(Path("/content").glob(f"**/{name}"))
    if matches:
        return matches[0]
    return None

def read_csv_required(name):
    p = find_csv(name)
    if p is None:
        raise FileNotFoundError(f"Cannot find {name}")
    print("Loaded:", p)
    return pd.read_csv(p)

def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s)

def safe_ap(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return average_precision_score(y, s)

def fit_logreg_prob(Xtr, ytr, Xte):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced")
    )
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]

print("Output:", OUT_STAT)

Output: /content/pmci_final_statistical_robustness


In [ ]:
def grouped_bootstrap_metric(y, score, groups, metric_fn, B=2000, seed=123):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score = np.asarray(score)
    groups = np.asarray(groups)
    ug = np.unique(groups)

    vals = []
    for _ in range(B):
        sampled = rng.choice(ug, size=len(ug), replace=True)
        idx = np.concatenate([np.where(groups == g)[0] for g in sampled])
        yy = y[idx]
        ss = score[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(metric_fn(yy, ss))

    vals = np.asarray(vals)
    return {
        "boot_mean": float(np.mean(vals)),
        "boot_lo": float(np.quantile(vals, 0.025)),
        "boot_hi": float(np.quantile(vals, 0.975)),
        "boot_n": int(len(vals)),
    }

def grouped_bootstrap_delta_auc(y, score_a, score_b, groups, B=2000, seed=123):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score_a = np.asarray(score_a)
    score_b = np.asarray(score_b)
    groups = np.asarray(groups)
    ug = np.unique(groups)

    vals = []
    for _ in range(B):
        sampled = rng.choice(ug, size=len(ug), replace=True)
        idx = np.concatenate([np.where(groups == g)[0] for g in sampled])
        yy = y[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, score_a[idx]) - roc_auc_score(yy, score_b[idx]))

    vals = np.asarray(vals)
    return {
        "delta_auc_mean": float(np.mean(vals)),
        "delta_auc_lo": float(np.quantile(vals, 0.025)),
        "delta_auc_hi": float(np.quantile(vals, 0.975)),
        "delta_auc_boot_n": int(len(vals)),
    }

def evaluate_with_ci(df, score_col, label_col="y", group_col="anchor", B=2000, seed=123):
    y = df[label_col].to_numpy().astype(int)
    s = df[score_col].to_numpy()
    g = df[group_col].to_numpy()

    auc_ci = grouped_bootstrap_metric(y, s, g, roc_auc_score, B=B, seed=seed)
    ap_ci = grouped_bootstrap_metric(y, s, g, average_precision_score, B=B, seed=seed+1)

    return {
        "score_col": score_col,
        "roc_auc": safe_auc(y, s),
        "roc_auc_boot_mean": auc_ci["boot_mean"],
        "roc_auc_lo": auc_ci["boot_lo"],
        "roc_auc_hi": auc_ci["boot_hi"],
        "pr_auc": safe_ap(y, s),
        "pr_auc_boot_mean": ap_ci["boot_mean"],
        "pr_auc_lo": ap_ci["boot_lo"],
        "pr_auc_hi": ap_ci["boot_hi"],
        "boot_n": auc_ci["boot_n"],
    }

print("Bootstrap helpers ready.")

Bootstrap helpers ready.


In [ ]:
hard_val = read_csv_required("hard_val_scores_K16_K32.csv")
hard_test = read_csv_required("hard_test_scores_K16_K32.csv")

B_MAIN = 2000
N_LABEL_PERM = 500

hard_rows = []
perm_rows = []

for K in [16, 32]:
    ytr = hard_val["y"].to_numpy().astype(int)
    yte = hard_test["y"].to_numpy().astype(int)
    gte = hard_test["anchor"].to_numpy()

    dtr = hard_val[["direct"]].to_numpy()
    dte = hard_test[["direct"]].to_numpy()

    ptr = hard_val[[f"pmci_K{K}"]].to_numpy()
    pte = hard_test[[f"pmci_K{K}"]].to_numpy()

    Xtr = np.column_stack([dtr, ptr])
    Xte = np.column_stack([dte, pte])

    prob_real = fit_logreg_prob(Xtr, ytr, Xte)

    tmp = hard_test.copy()
    tmp[f"direct_plus_pmci_K{K}_prob"] = prob_real

    for col in ["direct", f"pmci_K{K}", f"direct_plus_pmci_K{K}_prob"]:
        row = evaluate_with_ci(tmp, col, B=B_MAIN, seed=1000+K)
        row.update({
            "experiment": "hard_negative_grouped_bootstrap_CI",
            "K": K,
            "n_test": len(tmp),
            "positive_rate": float(tmp["y"].mean()),
        })
        hard_rows.append(row)

    # ΔAUC: PMCI vs Direct, Direct+PMCI vs Direct
    delta_pmci = grouped_bootstrap_delta_auc(
        yte, tmp[f"pmci_K{K}"].to_numpy(), tmp["direct"].to_numpy(), gte,
        B=B_MAIN, seed=2000+K
    )
    delta_fused = grouped_bootstrap_delta_auc(
        yte, tmp[f"direct_plus_pmci_K{K}_prob"].to_numpy(), tmp["direct"].to_numpy(), gte,
        B=B_MAIN, seed=3000+K
    )

    hard_rows.append({
        "experiment": "hard_negative_delta_auc_CI",
        "K": K,
        "score_col": f"pmci_K{K}_minus_direct",
        **delta_pmci
    })
    hard_rows.append({
        "experiment": "hard_negative_delta_auc_CI",
        "K": K,
        "score_col": f"direct_plus_pmci_K{K}_minus_direct",
        **delta_fused
    })

    # Label permutation control: train with permuted labels, evaluate against true test labels
    rng = np.random.default_rng(5000 + K)
    aucs = []
    for i in range(N_LABEL_PERM):
        ytr_perm = rng.permutation(ytr)
        try:
            prob_perm = fit_logreg_prob(Xtr, ytr_perm, Xte)
            aucs.append(safe_auc(yte, prob_perm))
        except Exception:
            pass

    aucs = np.asarray(aucs)
    perm_rows.append({
        "experiment": "label_permutation_control",
        "K": K,
        "n_permutations": int(len(aucs)),
        "real_auc": safe_auc(yte, prob_real),
        "perm_auc_mean": float(np.mean(aucs)),
        "perm_auc_std": float(np.std(aucs)),
        "perm_auc_lo": float(np.quantile(aucs, 0.025)),
        "perm_auc_hi": float(np.quantile(aucs, 0.975)),
        "permutation_p_real_greater": float((1 + np.sum(aucs >= safe_auc(yte, prob_real))) / (len(aucs) + 1))
    })

hard_ci_df = pd.DataFrame(hard_rows)
perm_df = pd.DataFrame(perm_rows)

hard_ci_df.to_csv(OUT_STAT / "hard_negative_grouped_bootstrap_CI_K16_K32.csv", index=False)
perm_df.to_csv(OUT_STAT / "label_permutation_control_K16_K32.csv", index=False)

display(hard_ci_df)
display(perm_df)

Loaded: /content/pmci_revision_results_final_extra/hard_val_scores_K16_K32.csv
Loaded: /content/pmci_revision_results_final_extra/hard_test_scores_K16_K32.csv


,score_col,roc_auc,roc_auc_boot_mean,roc_auc_lo,roc_auc_hi,pr_auc,pr_auc_boot_mean,pr_auc_lo,pr_auc_hi,boot_n,experiment,K,n_test,positive_rate,delta_auc_mean,delta_auc_lo,delta_auc_hi,delta_auc_boot_n
0,direct,0.484431,0.484289,0.476973,0.491756,0.045209,0.045387,0.044571,0.046458,2000.0,hard_negative_grouped_bootstrap_CI,16,30240.0,0.047619,NaN,NaN,NaN,NaN
1,pmci_K16,0.572225,0.572264,0.560532,0.584397,0.064133,0.064758,0.060901,0.069034,2000.0,hard_negative_grouped_bootstrap_CI,16,30240.0,0.047619,NaN,NaN,NaN,NaN
2,direct_plus_pmci_K16_prob,0.572802,0.572959,0.563229,0.583100,0.083430,0.084801,0.074059,0.097085,2000.0,hard_negative_grouped_bootstrap_CI,16,30240.0,0.047619,NaN,NaN,NaN,NaN
3,pmci_K16_minus_direct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hard_negative_delta_auc_CI,16,NaN,NaN,0.087767,0.073998,0.100972,2000.0
4,direct_plus_pmci_K16_minus_direct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hard_negative_delta_auc_CI,16,NaN,NaN,0.088441,0.074012,0.103262,2000.0
5,direct,0.484431,0.484484,0.476897,0.491544,0.045209,0.045388,0.044561,0.046379,2000.0,hard_negative_grouped_bootstrap_CI,32,30240.0,0.047619,NaN,NaN,NaN,NaN
6,pmci_K32,0.569069,0.568902,0.558323,0.579943,0.055293,0.055788,0.053485,0.058593,2000.0,hard_negative_grouped_bootstrap_CI,32,30240.0,0.047619,NaN,NaN,NaN,NaN
7,direct_plus_pmci_K32_prob,0.582499,0.582316,0.572648,0.591899,0.082229,0.083225,0.074250,0.093575,2000.0,hard_negative_grouped_bootstrap_CI,32,30240.0,0.047619,NaN,NaN,NaN,NaN
8,pmci_K32_minus_direct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hard_negative_delta_auc_CI,32,NaN,NaN,0.084532,0.071359,0.097560,2000.0
9,direct_plus_pmci_K32_minus_direct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hard_negative_delta_auc_CI,32,NaN,NaN,0.098068,0.082896,0.112961,2000.0


,experiment,K,n_permutations,real_auc,perm_auc_mean,perm_auc_std,perm_auc_lo,perm_auc_hi,permutation_p_real_greater
0,label_permutation_control,16,500,0.572802,0.500896,0.053395,0.420033,0.580502,0.129741
1,label_permutation_control,32,500,0.582499,0.500445,0.058668,0.401107,0.599218,0.103792


In [ ]:
def anchor_subsample_auc(df, score_col, fractions=(0.5, 0.7, 0.9), R=1000, seed=123):
    rng = np.random.default_rng(seed)
    anchors = np.unique(df["anchor"].to_numpy())
    rows = []

    for frac in fractions:
        m = max(2, int(round(len(anchors) * frac)))
        vals = []
        for _ in range(R):
            chosen = rng.choice(anchors, size=m, replace=False)
            sub = df[df["anchor"].isin(chosen)]
            y = sub["y"].to_numpy().astype(int)
            s = sub[score_col].to_numpy()
            if len(np.unique(y)) < 2:
                continue
            vals.append(roc_auc_score(y, s))

        vals = np.asarray(vals)
        rows.append({
            "score_col": score_col,
            "anchor_fraction": frac,
            "n_repeats": int(len(vals)),
            "auc_mean": float(np.mean(vals)),
            "auc_std": float(np.std(vals)),
            "auc_lo": float(np.quantile(vals, 0.025)),
            "auc_hi": float(np.quantile(vals, 0.975)),
        })

    return pd.DataFrame(rows)

sample_rows = []

# Add fused probs again
hard_aug = hard_test.copy()

for K in [16, 32]:
    ytr = hard_val["y"].to_numpy().astype(int)
    Xtr = hard_val[["direct", f"pmci_K{K}"]].to_numpy()
    Xte = hard_test[["direct", f"pmci_K{K}"]].to_numpy()
    hard_aug[f"direct_plus_pmci_K{K}_prob"] = fit_logreg_prob(Xtr, ytr, Xte)

for col in ["direct", "pmci_K16", "pmci_K32", "direct_plus_pmci_K16_prob", "direct_plus_pmci_K32_prob"]:
    sdf = anchor_subsample_auc(hard_aug, col, fractions=(0.5, 0.7, 0.9), R=1000, seed=7000)
    sdf["experiment"] = "anchor_subsample_sensitivity"
    sample_rows.append(sdf)

sample_df = pd.concat(sample_rows, ignore_index=True)
sample_df.to_csv(OUT_STAT / "anchor_subsample_sensitivity_hard_bank.csv", index=False)

display(sample_df)

,score_col,anchor_fraction,n_repeats,auc_mean,auc_std,auc_lo,auc_hi,experiment
0,direct,0.5,1000,0.484223,0.003657,0.477022,0.491405,anchor_subsample_sensitivity
1,direct,0.7,1000,0.484392,0.002369,0.479829,0.489160,anchor_subsample_sensitivity
2,direct,0.9,1000,0.484436,0.001276,0.481970,0.486964,anchor_subsample_sensitivity
3,pmci_K16,0.5,1000,0.572316,0.006018,0.560237,0.584520,anchor_subsample_sensitivity
4,pmci_K16,0.7,1000,0.572156,0.003965,0.564579,0.579912,anchor_subsample_sensitivity
5,pmci_K16,0.9,1000,0.572118,0.001949,0.568223,0.575984,anchor_subsample_sensitivity
6,pmci_K32,0.5,1000,0.568954,0.005479,0.558512,0.579578,anchor_subsample_sensitivity
7,pmci_K32,0.7,1000,0.569010,0.003568,0.562477,0.576276,anchor_subsample_sensitivity
8,pmci_K32,0.9,1000,0.569113,0.001796,0.565602,0.572669,anchor_subsample_sensitivity
9,direct_plus_pmci_K16_prob,0.5,1000,0.572971,0.005121,0.562290,0.582594,anchor_subsample_sensitivity


In [ ]:
random_bank = read_csv_required("random_test_scores_K16_K32.csv")

def decile_bootstrap_table(df, K, B=2000, seed=123):
    rng = np.random.default_rng(seed)
    work = df.copy()

    rank_score = pd.Series(work[f"pmci_K{K}"]).rank(method="first")
    work["pmci_decile"] = pd.qcut(rank_score, 10, labels=False) + 1

    rows = []
    for decile, g in work.groupby("pmci_decile"):
        y = g["y"].to_numpy().astype(int)
        mismatch = (y == 0).astype(float)

        vals = []
        for _ in range(B):
            idx = rng.integers(0, len(g), size=len(g))
            vals.append(float(np.mean(mismatch[idx])))

        vals = np.asarray(vals)
        rows.append({
            "experiment": "pmci_decile_mismatch_rate_CI",
            "K": K,
            "pmci_decile": int(decile),
            "n": len(g),
            "pmci_min": float(g[f"pmci_K{K}"].min()),
            "pmci_max": float(g[f"pmci_K{K}"].max()),
            "pmci_mean": float(g[f"pmci_K{K}"].mean()),
            "positive_rate": float(np.mean(y)),
            "mismatch_rate": float(np.mean(mismatch)),
            "mismatch_rate_lo": float(np.quantile(vals, 0.025)),
            "mismatch_rate_hi": float(np.quantile(vals, 0.975)),
        })

    return pd.DataFrame(rows)

def monotonic_permutation_test(df, K, R=5000, seed=123):
    rng = np.random.default_rng(seed)
    work = df.copy()

    rank_score = pd.Series(work[f"pmci_K{K}"]).rank(method="first")
    work["pmci_decile"] = pd.qcut(rank_score, 10, labels=False) + 1

    observed_rates = work.groupby("pmci_decile")["y"].mean().to_numpy()
    deciles = np.arange(1, 11)

    observed_rho = pd.Series(deciles).corr(pd.Series(observed_rates), method="spearman")

    y = work["y"].to_numpy().copy()
    null_rhos = []

    for _ in range(R):
        yp = rng.permutation(y)
        tmp = work.copy()
        tmp["yp"] = yp
        rates = tmp.groupby("pmci_decile")["yp"].mean().to_numpy()
        rho = pd.Series(deciles).corr(pd.Series(rates), method="spearman")
        null_rhos.append(rho)

    null_rhos = np.asarray(null_rhos)
    p = (1 + np.sum(null_rhos >= observed_rho)) / (len(null_rhos) + 1)

    return {
        "experiment": "pmci_decile_monotonic_permutation",
        "K": K,
        "observed_spearman_rho": float(observed_rho),
        "null_rho_mean": float(np.mean(null_rhos)),
        "null_rho_std": float(np.std(null_rhos)),
        "permutation_p_positive_monotonic": float(p),
        "n_permutations": int(len(null_rhos)),
    }

decile_tables = []
trend_rows = []

for K in [16, 32]:
    decile_tables.append(decile_bootstrap_table(random_bank, K, B=2000, seed=100+K))
    trend_rows.append(monotonic_permutation_test(random_bank, K, R=5000, seed=200+K))

decile_ci_df = pd.concat(decile_tables, ignore_index=True)
trend_df = pd.DataFrame(trend_rows)

decile_ci_df.to_csv(OUT_STAT / "pmci_decile_mismatch_rate_CI_K16_K32.csv", index=False)
trend_df.to_csv(OUT_STAT / "pmci_decile_monotonic_permutation_K16_K32.csv", index=False)

display(decile_ci_df)
display(trend_df)

Loaded: /content/pmci_revision_results_final_extra/random_test_scores_K16_K32.csv


,experiment,K,pmci_decile,n,pmci_min,pmci_max,pmci_mean,positive_rate,mismatch_rate,mismatch_rate_lo,mismatch_rate_hi
0,pmci_decile_mismatch_rate_CI,16,1,288,0.099417,0.137307,0.124151,0.000000,1.000000,1.000000,1.000000
1,pmci_decile_mismatch_rate_CI,16,2,288,0.137354,0.197733,0.162643,0.048611,0.951389,0.923611,0.975694
2,pmci_decile_mismatch_rate_CI,16,3,288,0.197855,0.321631,0.253485,0.315972,0.684028,0.628472,0.739583
3,pmci_decile_mismatch_rate_CI,16,4,288,0.321755,0.525172,0.422340,0.368056,0.631944,0.579774,0.684028
4,pmci_decile_mismatch_rate_CI,16,5,288,0.525189,0.688609,0.618419,0.503472,0.496528,0.437500,0.555556
5,pmci_decile_mismatch_rate_CI,16,6,288,0.688969,0.768386,0.728843,0.729167,0.270833,0.222135,0.322917
6,pmci_decile_mismatch_rate_CI,16,7,288,0.768775,0.821786,0.796820,0.829861,0.170139,0.131944,0.215278
7,pmci_decile_mismatch_rate_CI,16,8,288,0.821858,0.863245,0.843626,0.812500,0.187500,0.145833,0.232639
8,pmci_decile_mismatch_rate_CI,16,9,288,0.863425,0.912292,0.884746,0.687500,0.312500,0.260417,0.368056
9,pmci_decile_mismatch_rate_CI,16,10,288,0.912433,0.982796,0.942715,0.704861,0.295139,0.243056,0.347222


,experiment,K,observed_spearman_rho,null_rho_mean,null_rho_std,permutation_p_positive_monotonic,n_permutations
0,pmci_decile_monotonic_permutation,16,0.806061,-0.002046,0.332824,0.003799,5000
1,pmci_decile_monotonic_permutation,32,0.984807,0.008603,0.337290,0.000200,5000


In [ ]:
rank_scores = read_csv_required("balanced_rank_hard_test_scores.csv")

rank_rows = []

for rank in sorted(rank_scores["rank"].unique()):
    sub = rank_scores[rank_scores["rank"] == rank].copy()

    for col in ["direct", "pmci_K16", "pmci_K32"]:
        row = evaluate_with_ci(sub, col, B=2000, seed=9000+int(rank))
        row.update({
            "experiment": "balanced_rank_grouped_CI",
            "rank": int(rank),
            "n_test": len(sub),
            "positive_rate": float(sub["y"].mean())
        })
        rank_rows.append(row)

rank_ci_df = pd.DataFrame(rank_rows)
rank_ci_df.to_csv(OUT_STAT / "balanced_rank_grouped_CI.csv", index=False)

display(rank_ci_df)

Loaded: /content/pmci_optional_final_experiments/balanced_rank_hard_test_scores.csv


,score_col,roc_auc,roc_auc_boot_mean,roc_auc_lo,roc_auc_hi,pr_auc,pr_auc_boot_mean,pr_auc_lo,pr_auc_hi,boot_n,experiment,rank,n_test,positive_rate
0,direct,0.385592,0.385317,0.377621,0.393054,0.429142,0.429578,0.426115,0.433192,2000,balanced_rank_grouped_CI,1,2880,0.5
1,pmci_K16,0.554577,0.554576,0.540500,0.568808,0.553686,0.554465,0.539431,0.570297,2000,balanced_rank_grouped_CI,1,2880,0.5
2,pmci_K32,0.524810,0.524901,0.511844,0.538424,0.508221,0.508820,0.498682,0.519656,2000,balanced_rank_grouped_CI,1,2880,0.5
3,direct,0.446412,0.446427,0.438798,0.453447,0.461988,0.462754,0.458221,0.467489,2000,balanced_rank_grouped_CI,5,2880,0.5
4,pmci_K16,0.560923,0.560898,0.545677,0.576164,0.559709,0.560072,0.543510,0.576893,2000,balanced_rank_grouped_CI,5,2880,0.5
5,pmci_K32,0.541671,0.541575,0.527691,0.555944,0.516611,0.517542,0.506528,0.529234,2000,balanced_rank_grouped_CI,5,2880,0.5
6,direct,0.490596,0.490625,0.482750,0.498460,0.489646,0.490554,0.484679,0.496589,2000,balanced_rank_grouped_CI,10,2880,0.5
7,pmci_K16,0.575200,0.575479,0.559952,0.591559,0.569768,0.570525,0.554320,0.586887,2000,balanced_rank_grouped_CI,10,2880,0.5
8,pmci_K32,0.574089,0.574211,0.559882,0.589795,0.541868,0.542573,0.530999,0.555218,2000,balanced_rank_grouped_CI,10,2880,0.5
9,direct,0.545555,0.545795,0.537161,0.553898,0.535163,0.535880,0.528227,0.543861,2000,balanced_rank_grouped_CI,20,2880,0.5


In [ ]:
summary = {
    "output_dir": str(OUT_STAT),
    "files": sorted([p.name for p in OUT_STAT.glob("*")])
}

with open(OUT_STAT / "statistical_robustness_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive(
    "/content/pmci_final_statistical_robustness",
    "zip",
    str(OUT_STAT)
)

print("DONE ✅")
print("ZIP:", zip_path)
print(json.dumps(summary, indent=2))

DONE ✅
ZIP: /content/pmci_final_statistical_robustness.zip
{
  "output_dir": "/content/pmci_final_statistical_robustness",
  "files": [
    "anchor_subsample_sensitivity_hard_bank.csv",
    "balanced_rank_grouped_CI.csv",
    "hard_negative_grouped_bootstrap_CI_K16_K32.csv",
    "label_permutation_control_K16_K32.csv",
    "pmci_decile_mismatch_rate_CI_K16_K32.csv",
    "pmci_decile_monotonic_permutation_K16_K32.csv"
  ]
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("figures_revised")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

models = [
    "DirectCosine\n(K=0)",
    "PMCI\n(K=8)",
    "PMCI\n(K=16)",
    "PMCI\n(K=32)"
]

conditions = ["Cross-video", "Same-video", "Temporal-shift", "Shuffle"]

auc_fig2 = np.array([
    [0.930, 0.923, 0.923, 0.926],
    [0.711, 0.704, 0.704, 0.706],
    [0.742, 0.737, 0.737, 0.740],
    [0.702, 0.699, 0.699, 0.701],
])

std_fig2 = np.array([
    [0.011, 0.009, 0.009, 0.012],
    [0.047, 0.055, 0.055, 0.055],
    [0.027, 0.030, 0.030, 0.030],
    [0.009, 0.006, 0.006, 0.006],
])

x = np.arange(len(models))
width = 0.18

fig, ax = plt.subplots(figsize=(11, 7.2))

for i, condition in enumerate(conditions):
    offset = (i - 1.5) * width
    bars = ax.bar(
        x + offset,
        auc_fig2[:, i],
        width,
        yerr=std_fig2[:, i],
        capsize=4,
        label=condition,
        edgecolor="black",
        linewidth=0.6
    )

    for j, bar in enumerate(bars):
        height = bar.get_height()
        err = std_fig2[j, i]

        # Поднимаем подписи заметно выше error bar
        y_text = height + err + 0.018

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_text,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=90
        )

ax.set_title("Subject-independent RAVDESS validation", pad=18)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Model")
ax.set_xticks(x)
ax.set_xticklabels(models)

# Увеличиваем верхний предел, чтобы надписи не упирались
ax.set_ylim(0.60, 1.05)

ax.axhline(0.5, linestyle="--", linewidth=1, color="black")

# Легенду выносим НАД графиком, чтобы ничего не перекрывала
ax.legend(
    title="Mismatch condition",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.22),
    ncol=4,
    frameon=True
)

# Дополнительный верхний отступ под legend
fig.subplots_adjust(top=0.78, bottom=0.14)

fig.savefig(OUT_DIR / "figure_2_ravdess_validation_fixed.png", bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_2_ravdess_validation_fixed.pdf", bbox_inches="tight")
plt.close(fig)

print(f"Saved to: {OUT_DIR.resolve()}")

Saved to: /content/figures_revised


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("figures_revised")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.family": "Arial",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

models = [
    "DirectCosine\n(K=0)",
    "PMCI\n(K=8)",
    "PMCI\n(K=16)",
    "PMCI\n(K=32)"
]

conditions = ["Cross-video", "Same-video", "Temporal-shift", "Shuffle"]

auc = np.array([
    [0.930, 0.923, 0.923, 0.926],
    [0.711, 0.704, 0.704, 0.706],
    [0.742, 0.737, 0.737, 0.740],
    [0.702, 0.699, 0.699, 0.701],
])

std = np.array([
    [0.011, 0.009, 0.009, 0.012],
    [0.047, 0.055, 0.055, 0.055],
    [0.027, 0.030, 0.030, 0.030],
    [0.009, 0.006, 0.006, 0.006],
])

x = np.arange(len(models))
width = 0.18

fig, ax = plt.subplots(figsize=(10.5, 5.8))

for i, condition in enumerate(conditions):
    offset = (i - 1.5) * width
    bars = ax.bar(
        x + offset,
        auc[:, i],
        width,
        yerr=std[:, i],
        capsize=4,
        label=condition,
        edgecolor="black",
        linewidth=0.5
    )

    for j, bar in enumerate(bars):
        height = bar.get_height()
        err = std[j, i]
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + err + 0.012,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

ax.set_title("Subject-independent RAVDESS validation")
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Model")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.58, 1.02)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    label="Chance level"
)

# MDPI-safe: legend outside to the right, not covering data or labels
ax.legend(
    title="Mismatch condition",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True
)

fig.tight_layout(rect=[0, 0, 0.82, 1])

fig.savefig(OUT_DIR / "figure_2_ravdess_validation_mdpi.png", bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_2_ravdess_validation_mdpi.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("figures_revised")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

models = [
    "DirectCosine\n(K=0)",
    "PMCI\n(K=8)",
    "PMCI\n(K=16)",
    "PMCI\n(K=32)"
]

conditions = ["Cross-video", "Same-video", "Temporal-shift", "Shuffle"]

auc = np.array([
    [0.930, 0.923, 0.923, 0.926],
    [0.711, 0.704, 0.704, 0.706],
    [0.742, 0.737, 0.737, 0.740],
    [0.702, 0.699, 0.699, 0.701],
])

std = np.array([
    [0.011, 0.009, 0.009, 0.012],
    [0.047, 0.055, 0.055, 0.055],
    [0.027, 0.030, 0.030, 0.030],
    [0.009, 0.006, 0.006, 0.006],
])

x = np.arange(len(models))
width = 0.18

fig, ax = plt.subplots(figsize=(11.5, 5.6))

for i, condition in enumerate(conditions):
    offset = (i - 1.5) * width

    bars = ax.bar(
        x + offset,
        auc[:, i],
        width,
        yerr=std[:, i],
        capsize=4,
        label=condition,
        edgecolor="black",
        linewidth=0.5
    )

    for j, bar in enumerate(bars):
        height = bar.get_height()
        err = std[j, i]

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + err + 0.020,   # цифры выше error bar
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

chance_line = ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    label="Chance level"
)

ax.set_title("Subject-independent RAVDESS validation", pad=12)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Model")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.55, 1.05)

# Legend справа, отдельным блоком
ax.legend(
    title="Mismatch condition",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
    borderaxespad=0.0
)

# Оставляем место справа под legend
fig.subplots_adjust(right=0.78, bottom=0.16, top=0.90)

fig.savefig(OUT_DIR / "figure_2_ravdess_validation_right_legend.png", dpi=600)
fig.savefig(OUT_DIR / "figure_2_ravdess_validation_right_legend.pdf")
plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# Output folder
# =========================
OUT_DIR = Path("figures_revised")
OUT_DIR.mkdir(exist_ok=True)

# =========================
# Global style
# =========================
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def save_figure(fig, filename):
    fig.savefig(OUT_DIR / f"{filename}.png", dpi=600, bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Figure 2. Subject-independent RAVDESS validation
# ============================================================

models = [
    "DirectCosine\n(K=0)",
    "PMCI\n(K=8)",
    "PMCI\n(K=16)",
    "PMCI\n(K=32)"
]

conditions = ["Cross-video", "Same-video", "Temporal-shift", "Shuffle"]

auc_fig2 = np.array([
    [0.930, 0.923, 0.923, 0.926],
    [0.711, 0.704, 0.704, 0.706],
    [0.742, 0.737, 0.737, 0.740],
    [0.702, 0.699, 0.699, 0.701],
])

std_fig2 = np.array([
    [0.011, 0.009, 0.009, 0.012],
    [0.047, 0.055, 0.055, 0.055],
    [0.027, 0.030, 0.030, 0.030],
    [0.009, 0.006, 0.006, 0.006],
])

x = np.arange(len(models))
width = 0.18

fig, ax = plt.subplots(figsize=(10.5, 5.6))

for i, condition in enumerate(conditions):
    offset = (i - 1.5) * width
    bars = ax.bar(
        x + offset,
        auc_fig2[:, i],
        width,
        yerr=std_fig2[:, i],
        capsize=4,
        label=condition,
        edgecolor="black",
        linewidth=0.5
    )

    for j, bar in enumerate(bars):
        height = bar.get_height()
        err = std_fig2[j, i]
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + err + 0.020,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    label="Chance level"
)

ax.set_title("Subject-independent RAVDESS validation", pad=12)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Model")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.55, 1.05)

ax.legend(
    title="Mismatch condition",
    loc="upper right",
    bbox_to_anchor=(0.985, 0.965),
    ncol=2,
    frameon=True,
    framealpha=0.95,
    borderpad=0.6,
    labelspacing=0.4,
    handlelength=1.6,
    columnspacing=1.0,
    fontsize=9,
    title_fontsize=10
)

fig.subplots_adjust(left=0.10, right=0.96, bottom=0.16, top=0.90)
save_figure(fig, "figure_2_ravdess_validation")


# ============================================================
# Figure 3. Modality-disruption controls
# ============================================================

models_fig3 = [
    "MELD\nPMCI K=8",
    "RAVDESS\nDirectCosine K=0",
    "RAVDESS\nPMCI K=8",
    "RAVDESS\nPMCI K=16"
]

conditions_fig3 = ["Normal", "Face zero", "Body zero", "Body shuffled"]

auc_fig3 = np.array([
    [0.755, 0.501, 0.502, 0.508],
    [0.930, 0.504, 0.508, 0.505],
    [0.711, 0.500, 0.506, 0.511],
    [0.742, 0.510, 0.505, 0.519],
])

x = np.arange(len(models_fig3))
width = 0.18

fig, ax = plt.subplots(figsize=(10.8, 5.6))

for i, condition in enumerate(conditions_fig3):
    offset = (i - 1.5) * width
    bars = ax.bar(
        x + offset,
        auc_fig3[:, i],
        width,
        label=condition,
        edgecolor="black",
        linewidth=0.5
    )

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.018,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    label="Chance level"
)

ax.set_title("Modality-disruption controls", pad=12)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Dataset / model")
ax.set_xticks(x)
ax.set_xticklabels(models_fig3)
ax.set_ylim(0.45, 1.05)

ax.legend(
    title="Condition",
    loc="upper right",
    bbox_to_anchor=(0.985, 0.965),
    ncol=2,
    frameon=True,
    framealpha=0.95,
    borderpad=0.6,
    labelspacing=0.4,
    handlelength=1.6,
    columnspacing=1.0,
    fontsize=9,
    title_fontsize=10
)

fig.subplots_adjust(left=0.10, right=0.96, bottom=0.17, top=0.90)
save_figure(fig, "figure_3_modality_disruption_controls")


# ============================================================
# Figure 4. RAVDESS-to-MELD adaptation curve
# ============================================================

target_data = np.array([0, 10, 20, 50, 100])

cross_video = np.array([0.501, 0.578, 0.573, 0.582, 0.596])
same_video = np.array([0.536, 0.536, 0.536, 0.570, 0.554])
temporal_shift = np.array([0.539, 0.533, 0.528, 0.563, 0.544])
shuffle = np.array([0.503, 0.569, 0.577, 0.585, 0.593])

series = [
    ("Cross-video", cross_video),
    ("Same-video", same_video),
    ("Temporal-shift", temporal_shift),
    ("Shuffle", shuffle),
]

fig, ax = plt.subplots(figsize=(9.8, 5.6))

for label, values in series:
    ax.plot(
        target_data,
        values,
        marker="o",
        linewidth=2,
        markersize=5,
        label=label
    )

    for x_val, y_val in zip(target_data, values):
        ax.text(
            x_val,
            y_val + 0.007,
            f"{y_val:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    label="Chance level"
)

ax.set_title("RAVDESS-to-MELD adaptation curve", pad=12)
ax.set_xlabel("Target-domain MELD training data (%)")
ax.set_ylabel("ROC-AUC")
ax.set_xticks(target_data)

# Верхний предел специально поднят, чтобы legend стояла внутри пустой зоны
ax.set_ylim(0.48, 0.68)

ax.legend(
    title="Mismatch condition",
    loc="upper right",
    bbox_to_anchor=(0.985, 0.965),
    ncol=2,
    frameon=True,
    framealpha=0.95,
    borderpad=0.6,
    labelspacing=0.4,
    handlelength=1.8,
    columnspacing=1.0,
    fontsize=9,
    title_fontsize=10
)

fig.subplots_adjust(left=0.10, right=0.96, bottom=0.15, top=0.90)
save_figure(fig, "figure_4_ravdess_to_meld_adaptation")

print(f"Saved figures to: {OUT_DIR.resolve()}")

Saved figures to: /content/figures_revised


In [ ]:
# ============================================================
# Figure 4. RAVDESS-to-MELD adaptation curve
# Clean MDPI-style version
# ============================================================

target_data = np.array([0, 10, 20, 50, 100])

cross_video = np.array([0.501, 0.578, 0.573, 0.582, 0.596])
same_video = np.array([0.536, 0.536, 0.536, 0.570, 0.554])
temporal_shift = np.array([0.539, 0.533, 0.528, 0.563, 0.544])
shuffle = np.array([0.503, 0.569, 0.577, 0.585, 0.593])

series = [
    ("Cross-video", cross_video, "o", "-"),
    ("Same-video", same_video, "s", "--"),
    ("Temporal-shift", temporal_shift, "^", "-."),
    ("Shuffle", shuffle, "D", "-"),
]

fig, ax = plt.subplots(figsize=(8.8, 5.2))

for label, values, marker, linestyle in series:
    ax.plot(
        target_data,
        values,
        marker=marker,
        linestyle=linestyle,
        linewidth=2,
        markersize=6,
        label=label
    )

# Chance line
ax.axhline(
    0.5,
    linestyle=":",
    linewidth=1.2,
    color="black",
    label="Chance level"
)

ax.set_title("RAVDESS-to-MELD adaptation curve", pad=12)
ax.set_xlabel("Target-domain MELD training data (%)")
ax.set_ylabel("ROC-AUC")

ax.set_xticks(target_data)
ax.set_ylim(0.49, 0.625)
ax.set_yticks(np.arange(0.50, 0.626, 0.025))

# Compact legend inside the empty upper-left area
ax.legend(
    title="Mismatch condition",
    loc="upper left",
    frameon=True,
    framealpha=0.95,
    borderpad=0.6,
    labelspacing=0.4,
    handlelength=2.0,
    fontsize=9,
    title_fontsize=10
)

ax.grid(True, alpha=0.25)

fig.subplots_adjust(left=0.11, right=0.97, bottom=0.16, top=0.90)

fig.savefig(OUT_DIR / "figure_4_ravdess_to_meld_adaptation_clean.png", dpi=600, bbox_inches="tight")
fig.savefig(OUT_DIR / "figure_4_ravdess_to_meld_adaptation_clean.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
# ============================================================
# REVIEWER-ONCE CELL 1 — metadata and helper functions
# ============================================================
import re, json, math, warnings, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

OUT_REVIEWER_ONCE = Path("/content/pmci_reviewer_once_extra_checks")
OUT_REVIEWER_ONCE.mkdir(parents=True, exist_ok=True)

required_globals = ["data", "std", "train_idx", "val_idx", "test_idx", "trained_models", "score_pairs"]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}. Run the main notebook first.")

DEVICE_REVIEWER = globals().get("DEVICE_TORCH", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

def _unwrap_model(obj):
    if isinstance(obj, dict):
        return obj.get("model", obj.get("net", obj))
    return obj

def get_model_safe(name):
    if name not in trained_models:
        raise RuntimeError(f"{name} not found. Available models: {list(trained_models.keys())}")
    m = _unwrap_model(trained_models[name])
    m.to(DEVICE_REVIEWER)
    m.eval()
    return m

def score_model(model, face_np, body_np, batch_size=512):
    return np.asarray(
        score_pairs(
            model,
            np.ascontiguousarray(face_np, dtype=np.float32),
            np.ascontiguousarray(body_np, dtype=np.float32),
            batch_size=batch_size
        )
    ).reshape(-1)

def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

def safe_ap(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, s))

def fit_logreg_prob(Xtr, ytr, Xte):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, class_weight="balanced")
    )
    clf.fit(np.asarray(Xtr), np.asarray(ytr).astype(int))
    return clf.predict_proba(np.asarray(Xte))[:, 1]

def strip_window_suffix(video_id):
    s = str(video_id)
    return re.sub(r"__w\d+$", "", s)

def parse_window_id(video_id):
    s = str(video_id)
    m = re.search(r"__w(\d+)$", s)
    return int(m.group(1)) if m else 0

def make_split_meta(split_name, indices):
    ids = np.asarray(indices, dtype=int)
    video_ids = data["video_id"][ids]
    return pd.DataFrame({
        "split": split_name,
        "local_idx": np.arange(len(ids), dtype=int),
        "global_idx": ids,
        "video_id": [str(v) for v in video_ids],
        "source_video": [strip_window_suffix(v) for v in video_ids],
        "window_id": [parse_window_id(v) for v in video_ids],
        "subject_id": data["subject_id"][ids].astype(int),
        "emotion_id": data["emotion_id"][ids].astype(int),
    })

train_meta = make_split_meta("train", train_idx)
val_meta = make_split_meta("val", val_idx)
test_meta = make_split_meta("test", test_idx)

all_meta = pd.concat([train_meta, val_meta, test_meta], ignore_index=True)
all_meta.to_csv(OUT_REVIEWER_ONCE / "metadata_all_splits_source_video_windows.csv", index=False)

def split_meta_summary(meta):
    return {
        "n_windows": int(len(meta)),
        "n_source_videos": int(meta["source_video"].nunique()),
        "n_actors": int(meta["subject_id"].nunique()),
        "mean_windows_per_source_video": float(meta.groupby("source_video").size().mean()),
        "min_windows_per_source_video": int(meta.groupby("source_video").size().min()),
        "max_windows_per_source_video": int(meta.groupby("source_video").size().max()),
    }

summary = pd.DataFrame([
    {"split": "train", **split_meta_summary(train_meta)},
    {"split": "val", **split_meta_summary(val_meta)},
    {"split": "test", **split_meta_summary(test_meta)},
])
summary.to_csv(OUT_REVIEWER_ONCE / "metadata_split_summary.csv", index=False)

print("Reviewer-once output:", OUT_REVIEWER_ONCE)
display(summary)

Reviewer-once output: /content/pmci_reviewer_once_extra_checks


,split,n_windows,n_source_videos,n_actors,mean_windows_per_source_video,min_windows_per_source_video,max_windows_per_source_video
0,train,8160,2040,17,4.0,4,4
1,val,1920,480,4,4.0,4,4
2,test,1440,360,3,4.0,4,4


In [ ]:

# ============================================================
# REVIEWER-ONCE CELL 2 — one-window-per-source-video control
# FIXED FULL VERSION
# ============================================================

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1) Load trained models safely
# ------------------------------------------------------------
DIRECT = get_model_safe("DirectCosine_K0")

PMCI16 = get_model_safe("PMCI_K16") if "PMCI_K16" in trained_models else None
PMCI32 = get_model_safe("PMCI_K32") if "PMCI_K32" in trained_models else None

models_for_control = [("DirectCosine_K0", DIRECT)]

if PMCI16 is not None:
    models_for_control.append(("PMCI_K16", PMCI16))

if PMCI32 is not None:
    models_for_control.append(("PMCI_K32", PMCI32))


# ------------------------------------------------------------
# 2) Recreate standardized train/val/test arrays safely
#    Do NOT use global variable "std" here.
# ------------------------------------------------------------
def reviewer_standardize_train_only(face, body, train_idx, val_idx, test_idx):
    """
    Recreates standardized split arrays from the original data.
    This avoids using the global variable 'std', which may be overwritten
    by plotting cells.
    """
    face = np.asarray(face, dtype=np.float32)
    body = np.asarray(body, dtype=np.float32)

    train_idx = np.asarray(train_idx, dtype=np.int64)
    val_idx = np.asarray(val_idx, dtype=np.int64)
    test_idx = np.asarray(test_idx, dtype=np.int64)

    T_face, D_face = face.shape[1], face.shape[2]
    T_body, D_body = body.shape[1], body.shape[2]

    face_scaler = StandardScaler()
    body_scaler = StandardScaler()

    face_scaler.fit(face[train_idx].reshape(-1, D_face))
    body_scaler.fit(body[train_idx].reshape(-1, D_body))

    def transform_face(x):
        x = np.asarray(x, dtype=np.float32)
        return face_scaler.transform(
            x.reshape(-1, D_face)
        ).reshape(x.shape).astype(np.float32)

    def transform_body(x):
        x = np.asarray(x, dtype=np.float32)
        return body_scaler.transform(
            x.reshape(-1, D_body)
        ).reshape(x.shape).astype(np.float32)

    return {
        "train_face": transform_face(face[train_idx]),
        "train_body": transform_body(body[train_idx]),
        "val_face": transform_face(face[val_idx]),
        "val_body": transform_body(body[val_idx]),
        "test_face": transform_face(face[test_idx]),
        "test_body": transform_body(body[test_idx]),
    }


std_reviewer_once = reviewer_standardize_train_only(
    data["face"],
    data["body"],
    train_idx,
    val_idx,
    test_idx
)

TEST_FACE = std_reviewer_once["test_face"]
TEST_BODY = std_reviewer_once["test_body"]

print("Recreated standardized test arrays:")
print("TEST_FACE:", TEST_FACE.shape)
print("TEST_BODY:", TEST_BODY.shape)


# ------------------------------------------------------------
# 3) Helper functions
# ------------------------------------------------------------
def as_numpy_array(x):
    """
    Safely convert torch / numpy / list-like arrays to numpy.
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def choose_one_window_per_source(meta, seed=123):
    """
    Randomly chooses one temporal window from each original source video.
    This controls for pseudo-replication caused by multiple windows
    from the same source video.
    """
    rng = np.random.default_rng(seed)
    chosen = []

    for _, g in meta.groupby("source_video"):
        chosen.append(int(rng.choice(g["local_idx"].to_numpy())))

    return np.array(sorted(chosen), dtype=np.int64)


def build_balanced_random_bank_from_indices(
    indices,
    meta,
    n_random_per_anchor=1,
    seed=123,
    exclude_same_source=True
):
    """
    Builds a balanced true-vs-random mismatch bank using only selected windows.

    Positive pair:
        face_i with body_i

    Negative pair:
        face_i with body_j, where j comes from another selected source video.
    """
    rng = np.random.default_rng(seed)
    indices = np.asarray(indices, dtype=np.int64)

    meta_local = meta.set_index("local_idx")
    rows = []

    for i in indices:
        i = int(i)
        src_i = meta_local.loc[i, "source_video"]

        # Positive pair
        rows.append({
            "anchor": i,
            "face_idx": i,
            "body_idx": i,
            "y": 1,
            "pair_type": "true_pair"
        })

        # Candidate negative bodies
        candidates = indices[indices != i]

        if exclude_same_source:
            candidates = np.array([
                int(j) for j in candidates
                if meta_local.loc[int(j), "source_video"] != src_i
            ], dtype=np.int64)

        if len(candidates) == 0:
            continue

        for _ in range(n_random_per_anchor):
            j = int(rng.choice(candidates))

            rows.append({
                "anchor": i,
                "face_idx": i,
                "body_idx": j,
                "y": 0,
                "pair_type": "random_mismatch"
            })

    return pd.DataFrame(rows)


def score_bank_on_split(model, face_np, body_np, bank):
    """
    Scores all pairs from a pair bank.
    Uses local split indices, so face_idx/body_idx must refer to TEST_FACE/TEST_BODY.
    """
    face_np = as_numpy_array(face_np)
    body_np = as_numpy_array(body_np)

    fi = bank["face_idx"].to_numpy(dtype=np.int64)
    bi = bank["body_idx"].to_numpy(dtype=np.int64)

    face_batch = np.ascontiguousarray(face_np[fi], dtype=np.float32)
    body_batch = np.ascontiguousarray(body_np[bi], dtype=np.float32)

    return score_model(model, face_batch, body_batch)


# ------------------------------------------------------------
# 4) Run one-window-per-source-video repeated control
# ------------------------------------------------------------
R_ONE_WINDOW = 100
one_rows = []

for r in range(R_ONE_WINDOW):
    chosen = choose_one_window_per_source(
        test_meta,
        seed=12000 + r
    )

    bank = build_balanced_random_bank_from_indices(
        chosen,
        test_meta,
        n_random_per_anchor=1,
        seed=13000 + r,
        exclude_same_source=True
    )

    y = bank["y"].to_numpy(dtype=np.int64)

    for model_name, model in models_for_control:
        s = score_bank_on_split(
            model,
            TEST_FACE,
            TEST_BODY,
            bank
        )

        one_rows.append({
            "experiment": "one_window_per_source_video_control",
            "repeat": r,
            "model": model_name,
            "n_source_videos": int(len(chosen)),
            "n_pairs": int(len(bank)),
            "positive_rate": float(np.mean(y)),
            "roc_auc": safe_auc(y, s),
            "pr_auc": safe_ap(y, s),
            "pos_mean": float(np.mean(s[y == 1])),
            "neg_mean": float(np.mean(s[y == 0])),
        })


# ------------------------------------------------------------
# 5) Save results
# ------------------------------------------------------------
one_df = pd.DataFrame(one_rows)

one_summary = one_df.groupby("model", as_index=False).agg(
    repeats=("repeat", "nunique"),
    n_source_videos_mean=("n_source_videos", "mean"),
    n_pairs_mean=("n_pairs", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", "std"),
    roc_auc_lo=("roc_auc", lambda x: float(np.quantile(x, 0.025))),
    roc_auc_hi=("roc_auc", lambda x: float(np.quantile(x, 0.975))),
    pr_auc_mean=("pr_auc", "mean"),
    pr_auc_std=("pr_auc", "std"),
    pos_mean=("pos_mean", "mean"),
    neg_mean=("neg_mean", "mean"),
)

one_df.to_csv(
    OUT_REVIEWER_ONCE / "one_window_per_source_video_control_repeats.csv",
    index=False
)

one_summary.to_csv(
    OUT_REVIEWER_ONCE / "one_window_per_source_video_control_summary.csv",
    index=False
)

print("Saved:")
print(" -", OUT_REVIEWER_ONCE / "one_window_per_source_video_control_repeats.csv")
print(" -", OUT_REVIEWER_ONCE / "one_window_per_source_video_control_summary.csv")

display(one_summary)

Recreated standardized test arrays:
TEST_FACE: (1440, 12, 30)
TEST_BODY: (1440, 12, 72)
Saved:
 - /content/pmci_reviewer_once_extra_checks/one_window_per_source_video_control_repeats.csv
 - /content/pmci_reviewer_once_extra_checks/one_window_per_source_video_control_summary.csv


,model,repeats,n_source_videos_mean,n_pairs_mean,roc_auc_mean,roc_auc_std,roc_auc_lo,roc_auc_hi,pr_auc_mean,pr_auc_std,pos_mean,neg_mean
0,DirectCosine_K0,100,360.0,720.0,0.941745,0.008764,0.928104,0.959554,0.920458,0.014330,0.822428,0.496474
1,PMCI_K16,100,360.0,720.0,0.807389,0.016278,0.780673,0.834253,0.741254,0.022517,0.743223,0.407110
2,PMCI_K32,100,360.0,720.0,0.892851,0.011828,0.875125,0.921067,0.856346,0.018400,0.823665,0.392397


In [ ]:
# ============================================================
# REVIEWER-ONCE CELL 3 — FAST source-video / actor grouped bootstrap

# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1) Fast grouped bootstrap for ROC-AUC
# ------------------------------------------------------------
def grouped_bootstrap_auc_fast(
    y,
    score,
    groups,
    B=300,
    seed=123
):
    """
    Fast grouped bootstrap for ROC-AUC.

    Resamples groups with replacement and computes ROC-AUC.
    This is enough for reviewer-facing clustered uncertainty.
    """
    rng = np.random.default_rng(seed)

    y = np.asarray(y).astype(int)
    score = np.asarray(score)
    groups = np.asarray(groups)

    unique_groups = np.unique(groups)

    # Precompute indices for each group
    group_to_idx = {
        g: np.where(groups == g)[0]
        for g in unique_groups
    }

    vals = []

    for _ in range(B):
        sampled_groups = rng.choice(
            unique_groups,
            size=len(unique_groups),
            replace=True
        )

        idx_parts = [group_to_idx[g] for g in sampled_groups]
        idx = np.concatenate(idx_parts)

        if len(np.unique(y[idx])) < 2:
            continue

        vals.append(float(roc_auc_score(y[idx], score[idx])))

    vals = np.asarray(vals)

    if len(vals) == 0:
        return {
            "boot_mean": np.nan,
            "boot_lo": np.nan,
            "boot_hi": np.nan,
            "boot_n": 0
        }

    return {
        "boot_mean": float(np.mean(vals)),
        "boot_lo": float(np.quantile(vals, 0.025)),
        "boot_hi": float(np.quantile(vals, 0.975)),
        "boot_n": int(len(vals)),
    }


# ------------------------------------------------------------
# 2) Locate hard-negative score files
# ------------------------------------------------------------
def locate_csv_fast(name):
    candidates = [
        globals().get("OUT_FINAL", None),
        globals().get("OUT_STAT", None),
        globals().get("REV_DIR", None),
        Path("/content/pmci_revision_results_final_extra"),
        Path("/content/pmci_final_statistical_robustness"),
        Path("/content/cias_q1_min/results_pmci_reviewer_continuation_quiet"),
        OUT_REVIEWER_ONCE,
    ]

    for base in candidates:
        if base is None:
            continue

        p = Path(base) / name
        if p.exists():
            return p

    return None


hard_test_path = locate_csv_fast("hard_test_scores_K16_K32.csv")
hard_val_path = locate_csv_fast("hard_val_scores_K16_K32.csv")

if hard_test_path is None:
    raise RuntimeError(
        "hard_test_scores_K16_K32.csv not found. "
        "Run the final hard-negative score cell first."
    )

print("Using hard test scores:", hard_test_path)

hard_scores = pd.read_csv(hard_test_path)

if hard_val_path is not None:
    print("Using hard val scores:", hard_val_path)
else:
    print("No hard val scores found; fused Direct+PMCI scores will be skipped.")


# ------------------------------------------------------------
# 3) Add source-video and actor cluster labels
# ------------------------------------------------------------
meta_local = test_meta.set_index("local_idx")

hard_scores["anchor_source_video"] = hard_scores["anchor"].map(
    lambda i: meta_local.loc[int(i), "source_video"]
    if int(i) in meta_local.index else "unknown"
)

hard_scores["anchor_actor"] = hard_scores["anchor"].map(
    lambda i: int(meta_local.loc[int(i), "subject_id"])
    if int(i) in meta_local.index else -1
)

print("Hard scores shape:", hard_scores.shape)
print("n pairs:", len(hard_scores))
print("n anchors:", hard_scores["anchor"].nunique())
print("n source videos:", hard_scores["anchor_source_video"].nunique())
print("n actors:", hard_scores["anchor_actor"].nunique())


# ------------------------------------------------------------
# 4) Optional fused Direct+PMCI scores using validation hard bank
# ------------------------------------------------------------
if hard_val_path is not None:
    hard_val_scores = pd.read_csv(hard_val_path)

    for K in [16, 32]:
        col = f"pmci_K{K}"

        if col in hard_scores.columns and col in hard_val_scores.columns:
            hard_scores[f"direct_plus_pmci_K{K}_prob"] = fit_logreg_prob(
                hard_val_scores[["direct", col]].to_numpy(),
                hard_val_scores["y"].to_numpy().astype(int),
                hard_scores[["direct", col]].to_numpy(),
            )


# ------------------------------------------------------------
# 5) Select score columns
# ------------------------------------------------------------
eval_cols = [
    c for c in [
        "direct",
        "pmci_K16",
        "pmci_K32",
        "direct_plus_pmci_K16_prob",
        "direct_plus_pmci_K32_prob"
    ]
    if c in hard_scores.columns
]

print("Score columns:", eval_cols)

if len(eval_cols) == 0:
    raise RuntimeError("No score columns found in hard_scores.")


# ------------------------------------------------------------
# 6) Run clustered ROC bootstrap
# ------------------------------------------------------------
rows = []
y = hard_scores["y"].to_numpy().astype(int)

# Faster bootstrap counts.
# Actor-level may have few groups, so do not overdo it.
BOOT_BY_GROUP = {
    "anchor": 300,
    "anchor_source_video": 300,
    "anchor_actor": 200,
}

for group_col in ["anchor", "anchor_source_video", "anchor_actor"]:
    B = BOOT_BY_GROUP[group_col]

    for col in eval_cols:
        s = hard_scores[col].to_numpy()

        print(f"Bootstrapping {col} grouped by {group_col} with B={B}...")

        auc_ci = grouped_bootstrap_auc_fast(
            y,
            s,
            hard_scores[group_col].to_numpy(),
            B=B,
            seed=20000 + len(rows)
        )

        rows.append({
            "experiment": "hard_negative_clustered_bootstrap_fast",
            "group_col": group_col,
            "score_col": col,
            "n_pairs": int(len(hard_scores)),
            "n_groups": int(hard_scores[group_col].nunique()),
            "positive_rate": float(np.mean(y)),

            # Point estimates
            "roc_auc": safe_auc(y, s),
            "pr_auc": safe_ap(y, s),

            # Grouped bootstrap ROC-AUC interval
            "roc_auc_boot_mean": auc_ci["boot_mean"],
            "roc_auc_lo": auc_ci["boot_lo"],
            "roc_auc_hi": auc_ci["boot_hi"],
            "boot_n": auc_ci["boot_n"],
        })


cluster_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# 7) Save outputs
# ------------------------------------------------------------
hard_scores.to_csv(
    OUT_REVIEWER_ONCE / "hard_test_scores_with_source_video_actor_clusters.csv",
    index=False
)

cluster_df.to_csv(
    OUT_REVIEWER_ONCE / "hard_negative_source_video_actor_grouped_bootstrap.csv",
    index=False
)

print("Saved:")
print(" -", OUT_REVIEWER_ONCE / "hard_test_scores_with_source_video_actor_clusters.csv")
print(" -", OUT_REVIEWER_ONCE / "hard_negative_source_video_actor_grouped_bootstrap.csv")

display(cluster_df)

Using hard test scores: /content/pmci_revision_results_final_extra/hard_test_scores_K16_K32.csv
Using hard val scores: /content/pmci_revision_results_final_extra/hard_val_scores_K16_K32.csv
Hard scores shape: (30240, 11)
n pairs: 30240
n anchors: 1440
n source videos: 360
n actors: 3
Score columns: ['direct', 'pmci_K16', 'pmci_K32', 'direct_plus_pmci_K16_prob', 'direct_plus_pmci_K32_prob']
Bootstrapping direct grouped by anchor with B=300...
Bootstrapping pmci_K16 grouped by anchor with B=300...
Bootstrapping pmci_K32 grouped by anchor with B=300...
Bootstrapping direct_plus_pmci_K16_prob grouped by anchor with B=300...
Bootstrapping direct_plus_pmci_K32_prob grouped by anchor with B=300...
Bootstrapping direct grouped by anchor_source_video with B=300...
Bootstrapping pmci_K16 grouped by anchor_source_video with B=300...
Bootstrapping pmci_K32 grouped by anchor_source_video with B=300...
Bootstrapping direct_plus_pmci_K16_prob grouped by anchor_source_video with B=300...
Bootstrapping

,experiment,group_col,score_col,n_pairs,n_groups,positive_rate,roc_auc,pr_auc,roc_auc_boot_mean,roc_auc_lo,roc_auc_hi,boot_n
0,hard_negative_clustered_bootstrap_fast,anchor,direct,30240,1440,0.047619,0.484431,0.045209,0.484488,0.476474,0.490355,300
1,hard_negative_clustered_bootstrap_fast,anchor,pmci_K16,30240,1440,0.047619,0.572225,0.064133,0.571953,0.560405,0.583723,300
2,hard_negative_clustered_bootstrap_fast,anchor,pmci_K32,30240,1440,0.047619,0.569069,0.055293,0.569117,0.557976,0.579036,300
3,hard_negative_clustered_bootstrap_fast,anchor,direct_plus_pmci_K16_prob,30240,1440,0.047619,0.572802,0.083430,0.572964,0.562272,0.582372,300
4,hard_negative_clustered_bootstrap_fast,anchor,direct_plus_pmci_K32_prob,30240,1440,0.047619,0.582499,0.082229,0.582570,0.573151,0.592591,300
5,hard_negative_clustered_bootstrap_fast,anchor_source_video,direct,30240,360,0.047619,0.484431,0.045209,0.483663,0.473181,0.494926,300
6,hard_negative_clustered_bootstrap_fast,anchor_source_video,pmci_K16,30240,360,0.047619,0.572225,0.064133,0.572628,0.553055,0.592924,300
7,hard_negative_clustered_bootstrap_fast,anchor_source_video,pmci_K32,30240,360,0.047619,0.569069,0.055293,0.569907,0.552710,0.589360,300
8,hard_negative_clustered_bootstrap_fast,anchor_source_video,direct_plus_pmci_K16_prob,30240,360,0.047619,0.572802,0.083430,0.573213,0.557450,0.588853,300
9,hard_negative_clustered_bootstrap_fast,anchor_source_video,direct_plus_pmci_K32_prob,30240,360,0.047619,0.582499,0.082229,0.582007,0.569279,0.595639,300


In [ ]:
# ============================================================
# REVIEWER-ONCE CELL 4 — strict identity-excluded hard negatives
# ============================================================
def build_identity_excluded_hard_bank(
    face_np,
    body_np,
    meta,
    direct_model,
    hard_k=20,
    max_anchors=240,
    seed=123
):
    rng = np.random.default_rng(seed)

    n = len(face_np)
    anchors = np.arange(n, dtype=int)

    if max_anchors is not None and max_anchors < n:
        anchors = np.sort(
            rng.choice(anchors, size=max_anchors, replace=False)
        )

    meta_local = meta.set_index("local_idx")
    all_idx = np.arange(n, dtype=int)
    rows = []

    for count, i in enumerate(anchors, start=1):
        src_i = meta_local.loc[int(i), "source_video"]
        actor_i = int(meta_local.loc[int(i), "subject_id"])

        rows.append({
            "anchor": int(i),
            "face_idx": int(i),
            "body_idx": int(i),
            "y": 1,
            "pair_type": "true_pair",
            "anchor_source_video": src_i,
            "anchor_actor": actor_i,
            "body_source_video": src_i,
            "body_actor": actor_i,
        })

        candidates = []

        for j in all_idx:
            if j == i:
                continue

            src_j = meta_local.loc[int(j), "source_video"]
            actor_j = int(meta_local.loc[int(j), "subject_id"])

            if src_j == src_i:
                continue

            if actor_j == actor_i:
                continue

            candidates.append(int(j))

        candidates = np.array(candidates, dtype=int)

        if len(candidates) == 0:
            continue

        f_rep = np.repeat(
            face_np[i:i+1],
            repeats=len(candidates),
            axis=0
        )

        direct_s = score_model(
            direct_model,
            f_rep,
            body_np[candidates]
        )

        hk = min(hard_k, len(candidates))
        top_pos = np.argsort(direct_s)[-hk:][::-1]
        hard_idx = candidates[top_pos]

        for j in hard_idx:
            src_j = meta_local.loc[int(j), "source_video"]
            actor_j = int(meta_local.loc[int(j), "subject_id"])

            rows.append({
                "anchor": int(i),
                "face_idx": int(i),
                "body_idx": int(j),
                "y": 0,
                "pair_type": "identity_excluded_direct_hard_negative",
                "anchor_source_video": src_i,
                "anchor_actor": actor_i,
                "body_source_video": src_j,
                "body_actor": actor_j,
            })

    return pd.DataFrame(rows)

def score_identity_bank(bank, face_np, body_np, models):
    out = bank.copy()

    face_indices = out["face_idx"].to_numpy().astype(int)
    body_indices = out["body_idx"].to_numpy().astype(int)

    for name, model in models:
        out[name] = score_model(
            model,
            face_np[face_indices],
            body_np[body_indices]
        )

    return out

STRICT_HARD_K = 20
STRICT_MAX_ANCHORS = 240

strict_val_bank = build_identity_excluded_hard_bank(
    std["val_face"],
    std["val_body"],
    val_meta,
    DIRECT,
    hard_k=STRICT_HARD_K,
    max_anchors=STRICT_MAX_ANCHORS,
    seed=31001
)

strict_test_bank = build_identity_excluded_hard_bank(
    std["test_face"],
    std["test_body"],
    test_meta,
    DIRECT,
    hard_k=STRICT_HARD_K,
    max_anchors=STRICT_MAX_ANCHORS,
    seed=31002
)

strict_models = [("direct", DIRECT)]

if PMCI16 is not None:
    strict_models.append(("pmci_K16", PMCI16))

if PMCI32 is not None:
    strict_models.append(("pmci_K32", PMCI32))

strict_val_scores = score_identity_bank(
    strict_val_bank,
    std["val_face"],
    std["val_body"],
    strict_models
)

strict_test_scores = score_identity_bank(
    strict_test_bank,
    std["test_face"],
    std["test_body"],
    strict_models
)

for K in [16, 32]:
    col = f"pmci_K{K}"

    if col in strict_val_scores.columns and col in strict_test_scores.columns:
        strict_test_scores[f"direct_plus_pmci_K{K}_prob"] = fit_logreg_prob(
            strict_val_scores[["direct", col]].to_numpy(),
            strict_val_scores["y"].to_numpy().astype(int),
            strict_test_scores[["direct", col]].to_numpy(),
        )

strict_eval_cols = [
    c for c in [
        "direct",
        "pmci_K16",
        "pmci_K32",
        "direct_plus_pmci_K16_prob",
        "direct_plus_pmci_K32_prob"
    ]
    if c in strict_test_scores.columns
]

strict_rows = []
y = strict_test_scores["y"].to_numpy().astype(int)

for col in strict_eval_cols:
    s = strict_test_scores[col].to_numpy()

    auc_src = grouped_bootstrap_metric(
        y,
        s,
        strict_test_scores["anchor_source_video"].to_numpy(),
        roc_auc_score,
        B=2000,
        seed=33000 + len(strict_rows)
    )

    strict_rows.append({
        "experiment": "identity_excluded_hard_negative",
        "score_col": col,
        "n_val_pairs": int(len(strict_val_scores)),
        "n_test_pairs": int(len(strict_test_scores)),
        "n_test_anchor_source_videos": int(strict_test_scores["anchor_source_video"].nunique()),
        "positive_rate": float(np.mean(y)),
        "roc_auc": safe_auc(y, s),
        "pr_auc": safe_ap(y, s),
        "pos_mean": float(np.mean(s[y == 1])),
        "neg_mean": float(np.mean(s[y == 0])),
        "source_video_boot_auc_mean": auc_src["boot_mean"],
        "source_video_boot_auc_lo": auc_src["boot_lo"],
        "source_video_boot_auc_hi": auc_src["boot_hi"],
    })

strict_summary = pd.DataFrame(strict_rows)

strict_val_scores.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_val_scores.csv",
    index=False
)

strict_test_scores.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_test_scores.csv",
    index=False
)

strict_summary.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_negative_summary.csv",
    index=False
)

display(strict_summary)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
# ============================================================
# REVIEWER-ONCE CELL 5 — zip outputs
# ============================================================
zip_path = shutil.make_archive(
    str(OUT_REVIEWER_ONCE),
    "zip",
    OUT_REVIEWER_ONCE
)

print("Zipped reviewer-once extra checks:", zip_path)
print("Files:")

for p in sorted(OUT_REVIEWER_ONCE.glob("*")):
    print(" -", p.name)

In [ ]:
# ============================================================
# REVIEWER-ONCE CELL 4 — strict identity-excluded hard negatives
# FIXED FULL VERSION
#
# Purpose:
#   Test whether the hard-negative effect survives when negative pairs exclude:
#   1) the same source video;
#   2) the same actor.

# ============================================================

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1) Load models safely
# ------------------------------------------------------------
DIRECT = get_model_safe("DirectCosine_K0")

PMCI16 = get_model_safe("PMCI_K16") if "PMCI_K16" in trained_models else None
PMCI32 = get_model_safe("PMCI_K32") if "PMCI_K32" in trained_models else None

strict_models = [("direct", DIRECT)]

if PMCI16 is not None:
    strict_models.append(("pmci_K16", PMCI16))

if PMCI32 is not None:
    strict_models.append(("pmci_K32", PMCI32))

print("Models used:", [x[0] for x in strict_models])


# ------------------------------------------------------------
# 2) Recreate standardized arrays safely
# ------------------------------------------------------------
def reviewer_standardize_train_only_v2(face, body, train_idx, val_idx, test_idx):
    """
    Recreates standardized train/val/test arrays from the original data.
    This avoids using global variable 'std', which may be overwritten.
    """
    face = np.asarray(face, dtype=np.float32)
    body = np.asarray(body, dtype=np.float32)

    train_idx = np.asarray(train_idx, dtype=np.int64)
    val_idx = np.asarray(val_idx, dtype=np.int64)
    test_idx = np.asarray(test_idx, dtype=np.int64)

    T_face, D_face = face.shape[1], face.shape[2]
    T_body, D_body = body.shape[1], body.shape[2]

    face_scaler = StandardScaler()
    body_scaler = StandardScaler()

    face_scaler.fit(face[train_idx].reshape(-1, D_face))
    body_scaler.fit(body[train_idx].reshape(-1, D_body))

    def transform_face(x):
        x = np.asarray(x, dtype=np.float32)
        return face_scaler.transform(
            x.reshape(-1, D_face)
        ).reshape(x.shape).astype(np.float32)

    def transform_body(x):
        x = np.asarray(x, dtype=np.float32)
        return body_scaler.transform(
            x.reshape(-1, D_body)
        ).reshape(x.shape).astype(np.float32)

    return {
        "train_face": transform_face(face[train_idx]),
        "train_body": transform_body(body[train_idx]),
        "val_face": transform_face(face[val_idx]),
        "val_body": transform_body(body[val_idx]),
        "test_face": transform_face(face[test_idx]),
        "test_body": transform_body(body[test_idx]),
    }


# Reuse arrays from Cell 2 if they already exist; otherwise recreate.
if "std_reviewer_once" not in globals():
    std_reviewer_once = reviewer_standardize_train_only_v2(
        data["face"],
        data["body"],
        train_idx,
        val_idx,
        test_idx
    )

VAL_FACE = std_reviewer_once["val_face"]
VAL_BODY = std_reviewer_once["val_body"]
TEST_FACE = std_reviewer_once["test_face"]
TEST_BODY = std_reviewer_once["test_body"]

print("VAL_FACE:", VAL_FACE.shape)
print("VAL_BODY:", VAL_BODY.shape)
print("TEST_FACE:", TEST_FACE.shape)
print("TEST_BODY:", TEST_BODY.shape)


# ------------------------------------------------------------
# 3) Helper functions
# ------------------------------------------------------------
def as_numpy_array_v2(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def grouped_bootstrap_auc_fast_v2(
    y,
    score,
    groups,
    B=300,
    seed=123
):
    """
    Fast grouped bootstrap for ROC-AUC.
    Used here only for source-video grouped uncertainty.
    """
    rng = np.random.default_rng(seed)

    y = np.asarray(y).astype(int)
    score = np.asarray(score)
    groups = np.asarray(groups)

    unique_groups = np.unique(groups)

    group_to_idx = {
        g: np.where(groups == g)[0]
        for g in unique_groups
    }

    vals = []

    for _ in range(B):
        sampled_groups = rng.choice(
            unique_groups,
            size=len(unique_groups),
            replace=True
        )

        idx = np.concatenate([
            group_to_idx[g]
            for g in sampled_groups
        ])

        if len(np.unique(y[idx])) < 2:
            continue

        vals.append(float(roc_auc_score(y[idx], score[idx])))

    vals = np.asarray(vals)

    if len(vals) == 0:
        return {
            "boot_mean": np.nan,
            "boot_lo": np.nan,
            "boot_hi": np.nan,
            "boot_n": 0
        }

    return {
        "boot_mean": float(np.mean(vals)),
        "boot_lo": float(np.quantile(vals, 0.025)),
        "boot_hi": float(np.quantile(vals, 0.975)),
        "boot_n": int(len(vals)),
    }


def score_pair_arrays(model, face_np, body_np, batch_size=512):
    """
    Scores already aligned face/body arrays.
    """
    face_np = np.ascontiguousarray(as_numpy_array_v2(face_np), dtype=np.float32)
    body_np = np.ascontiguousarray(as_numpy_array_v2(body_np), dtype=np.float32)

    return np.asarray(
        score_model(
            model,
            face_np,
            body_np,
            batch_size=batch_size
        )
    ).reshape(-1)


def score_identity_bank(bank, face_np, body_np, models):
    """
    Scores a bank where face_idx/body_idx are local indices for the split arrays.
    """
    out = bank.copy()

    face_np = as_numpy_array_v2(face_np)
    body_np = as_numpy_array_v2(body_np)

    face_indices = out["face_idx"].to_numpy(dtype=np.int64)
    body_indices = out["body_idx"].to_numpy(dtype=np.int64)

    face_batch = np.ascontiguousarray(face_np[face_indices], dtype=np.float32)
    body_batch = np.ascontiguousarray(body_np[body_indices], dtype=np.float32)

    for name, model in models:
        out[name] = score_pair_arrays(
            model,
            face_batch,
            body_batch,
            batch_size=512
        )

    return out


def build_identity_excluded_hard_bank(
    face_np,
    body_np,
    meta,
    direct_model,
    hard_k=20,
    max_anchors=240,
    seed=123,
    verbose=True
):
    """
    Builds hard negatives where negative body windows are selected using
    DirectCosine scores, but candidates from the same actor and same source
    video are excluded.

    Positive:
        face_i with body_i

    Negative:
        face_i with body_j, where:
            source_video_j != source_video_i
            actor_j != actor_i
        and j is among top DirectCosine hard negatives.
    """
    rng = np.random.default_rng(seed)

    face_np = as_numpy_array_v2(face_np)
    body_np = as_numpy_array_v2(body_np)

    n = len(face_np)
    all_idx = np.arange(n, dtype=np.int64)

    anchors = np.arange(n, dtype=np.int64)

    if max_anchors is not None and max_anchors < n:
        anchors = np.sort(
            rng.choice(
                anchors,
                size=max_anchors,
                replace=False
            )
        )

    meta_local = meta.set_index("local_idx")

    rows = []

    for count, i in enumerate(anchors, start=1):
        i = int(i)

        src_i = meta_local.loc[i, "source_video"]
        actor_i = int(meta_local.loc[i, "subject_id"])

        # Positive pair
        rows.append({
            "anchor": i,
            "face_idx": i,
            "body_idx": i,
            "y": 1,
            "pair_type": "true_pair",
            "anchor_source_video": src_i,
            "anchor_actor": actor_i,
            "body_source_video": src_i,
            "body_actor": actor_i,
        })

        # Strict candidate negatives:
        # not same source video, not same actor
        candidates = []

        for j in all_idx:
            j = int(j)

            if j == i:
                continue

            src_j = meta_local.loc[j, "source_video"]
            actor_j = int(meta_local.loc[j, "subject_id"])

            if src_j == src_i:
                continue

            if actor_j == actor_i:
                continue

            candidates.append(j)

        candidates = np.asarray(candidates, dtype=np.int64)

        if len(candidates) == 0:
            continue

        # Score all candidates by DirectCosine and take hardest top-K
        face_rep = np.repeat(
            face_np[i:i + 1],
            repeats=len(candidates),
            axis=0
        )

        direct_scores = score_pair_arrays(
            direct_model,
            face_rep,
            body_np[candidates],
            batch_size=512
        )

        hk = min(int(hard_k), len(candidates))
        top_pos = np.argsort(direct_scores)[-hk:][::-1]
        hard_idx = candidates[top_pos]

        for j in hard_idx:
            j = int(j)

            src_j = meta_local.loc[j, "source_video"]
            actor_j = int(meta_local.loc[j, "subject_id"])

            rows.append({
                "anchor": i,
                "face_idx": i,
                "body_idx": j,
                "y": 0,
                "pair_type": "identity_excluded_direct_hard_negative",
                "anchor_source_video": src_i,
                "anchor_actor": actor_i,
                "body_source_video": src_j,
                "body_actor": actor_j,
            })

        if verbose and (count == 1 or count % 25 == 0 or count == len(anchors)):
            print(
                f"Built strict hard bank: {count}/{len(anchors)} anchors, "
                f"rows={len(rows)}"
            )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 4) Build strict identity-excluded hard-negative banks
# ------------------------------------------------------------

STRICT_HARD_K = 20
STRICT_MAX_ANCHORS = 240

print("\nBuilding strict validation hard-negative bank...")
strict_val_bank = build_identity_excluded_hard_bank(
    VAL_FACE,
    VAL_BODY,
    val_meta,
    DIRECT,
    hard_k=STRICT_HARD_K,
    max_anchors=STRICT_MAX_ANCHORS,
    seed=31001,
    verbose=True
)

print("\nBuilding strict test hard-negative bank...")
strict_test_bank = build_identity_excluded_hard_bank(
    TEST_FACE,
    TEST_BODY,
    test_meta,
    DIRECT,
    hard_k=STRICT_HARD_K,
    max_anchors=STRICT_MAX_ANCHORS,
    seed=31002,
    verbose=True
)

print("Strict val bank:", strict_val_bank.shape)
print("Strict test bank:", strict_test_bank.shape)


# ------------------------------------------------------------
# 5) Score strict banks
# ------------------------------------------------------------
print("\nScoring strict validation bank...")
strict_val_scores = score_identity_bank(
    strict_val_bank,
    VAL_FACE,
    VAL_BODY,
    strict_models
)

print("Scoring strict test bank...")
strict_test_scores = score_identity_bank(
    strict_test_bank,
    TEST_FACE,
    TEST_BODY,
    strict_models
)


# ------------------------------------------------------------
# 6) Optional Direct+PMCI fusion trained on strict val bank
# ------------------------------------------------------------
for K in [16, 32]:
    col = f"pmci_K{K}"

    if col in strict_val_scores.columns and col in strict_test_scores.columns:
        strict_test_scores[f"direct_plus_pmci_K{K}_prob"] = fit_logreg_prob(
            strict_val_scores[["direct", col]].to_numpy(),
            strict_val_scores["y"].to_numpy().astype(int),
            strict_test_scores[["direct", col]].to_numpy(),
        )


# ------------------------------------------------------------
# 7) Summarize results
# ------------------------------------------------------------
strict_eval_cols = [
    c for c in [
        "direct",
        "pmci_K16",
        "pmci_K32",
        "direct_plus_pmci_K16_prob",
        "direct_plus_pmci_K32_prob"
    ]
    if c in strict_test_scores.columns
]

strict_rows = []
y = strict_test_scores["y"].to_numpy(dtype=np.int64)

for col in strict_eval_cols:
    s = strict_test_scores[col].to_numpy()

    auc_src = grouped_bootstrap_auc_fast_v2(
        y,
        s,
        strict_test_scores["anchor_source_video"].to_numpy(),
        B=300,
        seed=33000 + len(strict_rows)
    )

    strict_rows.append({
        "experiment": "identity_excluded_hard_negative",
        "score_col": col,

        "n_val_pairs": int(len(strict_val_scores)),
        "n_test_pairs": int(len(strict_test_scores)),
        "n_test_anchors": int(strict_test_scores["anchor"].nunique()),
        "n_test_anchor_source_videos": int(strict_test_scores["anchor_source_video"].nunique()),
        "n_test_anchor_actors": int(strict_test_scores["anchor_actor"].nunique()),
        "hard_k": int(STRICT_HARD_K),
        "max_anchors": int(STRICT_MAX_ANCHORS),
        "positive_rate": float(np.mean(y)),

        "roc_auc": safe_auc(y, s),
        "pr_auc": safe_ap(y, s),
        "pos_mean": float(np.mean(s[y == 1])),
        "neg_mean": float(np.mean(s[y == 0])),

        "source_video_boot_auc_mean": auc_src["boot_mean"],
        "source_video_boot_auc_lo": auc_src["boot_lo"],
        "source_video_boot_auc_hi": auc_src["boot_hi"],
        "source_video_boot_n": auc_src["boot_n"],
    })

strict_summary = pd.DataFrame(strict_rows)


# ------------------------------------------------------------
# 8) Save outputs
# ------------------------------------------------------------
strict_val_bank.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_val_bank.csv",
    index=False
)

strict_test_bank.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_test_bank.csv",
    index=False
)

strict_val_scores.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_val_scores.csv",
    index=False
)

strict_test_scores.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_test_scores.csv",
    index=False
)

strict_summary.to_csv(
    OUT_REVIEWER_ONCE / "identity_excluded_hard_negative_summary.csv",
    index=False
)

print("\nSaved:")
print(" -", OUT_REVIEWER_ONCE / "identity_excluded_hard_val_bank.csv")
print(" -", OUT_REVIEWER_ONCE / "identity_excluded_hard_test_bank.csv")
print(" -", OUT_REVIEWER_ONCE / "identity_excluded_hard_val_scores.csv")
print(" -", OUT_REVIEWER_ONCE / "identity_excluded_hard_test_scores.csv")
print(" -", OUT_REVIEWER_ONCE / "identity_excluded_hard_negative_summary.csv")

display(strict_summary)

Models used: ['direct', 'pmci_K16', 'pmci_K32']
VAL_FACE: (1920, 12, 30)
VAL_BODY: (1920, 12, 72)
TEST_FACE: (1440, 12, 30)
TEST_BODY: (1440, 12, 72)

Building strict validation hard-negative bank...
Built strict hard bank: 1/240 anchors, rows=21
Built strict hard bank: 25/240 anchors, rows=525
Built strict hard bank: 50/240 anchors, rows=1050
Built strict hard bank: 75/240 anchors, rows=1575
Built strict hard bank: 100/240 anchors, rows=2100
Built strict hard bank: 125/240 anchors, rows=2625
Built strict hard bank: 150/240 anchors, rows=3150
Built strict hard bank: 175/240 anchors, rows=3675
Built strict hard bank: 200/240 anchors, rows=4200
Built strict hard bank: 225/240 anchors, rows=4725
Built strict hard bank: 240/240 anchors, rows=5040

Building strict test hard-negative bank...
Built strict hard bank: 1/240 anchors, rows=21
Built strict hard bank: 25/240 anchors, rows=525
Built strict hard bank: 50/240 anchors, rows=1050
Built strict hard bank: 75/240 anchors, rows=1575
Built s

,experiment,score_col,n_val_pairs,n_test_pairs,n_test_anchors,n_test_anchor_source_videos,n_test_anchor_actors,hard_k,max_anchors,positive_rate,roc_auc,pr_auc,pos_mean,neg_mean,source_video_boot_auc_mean,source_video_boot_auc_lo,source_video_boot_auc_hi,source_video_boot_n
0,identity_excluded_hard_negative,direct,5040,5040,240,189,3,20,240,0.047619,0.820536,0.370676,0.824641,0.733157,0.820429,0.793446,0.846552,300
1,identity_excluded_hard_negative,pmci_K16,5040,5040,240,189,3,20,240,0.047619,0.672496,0.085807,0.747660,0.559961,0.670869,0.624721,0.713114,300
2,identity_excluded_hard_negative,pmci_K32,5040,5040,240,189,3,20,240,0.047619,0.800154,0.149019,0.834453,0.537327,0.800363,0.771270,0.828999,300
3,identity_excluded_hard_negative,direct_plus_pmci_K16_prob,5040,5040,240,189,3,20,240,0.047619,0.656938,0.071506,0.475509,0.324898,0.654998,0.606520,0.702970,300
4,identity_excluded_hard_negative,direct_plus_pmci_K32_prob,5040,5040,240,189,3,20,240,0.047619,0.811232,0.216048,0.586140,0.331608,0.811885,0.782561,0.840108,300


In [ ]:
# ============================================================
# FINAL CELL — collect all PMCI revision outputs into one ZIP
# ============================================================

import os
import shutil
import json
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 1) Final package paths
# ------------------------------------------------------------
FINAL_PACK_DIR = Path("/content/pmci_all_revision_outputs_final_package")
FINAL_ZIP_PATH = Path("/content/pmci_all_revision_outputs_final_package.zip")

# Remove old package if exists
if FINAL_PACK_DIR.exists():
    shutil.rmtree(FINAL_PACK_DIR)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

FINAL_PACK_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2) Candidate result directories
# ------------------------------------------------------------
candidate_dirs = [
    # Main continuation / expanded-window outputs
    Path("/content/cias_q1_min/results_pmci_reviewer_continuation_quiet"),

    # Previously generated final packs / checks
    Path("/content/pmci_revision_results_final_extra"),
    Path("/content/pmci_optional_final_experiments"),
    Path("/content/pmci_last_extra_checks"),
    Path("/content/pmci_final_statistical_robustness"),
    Path("/content/pmci_final_audit_pack"),

    # New reviewer-once extra checks
    Path("/content/pmci_reviewer_once_extra_checks"),
]

# Also include globals if they exist
for var_name in ["OUT_FINAL", "OUT_STAT", "REV_DIR", "OUT_REVIEWER_ONCE"]:
    if var_name in globals():
        try:
            candidate_dirs.append(Path(globals()[var_name]))
        except Exception:
            pass

# Deduplicate while preserving order
seen = set()
unique_dirs = []

for p in candidate_dirs:
    p = Path(p)
    key = str(p.resolve()) if p.exists() else str(p)

    if key not in seen:
        seen.add(key)
        unique_dirs.append(p)


# ------------------------------------------------------------
# 3) Copy existing directories into final package
# ------------------------------------------------------------
included = []
missing = []

for src_dir in unique_dirs:
    if src_dir.exists() and src_dir.is_dir():
        dst_dir = FINAL_PACK_DIR / src_dir.name

        # Avoid name collision
        if dst_dir.exists():
            suffix = 2
            while (FINAL_PACK_DIR / f"{src_dir.name}_{suffix}").exists():
                suffix += 1
            dst_dir = FINAL_PACK_DIR / f"{src_dir.name}_{suffix}"

        shutil.copytree(
            src_dir,
            dst_dir,
            ignore=shutil.ignore_patterns(
                "__pycache__",
                ".ipynb_checkpoints",
                "*.tmp",
                "*.lock"
            )
        )

        included.append({
            "source": str(src_dir),
            "copied_to": str(dst_dir),
            "n_files": sum(1 for x in dst_dir.rglob("*") if x.is_file())
        })
    else:
        missing.append(str(src_dir))


# ------------------------------------------------------------
# 4) Add manifest / README
# ------------------------------------------------------------
manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "final_package_dir": str(FINAL_PACK_DIR),
    "final_zip_path": str(FINAL_ZIP_PATH),
    "included_directories": included,
    "missing_directories": missing,
    "note": (
        "This archive combines the main PMCI revision outputs, "
        "expanded-window results, optional checks, statistical robustness outputs, "
        "audit files, and final reviewer-targeted extra checks."
    )
}

with open(FINAL_PACK_DIR / "MANIFEST.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

readme_text = """# PMCI Revision Final Output Package

This ZIP archive combines the main result folders generated during the PMCI revision experiments.

Main purpose:
- expanded RAVDESS multi-window results;
- standard pair-matching results;
- hard-negative / non-redundancy checks;
- shuffled / random / untrained controls if available;
- statistical robustness outputs;
- reviewer-targeted extra checks:
  - one-window-per-source-video control;
  - source-video / actor grouped bootstrap;
  - strict identity-excluded hard negatives.

Important interpretation:
The expanded RAVDESS cache increases the number of pose-derived windows and uses all 24 actors with multiple temporal 12-frame crops per source video. However, multiple windows from the same source video should not be treated as fully independent behavioral samples. The reviewer-targeted controls are included to address this issue directly.
"""

with open(FINAL_PACK_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)


# ------------------------------------------------------------
# 5) Create ZIP
# ------------------------------------------------------------
zip_base = str(FINAL_PACK_DIR)
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=FINAL_PACK_DIR
)

print("DONE ✅")
print("Final ZIP:", zip_path)
print()
print("Included directories:")

for item in included:
    print(f" - {item['source']}  -> files: {item['n_files']}")

if missing:
    print()
    print("Missing / skipped directories:")

    for m in missing:
        print(" -", m)


# ------------------------------------------------------------
# 6) Optional Colab download
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print()
    print("Manual download path:")
    print(zip_path)

DONE ✅
Final ZIP: /content/pmci_all_revision_outputs_final_package.zip

Included directories:
 - /content/cias_q1_min/results_pmci_reviewer_continuation_quiet  -> files: 14
 - /content/pmci_revision_results_final_extra  -> files: 30
 - /content/pmci_optional_final_experiments  -> files: 8
 - /content/pmci_last_extra_checks  -> files: 4
 - /content/pmci_final_statistical_robustness  -> files: 7
 - /content/pmci_final_audit_pack  -> files: 9
 - /content/pmci_reviewer_once_extra_checks  -> files: 11
 - /content/pmci_revision_results_v6  -> files: 24


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# V7 CELL 1 — Setup for final figures and MELD sensitivity
# ============================================================
import os, json, math, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUT_V7 = Path("/content/pmci_meld_sensitivity_v7")
OUT_FIG_V7 = OUT_V7 / "figures_final"
OUT_MELD_V7 = OUT_V7 / "meld_sensitivity"

OUT_FIG_V7.mkdir(parents=True, exist_ok=True)
OUT_MELD_V7.mkdir(parents=True, exist_ok=True)

print("V7 output folder:", OUT_V7)
print("Final figures folder:", OUT_FIG_V7)
print("MELD sensitivity folder:", OUT_MELD_V7)

def save_status(name, **kwargs):
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"), **kwargs}
    path = OUT_MELD_V7 / name
    pd.DataFrame([row]).to_csv(path, index=False)
    print("saved:", path)
    return path

def save_df_v7(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("saved:", path)
    return path


In [ ]:
# ============================================================
# V7 CELL 2 — Final Figure 2 and Figure 3 generators
# Purpose:
#   Supersedes older figure cells that used the pre-revision RAVDESS values.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.30,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_figure_v7(fig, filename):
    png = OUT_FIG_V7 / f"{filename}.png"
    pdf = OUT_FIG_V7 / f"{filename}.pdf"
    svg = OUT_FIG_V7 / f"{filename}.svg"
    fig.savefig(png, dpi=600, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    print("saved:", png)
    print("saved:", pdf)
    print("saved:", svg)

# -----------------------------
# Figure 2: Expanded RAVDESS standard pair matching
# -----------------------------
models_f2 = ["DirectCosine\n(K=0)", "PMCI\n(K=8)", "PMCI\n(K=16)", "PMCI\n(K=32)"]
roc_auc_f2 = np.array([0.943, 0.851, 0.805, 0.887])
pr_auc_f2  = np.array([0.917, 0.814, 0.737, 0.847])

x = np.arange(len(models_f2))
width = 0.34

fig, ax = plt.subplots(figsize=(8.8, 5.6))
bars1 = ax.bar(x - width/2, roc_auc_f2, width, label="ROC-AUC")
bars2 = ax.bar(x + width/2, pr_auc_f2,  width, label="PR-AUC")

ax.set_title("Expanded RAVDESS standard pair-matching evaluation", fontweight="bold", pad=14)
ax.set_ylabel("Score")
ax.set_xlabel("Model", labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(models_f2)
ax.set_ylim(0.65, 1.00)
ax.set_yticks(np.arange(0.65, 1.001, 0.05))
ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.55)
ax.set_axisbelow(True)
ax.legend(title="Metric", loc="upper right", frameon=True)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:.3f}",
            xy=(bar.get_x() + bar.get_width()/2, h),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
        )

fig.tight_layout()
save_figure_v7(fig, "figure_2_expanded_ravdess_standard_pair_matching")

# -----------------------------
# Figure 3: Expanded RAVDESS disruption-control evaluation
# -----------------------------
models_f3 = ["DirectCosine\n(K=0)", "PMCI\n(K=8)", "PMCI\n(K=16)", "PMCI\n(K=32)"]
conditions_f3 = [
    "Normal pair\nmatching",
    "Random body",
    "Face zero",
    "Body zero",
    "Both time-\nreversed",
]

values_f3 = np.array([
    [0.943, 0.942, 0.980, 0.985, 0.480],
    [0.851, 0.838, 0.814, 0.794, 0.539],
    [0.805, 0.800, 0.819, 0.633, 0.501],
    [0.887, 0.893, 0.879, 0.840, 0.484],
])

x = np.arange(len(models_f3))
n_conditions = len(conditions_f3)
width = 0.14
offsets = (np.arange(n_conditions) - (n_conditions - 1) / 2) * width

fig, ax = plt.subplots(figsize=(14.5, 6.8))
all_bars = []

for j, condition in enumerate(conditions_f3):
    bars = ax.bar(x + offsets[j], values_f3[:, j], width, label=condition)
    all_bars.append(bars)

# Prevent value-label overlap by alternating vertical offsets.
vertical_offsets = [4, 10, 16, 10, 4]
horizontal_nudges = [-4, -2, 0, 2, 4]

for j, bars in enumerate(all_bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:.3f}",
            xy=(bar.get_x() + bar.get_width()/2, h),
            xytext=(horizontal_nudges[j], vertical_offsets[j]),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.2,
        )

ax.set_title("Expanded RAVDESS disruption-control evaluation", fontweight="bold", pad=12)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Model", labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(models_f3)
ax.set_ylim(0.0, 1.09)
ax.set_yticks(np.arange(0.0, 1.01, 0.2))
ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.55)
ax.set_axisbelow(True)

ax.legend(
    title="Condition",
    bbox_to_anchor=(1.01, 1.0),
    loc="upper left",
    borderaxespad=0.0,
    frameon=True,
    fontsize=9.3,
    title_fontsize=10.3,
)

fig.tight_layout(rect=[0, 0, 0.84, 1])
save_figure_v7(fig, "figure_3_expanded_ravdess_disruption_control")

print("\nRecommended captions:")
print("Figure 2. Expanded RAVDESS standard pair-matching evaluation: ROC–AUC and PR–AUC across models.")
print("Figure 3. Expanded RAVDESS disruption-control evaluation: ROC–AUC for normal pair matching and artificially disrupted counterparts.")
print("Table 4. Expanded RAVDESS disruption-control evaluation (ROC–AUC; true pairs versus disrupted counterparts).")


In [ ]:
# ============================================================
# V7 CELL 3 — MELD runtime discovery
# Purpose:
#   Find whether this runtime contains MELD arrays, metadata, or score files.
#   This cell does NOT modify the manuscript. It only audits what is available.
# ============================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd

def _shape_or_none(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None

def discover_meld_globals():
    rows = []

    for name, obj in sorted(globals().items()):
        lname = name.lower()

        if "meld" not in lname:
            continue

        row = {
            "name": name,
            "type": type(obj).__name__,
            "shape": str(_shape_or_none(obj)),
            "columns": "",
            "keys": "",
        }

        if isinstance(obj, pd.DataFrame):
            row["columns"] = ", ".join(map(str, obj.columns[:30]))

        if isinstance(obj, dict):
            try:
                row["keys"] = ", ".join(map(str, list(obj.keys())[:30]))
            except Exception:
                pass

        rows.append(row)

    return pd.DataFrame(rows)

meld_global_df = discover_meld_globals()
save_df_v7(meld_global_df, OUT_MELD_V7 / "meld_runtime_global_discovery.csv")
display(meld_global_df)

def limited_walk_for_meld_files(roots, max_depth=5, max_items=3000):
    rows = []
    exts_video = {".mp4", ".avi", ".mov", ".mkv"}
    exts_data = {".csv", ".pkl", ".pickle", ".npz", ".npy", ".parquet", ".json"}

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        root_depth = len(root.parts)
        n_seen = 0

        for dirpath, dirnames, filenames in os.walk(root):
            dpath = Path(dirpath)
            depth = len(dpath.parts) - root_depth
            if depth > max_depth:
                dirnames[:] = []
                continue

            # Avoid huge/system folders.
            dirnames[:] = [d for d in dirnames if d not in {".git", "__pycache__", "sample_data"}]

            joined = " ".join([str(dpath).lower()] + [f.lower() for f in filenames[:20]])
            if "meld" not in joined:
                continue

            for fn in filenames:
                p = dpath / fn
                lp = str(p).lower()
                if "meld" not in lp:
                    continue
                if p.suffix.lower() in exts_video | exts_data:
                    rows.append({
                        "path": str(p),
                        "suffix": p.suffix.lower(),
                        "kind": "video" if p.suffix.lower() in exts_video else "data",
                        "size_mb": round(p.stat().st_size / (1024**2), 3) if p.exists() else np.nan,
                    })
                    n_seen += 1
                    if n_seen >= max_items:
                        break
            if n_seen >= max_items:
                break

    return pd.DataFrame(rows)

candidate_roots = [Path("/content"), Path("/content/drive/MyDrive")]
meld_file_df = limited_walk_for_meld_files(candidate_roots)
save_df_v7(meld_file_df, OUT_MELD_V7 / "meld_file_discovery.csv")
display(meld_file_df.head(50))

print("MELD global candidates:", len(meld_global_df))
print("MELD file candidates:", len(meld_file_df))


In [ ]:
# ============================================================
# V7 CELL 4 — MELD existing-window stability check

# ============================================================
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd
import torch

def _safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

def _safe_ap(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, s))

def _bootstrap_ci(y, s, metric_fn, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    s = np.asarray(s)
    vals = []
    n = len(y)

    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(float(metric_fn(y[idx], s[idx])))

    vals = np.asarray(vals)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    return (
        float(np.mean(vals)),
        float(np.quantile(vals, 0.025)),
        float(np.quantile(vals, 0.975)),
        int(len(vals)),
    )

def _unwrap_model_v7(obj):
    if isinstance(obj, dict):
        return obj.get("model", obj.get("net", obj))
    return obj

def _score_model_v7(model, face_np, body_np, batch_size=512):
    if "score_pairs_safe" in globals():
        return score_pairs_safe(model, face_np, body_np, batch_size=batch_size)
    if "score_pairs" in globals():
        return score_pairs(model, face_np, body_np)
    device = globals().get("DEVICE_TORCH", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.to(device).eval()
    out = []
    with torch.no_grad():
        for st in range(0, len(face_np), batch_size):
            f = torch.tensor(np.ascontiguousarray(face_np[st:st+batch_size]).astype(np.float32), device=device)
            b = torch.tensor(np.ascontiguousarray(body_np[st:st+batch_size]).astype(np.float32), device=device)
            out.append(model.score(f, b).detach().cpu().numpy())
    return np.concatenate(out)

def find_meld_dataset_candidates():
    candidates = []

    # Dict-style candidates: meld_std, meld_data, etc.
    for name, obj in globals().items():
        if "meld" not in name.lower():
            continue

        if isinstance(obj, dict):
            keys = set(obj.keys())

            # Common standardized split format.
            if {"test_face", "test_body"}.issubset(keys):
                candidates.append({
                    "name": name,
                    "kind": "dict_test",
                    "test_face": obj["test_face"],
                    "test_body": obj["test_body"],
                    "meta": obj.get("test_meta", obj.get("meta", None)),
                })

            # Raw full arrays.
            if {"face", "body"}.issubset(keys):
                candidates.append({
                    "name": name,
                    "kind": "dict_full",
                    "test_face": obj["face"],
                    "test_body": obj["body"],
                    "meta": obj.get("meta", None),
                })

    # Separate variable-name candidates.
    possible_pairs = [
        ("meld_test_face", "meld_test_body"),
        ("test_face_meld", "test_body_meld"),
        ("MELD_test_face", "MELD_test_body"),
        ("meld_face", "meld_body"),
    ]
    for f_name, b_name in possible_pairs:
        if f_name in globals() and b_name in globals():
            candidates.append({
                "name": f"{f_name}+{b_name}",
                "kind": "array_pair",
                "test_face": globals()[f_name],
                "test_body": globals()[b_name],
                "meta": globals().get("meld_meta", globals().get("MELD_meta", None)),
            })

    # Remove invalid shapes.
    clean = []
    for c in candidates:
        try:
            f = np.asarray(c["test_face"])
            b = np.asarray(c["test_body"])
            if f.ndim == 3 and b.ndim == 3 and len(f) == len(b):
                c["n_windows"] = int(len(f))
                c["face_shape"] = str(f.shape)
                c["body_shape"] = str(b.shape)
                clean.append(c)
        except Exception:
            pass

    return clean

def make_true_vs_random_body(face_np, body_np, seed=42, max_n=None):
    rng = np.random.default_rng(seed)
    n = len(face_np)
    idx = np.arange(n)
    if max_n is not None and n > max_n:
        idx = np.sort(rng.choice(idx, size=max_n, replace=False))

    donor = rng.permutation(idx)
    same = donor == idx
    if np.any(same):
        donor[same] = np.roll(donor, 1)[same]

    f = np.concatenate([face_np[idx], face_np[idx]], axis=0)
    b = np.concatenate([body_np[idx], body_np[donor]], axis=0)
    y = np.concatenate([np.ones(len(idx)), np.zeros(len(idx))]).astype(int)
    anchor = np.concatenate([idx, idx]).astype(int)
    pair_type = np.array(["true_pair"] * len(idx) + ["random_body"] * len(idx))
    return f, b, y, anchor, pair_type

meld_candidates = find_meld_dataset_candidates()

candidate_report = pd.DataFrame([
    {
        "name": c["name"],
        "kind": c["kind"],
        "n_windows": c["n_windows"],
        "face_shape": c["face_shape"],
        "body_shape": c["body_shape"],
        "has_meta": c["meta"] is not None,
    }
    for c in meld_candidates
])
save_df_v7(candidate_report, OUT_MELD_V7 / "meld_dataset_candidates.csv")
display(candidate_report)

if not meld_candidates:
    save_status(
        "meld_existing_window_stability_status.csv",
        status="no_meld_arrays_found",
        interpretation="No standardized MELD arrays were found in runtime. Run the MELD preprocessing/evaluation notebook first, or keep MELD as the 402-window in-the-wild validation described in the manuscript."
    )
else:
    if "trained_models" not in globals():
        raise RuntimeError("trained_models not found. Run the main PMCI training cells first.")

    model_names = [m for m in ["DirectCosine_K0", "PMCI_K8", "PMCI_K16", "PMCI_K32"] if m in trained_models]
    rows = []
    score_rows = []

    for ds_id, c in enumerate(meld_candidates):
        face_m = np.asarray(c["test_face"], dtype=np.float32)
        body_m = np.asarray(c["test_body"], dtype=np.float32)

        f_bank, b_bank, y, anchor, pair_type = make_true_vs_random_body(
            face_m,
            body_m,
            seed=7700 + ds_id,
            max_n=None,
        )

        for model_name in model_names:
            model = _unwrap_model_v7(trained_models[model_name])
            s = _score_model_v7(model, f_bank, b_bank)
            auc = _safe_auc(y, s)
            ap = _safe_ap(y, s)
            boot_mean, boot_lo, boot_hi, boot_n = _bootstrap_ci(y, s, roc_auc_score, B=1000, seed=8000 + ds_id)

            rows.append({
                "experiment": "MELD_existing_windows_true_vs_random_body",
                "dataset_candidate": c["name"],
                "model": model_name,
                "n_windows": int(len(face_m)),
                "n_pairs": int(len(y)),
                "positive_rate": float(np.mean(y)),
                "roc_auc": auc,
                "pr_auc": ap,
                "roc_auc_boot_mean": boot_mean,
                "roc_auc_lo": boot_lo,
                "roc_auc_hi": boot_hi,
                "boot_n": boot_n,
                "pos_mean": float(np.mean(s[y == 1])),
                "neg_mean": float(np.mean(s[y == 0])),
            })

            score_rows.append(pd.DataFrame({
                "dataset_candidate": c["name"],
                "model": model_name,
                "y": y,
                "score": s,
                "anchor": anchor,
                "pair_type": pair_type,
            }))

    meld_stability_df = pd.DataFrame(rows)
    save_df_v7(meld_stability_df, OUT_MELD_V7 / "meld_existing_windows_random_body_bootstrap.csv")
    display(meld_stability_df)

    if score_rows:
        meld_scores_df = pd.concat(score_rows, ignore_index=True)
        save_df_v7(meld_scores_df, OUT_MELD_V7 / "meld_existing_windows_scores.csv")


In [ ]:
# ============================================================
# V7 CELL 5 — MELD multi-crop feasibility audit
# Purpose:
#   Decide whether a true MELD multi-window expansion is possible in this runtime.
#   This cell deliberately avoids fabricating extra "independent" MELD samples.
# ============================================================

def infer_meld_expansion_feasibility(meld_file_df, meld_global_df):
    rows = []

    n_video = 0
    n_data = 0
    if meld_file_df is not None and len(meld_file_df):
        n_video = int((meld_file_df["kind"] == "video").sum()) if "kind" in meld_file_df else 0
        n_data = int((meld_file_df["kind"] == "data").sum()) if "kind" in meld_file_df else 0

    has_runtime_arrays = False
    if meld_global_df is not None and len(meld_global_df):
        text = " ".join(
            (meld_global_df.get("keys", pd.Series(dtype=str)).astype(str).tolist()
             + meld_global_df.get("columns", pd.Series(dtype=str)).astype(str).tolist()
             + meld_global_df.get("name", pd.Series(dtype=str)).astype(str).tolist())
        ).lower()
        has_runtime_arrays = any(k in text for k in ["test_face", "test_body", "face", "body", "source_video", "clip"])

    if n_video > 0:
        status = "raw_meld_videos_found"
        recommendation = (
            "A true multi-crop MELD expansion may be possible, but it must use the same pose-quality filtering "
            "as RAVDESS and should be reported as a sensitivity check unless independent source clips increase."
        )
    elif has_runtime_arrays:
        status = "meld_arrays_found_no_raw_video_confirmed"
        recommendation = (
            "Existing MELD arrays can be bootstrapped or grouped if metadata exists. A true multi-crop expansion "
            "requires raw clips or saved frame/keypoint sequences."
        )
    elif n_data > 0:
        status = "meld_data_files_found_no_clear_arrays"
        recommendation = (
            "Some MELD data files were found. Inspect them before claiming expansion. If they only contain the 402-window "
            "cache, use them for stability checks rather than expansion."
        )
    else:
        status = "no_meld_raw_or_arrays_found"
        recommendation = (
            "Do not claim MELD expansion from this runtime. Keep MELD as a 402-window noisy in-the-wild validation setting "
            "and mention larger naturalistic validation as future work."
        )

    rows.append({
        "status": status,
        "n_meld_video_files_found": n_video,
        "n_meld_data_files_found": n_data,
        "has_runtime_meld_arrays_or_metadata": bool(has_runtime_arrays),
        "recommendation": recommendation,
    })

    return pd.DataFrame(rows)

feasibility_df = infer_meld_expansion_feasibility(meld_file_df, meld_global_df)
save_df_v7(feasibility_df, OUT_MELD_V7 / "meld_multicrop_feasibility_audit.csv")
display(feasibility_df)

print("\nSafe manuscript interpretation:")
print("MELD was retained as a noisy in-the-wild validation setting rather than treated as the primary large-scale dataset.")
print("If this audit does not find raw MELD clips or saved per-clip keypoint sequences, do not claim a MELD multi-crop expansion.")


In [ ]:
# ============================================================
# V7 CELL 6 — Optional grouped bootstrap for MELD if metadata exists
# Purpose:
#   If the runtime has source_video/clip metadata for MELD, quantify uncertainty by source clip.
# ============================================================
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

def grouped_bootstrap_auc_v7(y, score, groups, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(int)
    score = np.asarray(score)
    groups = np.asarray(groups)

    unique_groups = np.unique(groups)
    group_to_idx = {g: np.where(groups == g)[0] for g in unique_groups}

    vals = []
    for _ in range(B):
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([group_to_idx[g] for g in sampled])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(float(roc_auc_score(y[idx], score[idx])))

    vals = np.asarray(vals)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    return float(np.mean(vals)), float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975)), int(len(vals))

def infer_group_vector_from_meta(meta, n):
    if meta is None:
        return None, None

    try:
        if isinstance(meta, pd.DataFrame):
            m = meta.copy()
        else:
            m = pd.DataFrame(meta)

        for col in ["source_video", "video_id", "clip_id", "Dialogue_ID", "Utterance_ID", "file", "path"]:
            if col in m.columns and len(m) >= n:
                return col, m[col].astype(str).to_numpy()[:n]
    except Exception:
        return None, None

    return None, None

group_rows = []

if "meld_scores_df" not in globals() or not len(globals().get("meld_scores_df", [])):
    save_status(
        "meld_grouped_bootstrap_status.csv",
        status="no_meld_scores_available",
        interpretation="Run V7 CELL 4 with MELD arrays available before grouped bootstrap."
    )
else:
    for c in meld_candidates:
        group_col, group_vec_window = infer_group_vector_from_meta(c.get("meta", None), c["n_windows"])

        if group_vec_window is None:
            group_rows.append({
                "dataset_candidate": c["name"],
                "status": "no_group_metadata",
                "group_col": "",
                "n_groups": np.nan,
            })
            continue

        # The score bank duplicates each anchor once for positive and once for negative.
        group_vec_pair = np.concatenate([group_vec_window, group_vec_window])

        sub_all = meld_scores_df[meld_scores_df["dataset_candidate"] == c["name"]].copy()
        for model_name, sub in sub_all.groupby("model"):
            y = sub["y"].to_numpy().astype(int)
            s = sub["score"].to_numpy()

            if len(group_vec_pair) != len(y):
                group_rows.append({
                    "dataset_candidate": c["name"],
                    "model": model_name,
                    "status": "group_length_mismatch",
                    "group_col": group_col,
                    "n_groups": np.nan,
                })
                continue

            mean, lo, hi, boot_n = grouped_bootstrap_auc_v7(y, s, group_vec_pair, B=1000, seed=9000)
            group_rows.append({
                "dataset_candidate": c["name"],
                "model": model_name,
                "status": "ok",
                "group_col": group_col,
                "n_groups": int(len(np.unique(group_vec_pair))),
                "roc_auc": _safe_auc(y, s),
                "roc_auc_group_boot_mean": mean,
                "roc_auc_group_lo": lo,
                "roc_auc_group_hi": hi,
                "boot_n": boot_n,
            })

group_boot_df = pd.DataFrame(group_rows)
save_df_v7(group_boot_df, OUT_MELD_V7 / "meld_grouped_bootstrap_if_metadata_available.csv")
display(group_boot_df)


In [ ]:
# ============================================================
# V7 CELL 7 — Package V7 outputs
# ============================================================
summary = {
    "output_dir": str(OUT_V7),
    "figures": sorted([p.name for p in OUT_FIG_V7.glob("*")]),
    "meld_sensitivity_files": sorted([p.name for p in OUT_MELD_V7.glob("*")]),
}

with open(OUT_V7 / "v7_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = shutil.make_archive(
    "/content/pmci_meld_sensitivity_v7",
    "zip",
    str(OUT_V7)
)

print("DONE ✅ V7 package created")
print("ZIP:", zip_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))
